# Notebook 03 — Team02 Champion-Selected SageMaker Pipeline, Registry, Deployment & Monitoring

**Contributor:** s201_1552444F  
**Champion selection basis:** latest 22 Aug 2026 full-data Logistic Regression and XGBoost MLflow CSV evidence exports  
**Current selected champion:** **XGBoost**  
**Notebook configuration version:** `2026-08-22-cross-model-champion-v5`

**Evidence lock used for this update:**
- XGBoost export: `S202_8440104B_XGBoost_Experiments_Evidence_Runs.csv` — SHA-256 `d95703aa1e4b91197ca08fa545db7a7f10adb1e37a233938f19c50f3f6617228`
- Logistic Regression export: `S203_2730358J_Logistic_Regression_Experiment_Evidence_Runs.csv` — SHA-256 `f95bfd7d23b0257b2bec267064b9f661004531337be989f0728b0ae0e86a4153`
- Champion extraction follows the latest `final_holdout_evaluation` row and linked candidate/refinement/threshold run IDs.
- The latest CSVs do not expose the full-profile or training-split SHA-256 values. Notebook 03 retains those as operational lineage anchors and revalidates them directly against S3. The XGBoost final export does expose the exact comparison test-split hash.


This notebook does **not assume a model family before comparison**. It first applies the team's frozen champion rule to the latest final held-out evidence from Model A and Model B, then operationalises the selected champion through SageMaker.

### Champion-selection rule

1. Compare the latest **final hold-out** result from each model family on the same experiment lineage.
2. Use **held-out PR-AUC as the predefined primary champion metric**.
3. Freeze the winning model family, source run IDs, hyperparameters and decision threshold.
4. Do **not** retune or switch metrics inside MLOps.
5. Keep human approval before registry promotion/deployment.

### Current evidence outcome

| Model | Held-out PR-AUC | ROC-AUC | F1 | Recall | Balanced Accuracy |
|---|---:|---:|---:|---:|---:|
| Logistic Regression | 0.432749 | 0.817565 | 0.482005 | **0.656660** | **0.728438** |
| **XGBoost** | **0.464397** | **0.824820** | **0.492442** | 0.621388 | 0.726302 |

**Champion = XGBoost**, because its held-out PR-AUC is higher by **0.031649**. The higher Logistic Regression recall and balanced accuracy remain documented trade-offs rather than being hidden.

### One end-to-end flow

```text
Latest LR + XGBoost final experiment evidence
        ↓
Frozen champion rule = held-out PR-AUC
        ↓
Select champion model family + exact run lineage
        ↓
Cleaned BRFSS data in S3
        ↓
Data ingestion + lineage/hash verification
        ↓
Deterministic preprocessing (same 80/20 stratified split as modelling)
        ↓
Train frozen selected champion configuration
        ↓
Independent held-out regression/evaluation check + fairness evidence
        ↓
MLflow experiment / reproducibility logging
        ↓
SageMaker Model Registry (PendingManualApproval)
        ↓
Human-over-the-loop review and approval
        ↓
SageMaker Serverless Endpoint
        ↓
Streamlit / FastAPI demo inference
        ↓
CloudWatch + data/model drift monitoring
```

**Freeze-point note:** the hold-out set has now been used for final model-family selection. After this champion freeze, additional model-family search should not reuse the same hold-out for another selection decision; a new untouched final test set would be required for an unbiased new comparison.

## Human Control Panel — MLOps Lifecycle Decisions

The notebook first selects the champion from the latest final Logistic Regression and XGBoost evidence using the frozen primary metric, **held-out PR-AUC**. The current evidence selects **XGBoost**.

The model-selection decision and the MLOps lifecycle controls are intentionally separated:
- **model family selection:** evidence-driven and frozen before deployment;
- **pipeline execution:** human-controlled;
- **Model Registry approval:** human-controlled;
- **endpoint deployment:** human-controlled;
- **endpoint cleanup:** human-controlled.

The notebook contains a fail-safe: if future evidence selects a different family, the current XGBoost-specific training implementation will stop rather than silently deploy the wrong model.

In [1]:
# ============================================================
# TEAM02 MLOPS HUMAN CONTROL PANEL
# Contributor: s201_1552444F
# ============================================================

PIPELINE_RUN_MODE = "FINAL_CHAMPION"
MODEL_SELECTION_MODE = "AUTO_FROM_FROZEN_FINAL_EVIDENCE"

# Safe defaults should normally remain False until the preceding evidence is reviewed.
# Retain/change these values deliberately for the classroom execution sequence.
RUN_PIPELINE_NOW = True
APPROVE_REGISTERED_MODEL = True
DEPLOY_ENDPOINT_AFTER_APPROVAL = True
DELETE_ENDPOINT_NOW = True
CONFIRM_DELETE_ENDPOINT_NAME = "team02-diabetes-risk"  # must exactly equal ENDPOINT_NAME before deletion

print("Human Control Panel")
print("-" * 72)
print("Pipeline mode                 :", PIPELINE_RUN_MODE)
print("Champion selection mode       :", MODEL_SELECTION_MODE)
print("Run pipeline now              :", RUN_PIPELINE_NOW)
print("Approve registered model      :", APPROVE_REGISTERED_MODEL)
print("Deploy approved endpoint      :", DEPLOY_ENDPOINT_AFTER_APPROVAL)
print("Delete endpoint after testing :", DELETE_ENDPOINT_NOW)

Human Control Panel
------------------------------------------------------------------------
Pipeline mode                 : FINAL_CHAMPION
Champion selection mode       : AUTO_FROM_FROZEN_FINAL_EVIDENCE
Run pipeline now              : True
Approve registered model      : True
Deploy approved endpoint      : True
Delete endpoint after testing : True


## 0A. Select the Best Champion from the Latest Model A and Model B Evidence

The MLOps notebook must deploy the **team champion**, not simply the latest hyperparameter set from each modeller.

The latest attached experiment evidence provides two final model-family results, locked to the exact export hashes stated above:

- **Logistic Regression:** `lr_candidate_12` → final run `82bc9a4720e04a50ae942128305eb90d`, threshold **0.59**, held-out PR-AUC **0.432749**.
- **XGBoost:** `xgb_candidate_29` → refined to **450 estimators** → final run `057bb58d81564c0db9a628b224eab8f9`, threshold **0.50**, held-out PR-AUC **0.464397**.

The code below calculates the champion from the frozen comparison evidence rather than setting `BEST_MODEL_TYPE = "XGBoost"` in advance. With the current evidence, **XGBoost is selected**.

The selected champion's exact model parameters, threshold and MLflow lineage are then transferred into the SageMaker pipeline without retuning.

In [2]:
# ============================================================
# TEAM02 CROSS-MODEL CHAMPION SELECTION + FROZEN CONFIGURATION
# Source: 22 Aug 2026 full-data XGBoost + Logistic Regression MLflow CSV exports
# ============================================================

import json
from pathlib import Path

BEST_MODEL_CONFIG_VERSION = "2026-08-22-cross-model-champion-v5"
CHAMPION_PRIMARY_METRIC = "pr_auc"
CHAMPION_PRIMARY_METRIC_LABEL = "held-out PR-AUC"
CHAMPION_TIE_EPSILON = 1e-12

SOURCE_EVIDENCE_EXPORTS = {
    "XGBoost": {
        "logical_filename": "S202_8440104B_XGBoost_Experiments_Evidence_Runs.csv",
        "sha256": "d95703aa1e4b91197ca08fa545db7a7f10adb1e37a233938f19c50f3f6617228",
        "final_run_id": "057bb58d81564c0db9a628b224eab8f9",
        "candidate_run_id": "02807641d2814f48a084c7c451414917",
        "refinement_run_id": "e89987fd9aec4a11a1108c8afaf6561b",
        "threshold_run_id": "a1a47bcc23d844539d92f3a3ed8fec1e",
    },
    "LogisticRegression": {
        "logical_filename": "S203_2730358J_Logistic_Regression_Experiment_Evidence_Runs.csv",
        "sha256": "f95bfd7d23b0257b2bec267064b9f661004531337be989f0728b0ae0e86a4153",
        "final_run_id": "82bc9a4720e04a50ae942128305eb90d",
        "candidate_run_id": "74e8884c230349f3ad29021db15f54a2",
        "threshold_run_id": "4d0a1f65fd754905bb85a4459886cb10",
    },
}

# Model Registry / endpoint resources.
PROGRESS_MODEL_PACKAGE_GROUP = "team02-diabetes-risk-progress"
PROGRESS_ENDPOINT_NAME = "team02-diabetes-risk-test"
FINAL_MODEL_PACKAGE_GROUP = "team02-diabetes-risk"
FINAL_ENDPOINT_NAME = "team02-diabetes-risk"

# Shared MLflow configuration.
MLFLOW_APP_ARN = (
    "arn:aws:sagemaker:ap-southeast-1:044528205969:"
    "mlflow-app/app-YHNC33FF6GPZ"
)
MLFLOW_EXPERIMENT = "ITI113/team02/Experiment1"

# ------------------------------------------------------------
# Latest full-data apples-to-apples experiment lineage
# ------------------------------------------------------------
EXPERIMENT_DATA_SOURCE_S3_URI = (
    "s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/"
    "diabetes_binary_cleaned_dataset.csv"
)
EXPERIMENT_DATA_PROFILE = "full"
EXPERIMENT_PROFILE_FRACTION = 1.0
EXPERIMENT_PROFILE_ROWS = 253680
EXPERIMENT_TEST_SIZE = 0.20
EXPERIMENT_RANDOM_STATE = 42
EXPERIMENT_CV_FOLDS = 5

EXPERIMENT_PROFILE_SHA256 = (
    "6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5"
)
EXPERIMENT_TRAIN_SPLIT_SHA256 = (
    "93531a7b366471a63d29fd970abad6c5d04a2f933f9ffb2aa38dce1c7b286d65"
)
EXPERIMENT_TEST_SPLIT_SHA256 = (
    "12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1"
)
EXPERIMENT_DATA_VERSION = (
    "brfss2015-diabetes-binary-" + EXPERIMENT_PROFILE_SHA256[:12]
)

COMMON_DATA_LINEAGE = {
    "data_source_s3_uri": EXPERIMENT_DATA_SOURCE_S3_URI,
    "data_profile": EXPERIMENT_DATA_PROFILE,
    "profile_fraction": EXPERIMENT_PROFILE_FRACTION,
    "profile_rows": EXPERIMENT_PROFILE_ROWS,
    "test_size": EXPERIMENT_TEST_SIZE,
    "cv_folds": EXPERIMENT_CV_FOLDS,
    "random_state": EXPERIMENT_RANDOM_STATE,
    "profile_sha256": EXPERIMENT_PROFILE_SHA256,
    "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
    "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
    "latest_csv_test_split_sha256": "12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1",
    "latest_csv_hash_scope": (
        "Latest XGBoost export provides comparison_test_split_sha256. "
        "Profile/train hashes are Notebook 03 operational lineage anchors "
        "revalidated directly from S3 at runtime."
    ),
    "split_strategy": (
        "Deterministic stratified 80/20 train/test split; "
        "Stratified 5-Fold CV on training data is the modelling validation mechanism."
    ),
}

# ------------------------------------------------------------
# Model A — latest Logistic Regression final configuration (S203)
# ------------------------------------------------------------
LR_BEST_CONFIG = {
    "model_type": "LogisticRegression",
    "source_owner": "s203",
    "experiment_batch_id": "20260821T232057Z",
    "candidate_name": "lr_candidate_12",
    "candidate_run_id": "74e8884c230349f3ad29021db15f54a2",
    "threshold_tuning_run_id": "4d0a1f65fd754905bb85a4459886cb10",
    "final_holdout_run_id": "82bc9a4720e04a50ae942128305eb90d",
    "decision_threshold": 0.59,
    "model_params": {
        "C": 100.0,
        "class_weight": "balanced",
        "solver": "lbfgs",
        "penalty": "l2",
        "max_iter": 3000,
        "random_state": 42,
    },
    "selection_evidence": {
        "selection_policy": "required_optimized_class_weight=balanced",
        "primary_selection_metric": "cv_pr_auc_mean",
        "secondary_selection_metric": "cv_roc_auc_mean",
        "candidate_cv_pr_auc_mean": 0.429812989,
        "candidate_cv_f1_mean": 0.469485345,
        "candidate_cv_roc_auc_mean": 0.817737061,
        "optimal_oof_threshold": 0.59,
        "final_test_accuracy": 0.777593819,
        "final_test_precision": 0.380738270,
        "final_test_recall": 0.656660413,
        "final_test_f1": 0.482005141,
        "final_test_pr_auc": 0.432748858,
        "final_test_roc_auc": 0.817565379,
        "final_test_balanced_accuracy": 0.728437831,
    },
    "data_lineage": COMMON_DATA_LINEAGE,
}

# ------------------------------------------------------------
# Model B — latest XGBoost final configuration (S202)
# ------------------------------------------------------------
XGB_BEST_CONFIG = {
    "model_type": "XGBoost",
    "source_owner": "s202",
    "experiment_batch_id": "20260822T033734Z",
    "candidate_name": "xgb_candidate_29",
    "candidate_run_id": "02807641d2814f48a084c7c451414917",
    "n_estimators_refinement_run_id": "e89987fd9aec4a11a1108c8afaf6561b",
    "threshold_tuning_run_id": "a1a47bcc23d844539d92f3a3ed8fec1e",
    "final_holdout_run_id": "057bb58d81564c0db9a628b224eab8f9",
    "decision_threshold": 0.50,
    "candidate_n_estimators": 350,
    "model_params": {
        "n_estimators": 450,
        "max_depth": 4,
        "learning_rate": 0.03,
        "min_child_weight": 5.0,
        "subsample": 0.80,
        "colsample_bytree": 0.80,
        "reg_alpha": 0.10,
        "reg_lambda": 10.0,
        "gamma": 0.05,
        "scale_pos_weight": 3.0,
        "objective": "binary:logistic",
        "eval_metric": "aucpr",
        "tree_method": "hist",
        "n_jobs": -1,
        "random_state": 42,
    },
    "selection_evidence": {
        "primary_selection_metric": "cv_pr_auc_mean",
        "near_best_pr_auc_tolerance": 0.002,
        "candidate_cv_pr_auc_mean": 0.459491315,
        "candidate_cv_f1_mean": 0.492713148,
        "candidate_cv_roc_auc_mean": 0.825413606,
        "refined_n_estimators": 450,
        "refined_cv_pr_auc_mean": 0.4603,
        "refined_cv_f1_mean": 0.4937,
        "refined_cv_roc_auc_mean": 0.8256,
        "optimal_oof_threshold": 0.50,
        "threshold_objective": "f1",
        "final_test_pr_auc": 0.464397478,
        "final_test_roc_auc": 0.824820173,
        "final_test_f1": 0.492441889,
        "final_test_recall": 0.621388368,
        "final_test_precision": 0.407814809,
        "final_test_balanced_accuracy": 0.726302148,
        "final_test_f2": 0.562474526,
        "comparison_lr_test_pr_auc": 0.432748858,
        "xgb_minus_lr_test_pr_auc": 0.031648620,
        "comparison_method": "evidence_pinned_lr_run_and_rescore_on_xgb_test_rows",
        "comparison_test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
        "strict_lr_comparison_available": True,
        "lr_feature_schema_match": True,
        "lr_re_evaluated_on_xgb_test_rows": True,
    },
    "data_lineage": COMMON_DATA_LINEAGE,
    # Retained from the XGBoost final experiment as a deployment regression gate.
    "quality_gate": {
        "metric": "test_auc_roc",
        "minimum": 0.75,
    },
}

MODEL_CONFIGS = {
    "LogisticRegression": LR_BEST_CONFIG,
    "XGBoost": XGB_BEST_CONFIG,
}

MODEL_SELECTION_EVIDENCE = {
    "LogisticRegression": {
        "final_run_id": LR_BEST_CONFIG["final_holdout_run_id"],
        "candidate_name": LR_BEST_CONFIG["candidate_name"],
        "candidate_run_id": LR_BEST_CONFIG["candidate_run_id"],
        "threshold_tuning_run_id": LR_BEST_CONFIG["threshold_tuning_run_id"],
        "decision_threshold": LR_BEST_CONFIG["decision_threshold"],
        "accuracy": 0.777593819,
        "precision": 0.380738270,
        "recall": 0.656660413,
        "f1": 0.482005141,
        "pr_auc": 0.432748858,
        "roc_auc": 0.817565379,
        "balanced_accuracy": 0.728437831,
    },
    "XGBoost": {
        "final_run_id": XGB_BEST_CONFIG["final_holdout_run_id"],
        "candidate_name": XGB_BEST_CONFIG["candidate_name"],
        "candidate_run_id": XGB_BEST_CONFIG["candidate_run_id"],
        "refinement_run_id": XGB_BEST_CONFIG["n_estimators_refinement_run_id"],
        "threshold_tuning_run_id": XGB_BEST_CONFIG["threshold_tuning_run_id"],
        "decision_threshold": XGB_BEST_CONFIG["decision_threshold"],
        "accuracy": 0.798151214,
        "precision": 0.407814809,
        "recall": 0.621388368,
        "f1": 0.492441889,
        "f2": 0.562474526,
        "pr_auc": 0.464397478,
        "roc_auc": 0.824820173,
        "balanced_accuracy": 0.726302148,
        "brier": 0.133926098,
    },
}

# ------------------------------------------------------------
# Source-evidence provenance — exact CSV exports used for this v5 update
# ------------------------------------------------------------
SOURCE_EVIDENCE_MANIFEST = {
    "config_version": BEST_MODEL_CONFIG_VERSION,
    "champion_primary_metric": CHAMPION_PRIMARY_METRIC_LABEL,
    "source_exports": SOURCE_EVIDENCE_EXPORTS,
    "extraction_rule": (
        "Use each family's latest final_holdout_evaluation row, follow linked "
        "candidate/refinement/threshold run IDs, then compare final held-out PR-AUC."
    ),
    "xgboost": {
        "selected_hyperparameters": XGB_BEST_CONFIG["model_params"],
        "decision_threshold": XGB_BEST_CONFIG["decision_threshold"],
        "final_metrics": MODEL_SELECTION_EVIDENCE["XGBoost"],
    },
    "logistic_regression": {
        "selected_hyperparameters": LR_BEST_CONFIG["model_params"],
        "decision_threshold": LR_BEST_CONFIG["decision_threshold"],
        "final_metrics": MODEL_SELECTION_EVIDENCE["LogisticRegression"],
    },
    "lineage_limitations": {
        "profile_sha256_in_latest_csv_exports": False,
        "train_split_sha256_in_latest_csv_exports": False,
        "test_split_sha256_in_latest_xgboost_export": True,
        "test_split_sha256": "12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1",
    },
}
Path("source_experiment_evidence_manifest.json").write_text(
    json.dumps(SOURCE_EVIDENCE_MANIFEST, indent=2),
    encoding="utf-8",
)

# ------------------------------------------------------------
# Cross-model champion selection — do not hard-code model family.
# ------------------------------------------------------------
ranked_models = sorted(
    MODEL_SELECTION_EVIDENCE.items(),
    key=lambda item: float(item[1][CHAMPION_PRIMARY_METRIC]),
    reverse=True,
)

best_name, best_metrics = ranked_models[0]
runner_up_name, runner_up_metrics = ranked_models[1]
primary_metric_delta = (
    float(best_metrics[CHAMPION_PRIMARY_METRIC])
    - float(runner_up_metrics[CHAMPION_PRIMARY_METRIC])
)

if abs(primary_metric_delta) <= CHAMPION_TIE_EPSILON:
    raise RuntimeError(
        "Champion selection is tied on the predefined primary metric. "
        "Human review is required before deployment; do not switch metrics post-hoc."
    )

BEST_MODEL_TYPE = best_name
BEST_MODEL_CONFIG = MODEL_CONFIGS[BEST_MODEL_TYPE]
RUNNER_UP_MODEL_TYPE = runner_up_name

# The current implementation uses the XGBoost training/serving path because the
# latest cross-model evidence selects XGBoost. If future evidence selects LR,
# later fail-safe checks stop execution instead of silently deploying XGBoost.
if BEST_MODEL_TYPE == "XGBoost":
    BEST_MODEL_CONFIG["champion_rule"] = (
        "Held-out PR-AUC is the predefined cross-model primary metric; "
        "recall/FNR, balanced accuracy, F2, calibration and fairness remain "
        "mandatory documented trade-offs."
    )
else:
    BEST_MODEL_CONFIG["champion_rule"] = (
        "Held-out PR-AUC is the predefined cross-model primary metric; "
        "secondary trade-offs remain documented and require human review."
    )

BEST_MODEL_CONFIG_PATH = Path("team02_champion_model_config.json")
BEST_MODEL_CONFIG_PATH.write_text(
    json.dumps(
        {
            "config_version": BEST_MODEL_CONFIG_VERSION,
            "selection_primary_metric": CHAMPION_PRIMARY_METRIC_LABEL,
            "selected_model": BEST_MODEL_TYPE,
            "runner_up_model": RUNNER_UP_MODEL_TYPE,
            "primary_metric_delta": primary_metric_delta,
            "progress_model_package_group": PROGRESS_MODEL_PACKAGE_GROUP,
            "progress_endpoint_name": PROGRESS_ENDPOINT_NAME,
            "final_model_package_group": FINAL_MODEL_PACKAGE_GROUP,
            "final_endpoint_name": FINAL_ENDPOINT_NAME,
            "experiment_comparison": MODEL_SELECTION_EVIDENCE,
            "model_configs": MODEL_CONFIGS,
            "best_model": BEST_MODEL_CONFIG,
        },
        indent=2,
    ),
    encoding="utf-8",
)

# Backward-compatible filename for cells/artifacts that may still expect it.
Path("team02_best_model_config.json").write_text(
    BEST_MODEL_CONFIG_PATH.read_text(encoding="utf-8"),
    encoding="utf-8",
)

print("CROSS-MODEL CHAMPION SELECTION")
print("=" * 92)
print("Primary metric   :", CHAMPION_PRIMARY_METRIC_LABEL)
for model_name, metrics in ranked_models:
    print(f"{model_name:<20}: {metrics[CHAMPION_PRIMARY_METRIC]:.9f}")
print("-" * 92)
print("Selected champion:", BEST_MODEL_TYPE)
print("Runner-up        :", RUNNER_UP_MODEL_TYPE)
print("Metric delta     :", f"{primary_metric_delta:+.9f}")
print("Champion run     :", BEST_MODEL_CONFIG["final_holdout_run_id"])
print("Threshold        :", BEST_MODEL_CONFIG["decision_threshold"])
print("Config version   :", BEST_MODEL_CONFIG_VERSION)
print("=" * 92)

CROSS-MODEL CHAMPION SELECTION
Primary metric   : held-out PR-AUC
XGBoost             : 0.464397478
LogisticRegression  : 0.432748858
--------------------------------------------------------------------------------------------
Selected champion: XGBoost
Runner-up        : LogisticRegression
Metric delta     : +0.031648620
Champion run     : 057bb58d81564c0db9a628b224eab8f9
Threshold        : 0.5
Config version   : 2026-08-22-cross-model-champion-v5


In [3]:
# ============================================================
# 0B. OPTIONAL LOCAL VERIFICATION OF THE TWO SOURCE CSV EXPORTS
# ============================================================
# If matching CSV exports are present beside the notebook, verify that the
# frozen v5 handoff exactly matches their linked run rows. The pipeline does
# not require these local files; absence is recorded as NOT_PRESENT.

import csv
import hashlib
from pathlib import Path

_XGB_FINAL_RUN_ID = XGB_BEST_CONFIG["final_holdout_run_id"]
_XGB_CANDIDATE_RUN_ID = XGB_BEST_CONFIG["candidate_run_id"]
_XGB_REFINEMENT_RUN_ID = XGB_BEST_CONFIG["n_estimators_refinement_run_id"]
_XGB_THRESHOLD_RUN_ID = XGB_BEST_CONFIG["threshold_tuning_run_id"]
_XGB_THRESHOLD = float(XGB_BEST_CONFIG["decision_threshold"])
_XGB_MODEL_PARAMS = XGB_BEST_CONFIG["model_params"]

_LR_FINAL_RUN_ID = LR_BEST_CONFIG["final_holdout_run_id"]
_LR_CANDIDATE_RUN_ID = LR_BEST_CONFIG["candidate_run_id"]
_LR_THRESHOLD = float(LR_BEST_CONFIG["decision_threshold"])
_LR_MODEL_PARAMS = LR_BEST_CONFIG["model_params"]

def _file_sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def _read_export_rows(path):
    with open(path, "r", encoding="utf-8-sig", newline="") as f:
        return list(csv.DictReader(f))

def _norm_export_value(value):
    return str(value).strip().strip("'\"")

def _find_export(pattern, expected_sha256):
    candidates = sorted(Path(".").glob(pattern))
    if not candidates:
        return None
    exact = [p for p in candidates if _file_sha256(p) == expected_sha256]
    return exact[0] if exact else candidates[0]

def _row_by_run_id(rows, run_id):
    matches = [r for r in rows if r.get("Run ID") == run_id]
    if len(matches) != 1:
        raise RuntimeError(
            f"Expected exactly one evidence row for Run ID {run_id}; found {len(matches)}."
        )
    return matches[0]

def _assert_text(row, key, expected):
    observed = _norm_export_value(row.get(key, ""))
    if observed != str(expected):
        raise RuntimeError(
            f"Evidence mismatch for {key}: expected={expected!r}, observed={observed!r}"
        )

def _assert_float(row, key, expected, atol=1e-12):
    observed = float(_norm_export_value(row.get(key, "")))
    if abs(observed - float(expected)) > atol:
        raise RuntimeError(
            f"Evidence mismatch for {key}: expected={expected}, observed={observed}"
        )

verification = {}

xgb_export = _find_export(
    "S202_8440104B_XGBoost_Experiments_Evidence_Runs*.csv",
    SOURCE_EVIDENCE_EXPORTS["XGBoost"]["sha256"],
)
if xgb_export is None:
    verification["XGBoost"] = {"status": "NOT_PRESENT"}
else:
    rows = _read_export_rows(xgb_export)
    final = _row_by_run_id(rows, _XGB_FINAL_RUN_ID)
    refine = _row_by_run_id(rows, _XGB_REFINEMENT_RUN_ID)

    _assert_text(final, "selected_candidate_run_id", _XGB_CANDIDATE_RUN_ID)
    _assert_text(final, "n_estimators_refinement_run_id", _XGB_REFINEMENT_RUN_ID)
    _assert_text(final, "threshold_tuning_run_id", _XGB_THRESHOLD_RUN_ID)
    _assert_float(final, "n_estimators", _XGB_MODEL_PARAMS["n_estimators"])
    _assert_float(final, "max_depth", _XGB_MODEL_PARAMS["max_depth"])
    _assert_float(final, "learning_rate", _XGB_MODEL_PARAMS["learning_rate"])
    _assert_float(final, "scale_pos_weight", _XGB_MODEL_PARAMS["scale_pos_weight"])
    _assert_float(final, "selected_threshold", _XGB_THRESHOLD)
    _assert_float(final, "test_pr_auc", MODEL_SELECTION_EVIDENCE["XGBoost"]["pr_auc"])
    _assert_float(final, "test_auc_roc", MODEL_SELECTION_EVIDENCE["XGBoost"]["roc_auc"])
    _assert_text(final, "objective", _XGB_MODEL_PARAMS["objective"])
    _assert_text(final, "eval_metric", _XGB_MODEL_PARAMS["eval_metric"])
    _assert_text(final, "tree_method", _XGB_MODEL_PARAMS["tree_method"])
    for key in ["min_child_weight", "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "gamma"]:
        _assert_float(refine, key, _XGB_MODEL_PARAMS[key])

    verification["XGBoost"] = {
        "status": "VERIFIED",
        "path": str(xgb_export),
        "sha256": _file_sha256(xgb_export),
        "exact_export_hash_match": _file_sha256(xgb_export) == SOURCE_EVIDENCE_EXPORTS["XGBoost"]["sha256"],
    }

lr_export = _find_export(
    "S203_2730358J_Logistic_Regression_Experiment_Evidence_Runs*.csv",
    SOURCE_EVIDENCE_EXPORTS["LogisticRegression"]["sha256"],
)
if lr_export is None:
    verification["LogisticRegression"] = {"status": "NOT_PRESENT"}
else:
    rows = _read_export_rows(lr_export)
    final = _row_by_run_id(rows, _LR_FINAL_RUN_ID)
    candidate = _row_by_run_id(rows, _LR_CANDIDATE_RUN_ID)

    _assert_text(final, "selected_candidate_run_id", _LR_CANDIDATE_RUN_ID)
    _assert_float(final, "selected_threshold", _LR_THRESHOLD)
    _assert_float(final, "test_pr_auc", MODEL_SELECTION_EVIDENCE["LogisticRegression"]["pr_auc"])
    _assert_float(final, "test_auc_roc", MODEL_SELECTION_EVIDENCE["LogisticRegression"]["roc_auc"])
    _assert_float(candidate, "C", _LR_MODEL_PARAMS["C"])
    _assert_text(candidate, "class_weight", _LR_MODEL_PARAMS["class_weight"])
    _assert_text(candidate, "solver", _LR_MODEL_PARAMS["solver"])
    _assert_text(candidate, "penalty", _LR_MODEL_PARAMS["penalty"])
    _assert_float(candidate, "max_iter", _LR_MODEL_PARAMS["max_iter"])
    _assert_float(candidate, "random_state", _LR_MODEL_PARAMS["random_state"])

    verification["LogisticRegression"] = {
        "status": "VERIFIED",
        "path": str(lr_export),
        "sha256": _file_sha256(lr_export),
        "exact_export_hash_match": _file_sha256(lr_export) == SOURCE_EVIDENCE_EXPORTS["LogisticRegression"]["sha256"],
    }

Path("source_experiment_evidence_verification.json").write_text(
    json.dumps(verification, indent=2),
    encoding="utf-8",
)
print("SOURCE EXPERIMENT EVIDENCE VERIFICATION")
print(json.dumps(verification, indent=2))


SOURCE EXPERIMENT EVIDENCE VERIFICATION
{
  "XGBoost": {
    "status": "NOT_PRESENT"
  },
  "LogisticRegression": {
    "status": "NOT_PRESENT"
  }
}


### Human Decision Sequence

```text
Latest LR + XGBoost full-data final evidence
        ↓
Predefined champion metric = held-out PR-AUC
        ↓
Code ranks both model families on the frozen metric
        ↓
Current champion = XGBoost (0.464397 vs 0.432749)
        ↓
Freeze champion run + hyperparameters + threshold
        ↓
SageMaker Processing → Training → Evaluation
        ↓
Pre-existing ROC-AUC regression gate (≥ 0.75 for current XGBoost champion)
        ↓
Model Registry = PendingManualApproval
        ↓
Human review / governance evidence
        ↓
Approve → deploy → inference → monitor
```

This structure prevents the MLOps owner from independently changing the modelling outcome.

# Part C — MLOps, SageMaker Pipeline, Deployment & AI Governance

**Contributor: s201_1552444F**


## 0. Select the Pipeline Run Mode

The **model family is selected from the frozen final experiment evidence in Section 0A**. The current evidence selects XGBoost.

Only the MLOps lifecycle mode is manually selectable here; the champion family must not be manually overridden.

## 1. SageMaker SDK V3 Environment Validation

Verify that the modular SageMaker SDK V3 packages required by this notebook are available before constructing any AWS jobs. This early validation reduces environment-related failures later in the pipeline workflow.

In [4]:
# SageMaker SDK V3 modular-environment integrity check.
# AWS SDK V3 may be installed as separate packages rather than one umbrella distribution.

import importlib.metadata as _md
from pathlib import Path as _Path

REQUIRED_SAGEMAKER_DISTS = [
    "sagemaker-core",
    "sagemaker-train",
    "sagemaker-serve",
    "sagemaker-mlops",
]

def _dist_version(name):
    try:
        return _md.version(name)
    except _md.PackageNotFoundError:
        return None

def _distribution_has_real_file(dist_name, suffix):
    try:
        dist = _md.distribution(dist_name)
    except _md.PackageNotFoundError:
        return False, None

    for f in dist.files or []:
        if str(f).replace("\\", "/").endswith(suffix):
            real_path = _Path(dist.locate_file(f))
            return real_path.exists(), str(real_path)

    return False, None

print("SageMaker V3 modular package state:")
missing = []
for pkg in REQUIRED_SAGEMAKER_DISTS:
    v = _dist_version(pkg)
    print(f"  {pkg:16s}: {v or 'NOT INSTALLED'}")
    if v is None:
        missing.append(pkg)

serve_spec_ok, serve_spec_path = _distribution_has_real_file(
    "sagemaker-serve",
    "sagemaker/serve/spec/inference_spec.py",
)

print("Serve inference_spec present:", serve_spec_ok)
if serve_spec_path:
    print("Serve inference_spec path   :", serve_spec_path)

if missing:
    raise RuntimeError(
        "Required SageMaker V3 modular packages are missing: "
        + ", ".join(missing)
    )

if not serve_spec_ok:
    raise RuntimeError(
        "sagemaker-serve is installed but inference_spec.py is missing. "
        "The modular SageMaker installation is inconsistent."
    )

print("\nSageMaker modular package integrity check passed.")
print("No package reinstall is required.")


SageMaker V3 modular package state:
  sagemaker-core  : 2.20.0
  sagemaker-train : 1.20.0
  sagemaker-serve : 1.20.0
  sagemaker-mlops : 1.20.0
Serve inference_spec present: True
Serve inference_spec path   : /opt/conda/lib/python3.12/site-packages/sagemaker/serve/spec/inference_spec.py

SageMaker modular package integrity check passed.
No package reinstall is required.


### 1.1 Import Required SageMaker V3 Components

Import the exact SageMaker Core, Train, Serve and MLOps classes used to build the pipeline. The preflight fails before any AWS resources are created if the local SDK environment is incomplete or incompatible.

In [5]:
import importlib.metadata as _md

try:
    from sagemaker.core.helper.session_helper import Session, get_execution_role
    from sagemaker.core.workflow.pipeline_context import PipelineSession
    from sagemaker.core.processing import ScriptProcessor, FrameworkProcessor
    from sagemaker.train import ModelTrainer
    from sagemaker.serve.model_builder import ModelBuilder
    from sagemaker.mlops.workflow.pipeline import Pipeline
    from sagemaker.mlops.workflow.steps import ProcessingStep, TrainingStep
    from sagemaker.mlops.workflow.model_step import ModelStep
    from sagemaker.mlops.workflow import ConditionStep
except Exception as exc:
    raise RuntimeError(
        "SageMaker V3 import preflight failed before any AWS jobs were created. "
        f"Original import error: {type(exc).__name__}: {exc}"
    ) from exc

print("SageMaker V3 import preflight: PASS")
for pkg in ["sagemaker-core", "sagemaker-train", "sagemaker-serve", "sagemaker-mlops"]:
    try:
        print(f"{pkg:16s}:", _md.version(pkg))
    except _md.PackageNotFoundError:
        print(f"{pkg:16s}: NOT INSTALLED")


sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
SageMaker V3 import preflight: PASS
sagemaker-core  : 2.20.0
sagemaker-train : 1.20.0
sagemaker-serve : 1.20.0
sagemaker-mlops : 1.20.0


### 1.2 Generate Reusable Script/API MLOps Controls

Create a small reusable Python module for the operational control plane.

The functions accept AWS clients and explicit resource names rather than relying on notebook globals. This makes them reusable by this notebook, automated scripts/CI/CD, and a future authenticated Streamlit MLOps Admin application.

The module uses normal SageMaker, CloudWatch Logs and SageMaker Runtime APIs. It does **not** use SageMaker Inference Recommender.


In [6]:
%%writefile team02_mlops_control.py
import io
import json
import tarfile
import time
from urllib.parse import urlparse

import boto3
from botocore.exceptions import ClientError, WaiterError


def _error_code(exc):
    return exc.response.get("Error", {}).get("Code", "Unknown")


def _is_not_found(exc):
    return _error_code(exc) in {
        "ValidationException",
        "ResourceNotFound",
        "ResourceNotFoundException",
    }


def get_pipeline_execution_summary(sm, execution_arn):
    execution = sm.describe_pipeline_execution(
        PipelineExecutionArn=execution_arn
    )
    response = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn,
        SortOrder="Ascending",
        MaxResults=100,
    )

    steps = []
    for step in response.get("PipelineExecutionSteps", []):
        metadata = step.get("Metadata", {}) or {}
        processing = metadata.get("ProcessingJob", {}) or {}
        training = metadata.get("TrainingJob", {}) or {}
        model = metadata.get("Model", {}) or {}
        register_model = metadata.get("RegisterModel", {}) or {}

        steps.append({
            "step_name": step.get("StepName"),
            "step_status": step.get("StepStatus"),
            "failure_reason": step.get("FailureReason"),
            "processing_job_arn": processing.get("Arn"),
            "training_job_arn": training.get("Arn"),
            "model_arn": model.get("Arn"),
            "register_model_arn": register_model.get("Arn"),
            "start_time": step.get("StartTime"),
            "end_time": step.get("EndTime"),
        })

    return {
        "execution_arn": execution_arn,
        "pipeline_status": execution.get("PipelineExecutionStatus"),
        "failure_reason": execution.get("FailureReason"),
        "creation_time": execution.get("CreationTime"),
        "last_modified_time": execution.get("LastModifiedTime"),
        "steps": steps,
    }


def get_recent_pipeline_executions(sm, pipeline_name, max_results=10):
    response = sm.list_pipeline_executions(
        PipelineName=pipeline_name,
        SortOrder="Descending",
        MaxResults=max_results,
    )
    return response.get("PipelineExecutionSummaries", [])


def get_model_registry_summary(sm, model_package_group, max_results=20):
    response = sm.list_model_packages(
        ModelPackageGroupName=model_package_group,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=max_results,
    )
    rows = []
    for item in response.get("ModelPackageSummaryList", []):
        rows.append({
            "model_package_arn": item.get("ModelPackageArn"),
            "model_package_version": item.get("ModelPackageVersion"),
            "approval_status": item.get("ModelApprovalStatus"),
            "creation_time": item.get("CreationTime"),
        })
    return rows



def _parse_s3_uri(uri):
    parsed = urlparse(uri)
    if parsed.scheme != "s3":
        raise ValueError(f"Expected s3:// URI, received: {uri}")
    return parsed.netloc, parsed.path.lstrip("/")


def get_model_package_model_data_url(sm, model_package_arn):
    package = sm.describe_model_package(ModelPackageName=model_package_arn)
    containers = package.get("InferenceSpecification", {}).get("Containers", [])
    for container in containers:
        if container.get("ModelDataUrl"):
            return container["ModelDataUrl"]
        s3_source = container.get("ModelDataSource", {}).get("S3DataSource", {})
        if s3_source.get("S3Uri"):
            return s3_source["S3Uri"]
    raise RuntimeError("No model artifact S3 URI found in Model Package.")


def validate_model_package_artifact(sm, s3, model_package_arn):
    required_members = [
        "model.pkl",
        "model_metadata.json",
        "code/inference.py",
        "code/requirements.txt",
    ]
    model_data_url = get_model_package_model_data_url(sm, model_package_arn)
    bucket, key = _parse_s3_uri(model_data_url)
    body = s3.get_object(Bucket=bucket, Key=key)["Body"].read()

    with tarfile.open(fileobj=io.BytesIO(body), mode="r:gz") as tar:
        members = sorted(
            member.name.lstrip("./")
            for member in tar.getmembers()
            if member.isfile()
        )

    missing = [x for x in required_members if x not in members]
    return {
        "valid": not missing,
        "model_package_arn": model_package_arn,
        "model_data_url": model_data_url,
        "required_members": required_members,
        "missing_members": missing,
        "artifact_members": members,
    }

def approve_model_package(sm, model_package_arn, approval_description):
    before = sm.describe_model_package(
        ModelPackageName=model_package_arn
    )
    sm.update_model_package(
        ModelPackageArn=model_package_arn,
        ModelApprovalStatus="Approved",
        ApprovalDescription=approval_description,
    )
    after = sm.describe_model_package(
        ModelPackageName=model_package_arn
    )
    return {
        "model_package_arn": model_package_arn,
        "status_before": before.get("ModelApprovalStatus"),
        "status_after": after.get("ModelApprovalStatus"),
    }


def get_endpoint_summary(sm, endpoint_name):
    try:
        desc = sm.describe_endpoint(EndpointName=endpoint_name)
    except ClientError as exc:
        if _is_not_found(exc):
            return {
                "exists": False,
                "endpoint_name": endpoint_name,
                "endpoint_status": "NotCreated",
            }
        raise

    return {
        "exists": True,
        "endpoint_name": endpoint_name,
        "endpoint_status": desc.get("EndpointStatus"),
        "endpoint_arn": desc.get("EndpointArn"),
        "endpoint_config_name": desc.get("EndpointConfigName"),
        "creation_time": desc.get("CreationTime"),
        "last_modified_time": desc.get("LastModifiedTime"),
        "failure_reason": desc.get("FailureReason"),
    }



def get_endpoint_cloudwatch_log_tail(region_name, endpoint_name, max_events=100):
    logs = boto3.client("logs", region_name=region_name)
    log_group = f"/aws/sagemaker/Endpoints/{endpoint_name}"
    try:
        response = logs.describe_log_streams(
            logGroupName=log_group,
            orderBy="LastEventTime",
            descending=True,
            limit=20,
        )
    except ClientError as exc:
        if _is_not_found(exc):
            return []
        raise

    events = []
    for stream in response.get("logStreams", []):
        stream_name = stream["logStreamName"]
        data = logs.get_log_events(
            logGroupName=log_group,
            logStreamName=stream_name,
            startFromHead=False,
            limit=max_events,
        )
        for event in data.get("events", []):
            events.append({
                "timestamp": event.get("timestamp"),
                "message": event.get("message", "").rstrip(),
                "log_stream": stream_name,
            })
    events.sort(key=lambda x: x.get("timestamp") or 0)
    return events[-max_events:]


def diagnose_endpoint_failure(sm, endpoint_name, max_log_events=100):
    state = get_endpoint_summary(sm, endpoint_name)
    return {
        **state,
        "cloudwatch_log_group": f"/aws/sagemaker/Endpoints/{endpoint_name}",
        "cloudwatch_events": get_endpoint_cloudwatch_log_tail(
            sm.meta.region_name,
            endpoint_name,
            max_events=max_log_events,
        ),
    }

def deploy_serverless_model_package(
    sm,
    model_package_arn,
    role_arn,
    endpoint_name,
    tags,
    memory_size_mb=2048,
    max_concurrency=5,
):
    package = sm.describe_model_package(ModelPackageName=model_package_arn)
    if package.get("ModelApprovalStatus") != "Approved":
        raise RuntimeError("Deployment blocked: Model Package must be Approved.")

    current = get_endpoint_summary(sm, endpoint_name)

    if current["exists"] and current["endpoint_status"] == "Failed":
        result = diagnose_endpoint_failure(sm, endpoint_name)
        result.update({
            "deployment_succeeded": False,
            "deployment_action": "blocked_by_existing_failed_endpoint",
            "model_package_arn": model_package_arn,
            "requires_explicit_delete": True,
        })
        return result

    timestamp = int(time.time())
    model_name = f"{endpoint_name}-model-{timestamp}"
    config_name = f"{endpoint_name}-config-{timestamp}"

    sm.create_model(
        ModelName=model_name,
        PrimaryContainer={"ModelPackageName": model_package_arn},
        ExecutionRoleArn=role_arn,
        Tags=tags,
    )
    sm.create_endpoint_config(
        EndpointConfigName=config_name,
        ProductionVariants=[{
            "VariantName": "AllTraffic",
            "ModelName": model_name,
            "ServerlessConfig": {
                "MemorySizeInMB": int(memory_size_mb),
                "MaxConcurrency": int(max_concurrency),
            },
        }],
        Tags=tags,
    )

    if current["exists"]:
        sm.update_endpoint(EndpointName=endpoint_name, EndpointConfigName=config_name)
        action = "updated"
    else:
        sm.create_endpoint(
            EndpointName=endpoint_name,
            EndpointConfigName=config_name,
            Tags=tags,
        )
        action = "created"

    try:
        sm.get_waiter("endpoint_in_service").wait(EndpointName=endpoint_name)
    except WaiterError:
        result = diagnose_endpoint_failure(sm, endpoint_name)
        result.update({
            "deployment_succeeded": False,
            "deployment_action": action,
            "model_name": model_name,
            "endpoint_config_name": config_name,
            "model_package_arn": model_package_arn,
            "serverless_memory_mb": int(memory_size_mb),
            "serverless_max_concurrency": int(max_concurrency),
            "requires_explicit_delete": result.get("endpoint_status") == "Failed",
        })
        return result

    result = get_endpoint_summary(sm, endpoint_name)
    result.update({
        "deployment_succeeded": True,
        "deployment_action": action,
        "model_name": model_name,
        "endpoint_config_name": config_name,
        "model_package_arn": model_package_arn,
        "serverless_memory_mb": int(memory_size_mb),
        "serverless_max_concurrency": int(max_concurrency),
        "requires_explicit_delete": False,
        "cloudwatch_events": [],
    })
    return result

def invoke_json_endpoint(runtime, endpoint_name, payload):
    response = runtime.invoke_endpoint(
        EndpointName=endpoint_name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(payload).encode("utf-8"),
    )
    return json.loads(response["Body"].read().decode("utf-8"))


def delete_endpoint_if_exists(
    sm,
    endpoint_name,
    wait=True,
    poll_seconds=10,
    timeout_seconds=600,
):
    current = get_endpoint_summary(sm, endpoint_name)
    if not current["exists"]:
        return {
            "endpoint_name": endpoint_name,
            "deleted": False,
            "reason": "Endpoint does not exist.",
        }

    sm.delete_endpoint(EndpointName=endpoint_name)

    if wait:
        started = time.time()
        while time.time() - started < timeout_seconds:
            if not get_endpoint_summary(sm, endpoint_name)["exists"]:
                return {
                    "endpoint_name": endpoint_name,
                    "deleted": True,
                    "reason": "Endpoint deleted and absence confirmed.",
                }
            time.sleep(poll_seconds)

    return {
        "endpoint_name": endpoint_name,
        "deleted": True,
        "reason": "DeleteEndpoint API submitted.",
    }


def _find_log_streams(logs, log_group, job_name, limit=20):
    try:
        response = logs.describe_log_streams(
            logGroupName=log_group,
            logStreamNamePrefix=job_name,
            orderBy="LogStreamName",
            descending=False,
            limit=limit,
        )
    except ClientError as exc:
        if _is_not_found(exc):
            return []
        raise
    return [x["logStreamName"] for x in response.get("logStreams", [])]


def get_cloudwatch_job_log_tail(
    region_name,
    log_group,
    job_name,
    max_events=80,
):
    logs = boto3.client("logs", region_name=region_name)
    streams = _find_log_streams(logs, log_group, job_name)

    events = []
    for stream in streams:
        response = logs.get_log_events(
            logGroupName=log_group,
            logStreamName=stream,
            startFromHead=False,
            limit=max_events,
        )
        for event in response.get("events", []):
            events.append({
                "timestamp": event.get("timestamp"),
                "message": event.get("message", "").rstrip(),
                "log_stream": stream,
            })

    events.sort(key=lambda x: x.get("timestamp") or 0)
    return events[-max_events:]


def get_execution_cloudwatch_report(
    sm,
    execution_arn,
    region_name,
    max_events_per_job=60,
):
    summary = get_pipeline_execution_summary(sm, execution_arn)
    report = []

    for step in summary["steps"]:
        if step.get("processing_job_arn"):
            job_name = step["processing_job_arn"].rsplit("/", 1)[-1]
            report.append({
                "step_name": step["step_name"],
                "job_type": "ProcessingJob",
                "job_name": job_name,
                "log_group": "/aws/sagemaker/ProcessingJobs",
                "events": get_cloudwatch_job_log_tail(
                    region_name,
                    "/aws/sagemaker/ProcessingJobs",
                    job_name,
                    max_events=max_events_per_job,
                ),
            })

        if step.get("training_job_arn"):
            job_name = step["training_job_arn"].rsplit("/", 1)[-1]
            report.append({
                "step_name": step["step_name"],
                "job_type": "TrainingJob",
                "job_name": job_name,
                "log_group": "/aws/sagemaker/TrainingJobs",
                "events": get_cloudwatch_job_log_tail(
                    region_name,
                    "/aws/sagemaker/TrainingJobs",
                    job_name,
                    max_events=max_events_per_job,
                ),
            })

    return report


Overwriting team02_mlops_control.py


In [7]:
from team02_mlops_control import (
    approve_model_package,
    delete_endpoint_if_exists,
    diagnose_endpoint_failure,
    deploy_serverless_model_package,
    get_endpoint_summary,
    get_execution_cloudwatch_report,
    get_model_registry_summary,
    get_pipeline_execution_summary,
    get_recent_pipeline_executions,
    invoke_json_endpoint,
    validate_model_package_artifact,
)

print("Reusable script/API MLOps controls imported.")
print("Inference Recommender is not used by this workflow.")


Reusable script/API MLOps controls imported.
Inference Recommender is not used by this workflow.


## 2. Configure Team 02 AWS and MLflow Logging

Configure AWS, S3, the SageMaker-managed MLflow App and the **single selected-champion handoff**.


In [8]:
import io
import json
import os
import re
import time
from importlib.metadata import version as package_version
from pathlib import Path
from urllib.parse import urlparse

import boto3
from botocore.exceptions import ClientError
import mlflow
import pandas as pd

from sagemaker.core.helper.session_helper import Session, get_execution_role

TEAM_ID = "team02"
STUDENT_ID = "s201"
COURSE = "ITI113"
SEMESTER = "26S1"
PROJECT_NAME = "diabetes-risk"
MODELLING_PROJECT_NAME = "diabetes"
REGION = "ap-southeast-1"

BUCKET = "nyp-26s1-iti113"
PIPELINE_PREFIX = f"iti113/{TEAM_ID}/mlops/{PROJECT_NAME}"

PIPELINE_NAME = "team02-diabetes-risk"

VALID_PIPELINE_RUN_MODES = {
    "INTEGRATION_TEST",
    "PROGRESS_CHECK",
    "FINAL_CHAMPION",
}
if PIPELINE_RUN_MODE not in VALID_PIPELINE_RUN_MODES:
    raise ValueError(
        "PIPELINE_RUN_MODE must be one of: "
        + ", ".join(sorted(VALID_PIPELINE_RUN_MODES))
    )

if BEST_MODEL_TYPE != "XGBoost":
    raise RuntimeError(
        f"Latest evidence selected {BEST_MODEL_TYPE}, but this notebook's current "
        "training/serving implementation is XGBoost-specific. Stop and implement "
        "the selected family rather than silently deploying XGBoost."
    )

if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    MODEL_PACKAGE_GROUP = FINAL_MODEL_PACKAGE_GROUP
    ENDPOINT_NAME = FINAL_ENDPOINT_NAME
else:
    MODEL_PACKAGE_GROUP = PROGRESS_MODEL_PACKAGE_GROUP
    ENDPOINT_NAME = PROGRESS_ENDPOINT_NAME

PROCESSING_INSTANCE_TYPE = "ml.m5.large"
TRAINING_INSTANCE_TYPE = "ml.m5.large"

SOURCE_TARGET_COLUMN = "Diabetes_binary"
TARGET_COLUMN = "target"
FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

boto_session = boto3.Session(region_name=REGION)
sm_session = Session(
    boto_session=boto_session,
    default_bucket=BUCKET,
    default_bucket_prefix=f"{PIPELINE_PREFIX}/sdk-artifacts",
)

role = get_execution_role(sagemaker_session=sm_session)
region = sm_session.boto_region_name or REGION

s3 = boto_session.client("s3")
sm = boto_session.client("sagemaker")
sts = boto_session.client("sts")

caller = sts.get_caller_identity()
ACCOUNT_ID = caller["Account"]
CALLER_ARN = caller["Arn"]

role_team_match = re.search(
    r"ITI113[-_]?Team0*(\d+)",
    role,
    flags=re.IGNORECASE,
)
if role_team_match and int(role_team_match.group(1)) != 2:
    raise PermissionError(
        f"Execution role appears to belong to Team {role_team_match.group(1)}, "
        f"but this notebook is configured for {TEAM_ID}: {role}"
    )

print("SageMaker SDK   : V3 modular installation")
print("sagemaker-core  :", package_version("sagemaker-core"))
print("sagemaker-train :", package_version("sagemaker-train"))
print("sagemaker-serve :", package_version("sagemaker-serve"))
print("sagemaker-mlops :", package_version("sagemaker-mlops"))
print("AWS account     :", ACCOUNT_ID)
print("Caller ARN      :", CALLER_ARN)
print("Execution role  :", role)
print("Region          :", region)
print("Pipeline mode   :", PIPELINE_RUN_MODE)
print("Best model      :", BEST_MODEL_TYPE)
print("Pipeline        :", PIPELINE_NAME)
print("Package group   :", MODEL_PACKAGE_GROUP)
print("Endpoint name   :", ENDPOINT_NAME)

mlflow_app_desc = sm.describe_mlflow_app(Arn=MLFLOW_APP_ARN)
MLFLOW_APP_STATUS = (
    mlflow_app_desc.get("Status")
    or mlflow_app_desc.get("MlflowAppStatus")
    or "UNKNOWN"
)
MLFLOW_SERVER_VERSION = (
    mlflow_app_desc.get("MlflowVersion")
    or mlflow_app_desc.get("Version")
    or "UNKNOWN"
)

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT)
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT)

print("MLflow App       :", MLFLOW_APP_ARN)
print("MLflow status    :", MLFLOW_APP_STATUS)
print("MLflow server    :", MLFLOW_SERVER_VERSION)
print("MLflow client    :", mlflow.__version__)
print("MLflow experiment:", MLFLOW_EXPERIMENT)

# Frozen XGBoost evidence (current selected champion).
XGB_CANDIDATE_RUN_ID = XGB_BEST_CONFIG["candidate_run_id"]
XGB_REFINEMENT_RUN_ID = XGB_BEST_CONFIG["n_estimators_refinement_run_id"]
XGB_THRESHOLD_TUNING_RUN_ID = XGB_BEST_CONFIG["threshold_tuning_run_id"]
XGB_FINAL_RUN_ID = XGB_BEST_CONFIG["final_holdout_run_id"]
XGB_THRESHOLD = float(XGB_BEST_CONFIG["decision_threshold"])
XGB_MODEL_PARAMS = dict(XGB_BEST_CONFIG["model_params"])

# Generic champion aliases used by the MLOps handoff.
CHAMPION_CANDIDATE_RUN_ID = BEST_MODEL_CONFIG["candidate_run_id"]
CHAMPION_REFINEMENT_RUN_ID = BEST_MODEL_CONFIG.get("n_estimators_refinement_run_id")
CHAMPION_THRESHOLD_TUNING_RUN_ID = BEST_MODEL_CONFIG["threshold_tuning_run_id"]
CHAMPION_FINAL_RUN_ID = BEST_MODEL_CONFIG["final_holdout_run_id"]
CHAMPION_THRESHOLD = float(BEST_MODEL_CONFIG["decision_threshold"])
CHAMPION_MODEL_PARAMS = dict(BEST_MODEL_CONFIG["model_params"])

# Latest Logistic Regression comparison configuration from S203.
LR_CANDIDATE_RUN_ID = LR_BEST_CONFIG["candidate_run_id"]
LR_THRESHOLD_TUNING_RUN_ID = LR_BEST_CONFIG["threshold_tuning_run_id"]
LR_FINAL_COMPARISON_RUN_ID = LR_BEST_CONFIG["final_holdout_run_id"]
LR_THRESHOLD = float(LR_BEST_CONFIG["decision_threshold"])
LR_MODEL_PARAMS = dict(LR_BEST_CONFIG["model_params"])

# The exported CSV does not explicitly contain the experiment xgboost package version.
# Do not invent it here. The pipeline runtime is pinned separately below.
XGB_EXPERIMENT_VERSION = None

selected_handoff = {
    "model_type": BEST_MODEL_TYPE,
    "owner_student_id": BEST_MODEL_CONFIG["source_owner"],
    "project_name": MODELLING_PROJECT_NAME,
    "experiment_batch_id": BEST_MODEL_CONFIG["experiment_batch_id"],
    "parameter_source_run_id": CHAMPION_CANDIDATE_RUN_ID,
    "candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
    "n_estimators_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID,
    "threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
    "final_holdout_run_id": CHAMPION_FINAL_RUN_ID,
    "decision_threshold": CHAMPION_THRESHOLD,
    "model_params": CHAMPION_MODEL_PARAMS,
    "selection_metrics": BEST_MODEL_CONFIG["selection_evidence"],
    "comparison_logistic_regression": {
        "candidate_run_id": LR_CANDIDATE_RUN_ID,
        "threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
        "final_holdout_run_id": LR_FINAL_COMPARISON_RUN_ID,
        "decision_threshold": LR_THRESHOLD,
        "model_params": LR_MODEL_PARAMS,
        "selection_metrics": LR_BEST_CONFIG["selection_evidence"],
    },
    "data_lineage": BEST_MODEL_CONFIG["data_lineage"],
    "handoff_policy": "latest_full_data_cross_model_champion_heldout_pr_auc",
    "config_version": BEST_MODEL_CONFIG_VERSION,
}

BEST_MODEL_SELECTION_SOURCE = (
    "2026-08-22_cross_model_champion_same_holdout_pr_auc_primary"
)

approved = {
    "status": PIPELINE_RUN_MODE,
    "team_id": TEAM_ID,
    "project_name": PROJECT_NAME,
    "model_project_name": MODELLING_PROJECT_NAME,
    "data_version": EXPERIMENT_DATA_VERSION,
    # This is the stable dataframe content signature used by Model B,
    # not the raw S3 object byte checksum.
    "dataset_sha256": EXPERIMENT_PROFILE_SHA256,
    "dataset_signature_type": "pandas_hash_pandas_object_sha256",
    "data_source_s3_uri": EXPERIMENT_DATA_SOURCE_S3_URI,
    "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
    "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
    "champion_model_type": BEST_MODEL_TYPE,
    "champion_mlflow_run_id": CHAMPION_FINAL_RUN_ID,
    "source_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
    "source_parameter_run_id": CHAMPION_CANDIDATE_RUN_ID,
    "source_threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
    "source_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID,
    "source_final_run_id": CHAMPION_FINAL_RUN_ID,
    "comparison_lr_candidate_run_id": LR_CANDIDATE_RUN_ID,
    "comparison_lr_threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
    "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
    "comparison_lr_decision_threshold": LR_THRESHOLD,
    "comparison_lr_model_params": LR_MODEL_PARAMS,
    "decision_threshold": CHAMPION_THRESHOLD,
    "source_student_id": BEST_MODEL_CONFIG["source_owner"],
    "handoff_policy": selected_handoff["handoff_policy"],
    "best_model_config_version": BEST_MODEL_CONFIG_VERSION,
    "best_model_selection_source": BEST_MODEL_SELECTION_SOURCE,
    "quality_gate_metric": BEST_MODEL_CONFIG["quality_gate"]["metric"],
    "quality_gate_threshold": BEST_MODEL_CONFIG["quality_gate"]["minimum"],
    "approved_by": None,
    "approval_date": None,
}

def ensure_runtime_selection():
    return approved

CHAMPION_MODEL_TYPE = BEST_MODEL_TYPE
DECISION_THRESHOLD = CHAMPION_THRESHOLD
MODEL_PARAMS = dict(CHAMPION_MODEL_PARAMS)
EXPERIMENT_XGBOOST_VERSION = XGB_EXPERIMENT_VERSION

# Compatibility aliases for downstream cells.
MODEL_CATALOG_VERSION = BEST_MODEL_CONFIG_VERSION
FINAL_CHAMPION_DECISION_SOURCE = BEST_MODEL_SELECTION_SOURCE
MODEL_HANDOFF_MLFLOW_RUN_ID = None

print("\nFINAL-CHAMPION MLOPS HANDOFF")
print("=" * 92)
print("Model              :", CHAMPION_MODEL_TYPE)
print("Candidate run      :", CHAMPION_CANDIDATE_RUN_ID)
print("Refinement run     :", CHAMPION_REFINEMENT_RUN_ID)
print("Threshold run      :", CHAMPION_THRESHOLD_TUNING_RUN_ID)
print("Final model run    :", CHAMPION_FINAL_RUN_ID)
print("LR candidate run   :", LR_CANDIDATE_RUN_ID)
print("LR comparison run  :", LR_FINAL_COMPARISON_RUN_ID)
print("LR threshold       :", LR_THRESHOLD)
print("Decision threshold :", DECISION_THRESHOLD)
print("Config version     :", BEST_MODEL_CONFIG_VERSION)
print("Selection source   :", BEST_MODEL_SELECTION_SOURCE)
print("=" * 92)


SageMaker SDK   : V3 modular installation
sagemaker-core  : 2.20.0
sagemaker-train : 1.20.0
sagemaker-serve : 1.20.0
sagemaker-mlops : 1.20.0
AWS account     : 044528205969
Caller ARN      : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Execution role  : arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team02
Region          : ap-southeast-1
Pipeline mode   : FINAL_CHAMPION
Best model      : XGBoost
Pipeline        : team02-diabetes-risk
Package group   : team02-diabetes-risk
Endpoint name   : team02-diabetes-risk
MLflow App       : arn:aws:sagemaker:ap-southeast-1:044528205969:mlflow-app/app-YHNC33FF6GPZ
MLflow status    : Created
MLflow server    : 3.10.1
MLflow client    : 3.15.1
MLflow experiment: ITI113/team02/Experiment1

FINAL-CHAMPION MLOPS HANDOFF
Model              : XGBoost
Candidate run      : 02807641d2814f48a084c7c451414917
Refinement run     : e89987fd9aec4a11a1108c8afaf6561b
Threshold run      : a1a47bcc23d844539d92f3a

### Resource naming convention

This version uses one best-model path:

- Pipeline: `team02-diabetes-risk`
- Processing: `team02-preprocess-data`
- Training: `team02-train-model`
- Evaluation: `team02-evaluate-model`
- Registration: `team02-register-model`
- Progress Model Package Group: `team02-diabetes-risk-progress`
- Progress endpoint: `team02-diabetes-risk-test`
- Final Model Package Group: `team02-diabetes-risk`
- Final endpoint: `team02-diabetes-risk`

No LR-specific and XGBoost-specific parallel registries/endpoints are used.


## 3. Verify Exact Experiment Dataset and Split Lineage

The final MLOps pipeline consumes the **same cleaned binary S3 dataset URI and full-profile row count** recorded by the latest Logistic Regression and XGBoost exports.

Before any SageMaker job is created, this preflight verifies the operational lineage anchors and separates CSV-export evidence from direct S3 revalidation:
- S3 source URI;
- exact 253,680-row full profile;
- target and feature schema;
- no multiclass `Diabetes_012` leakage column;
- stable full-profile SHA-256 signature.

The managed preprocessing step then recreates the exact deterministic **80/20 stratified split** and verifies the logged train/test split signatures before training can continue.


> **Schema-order correction:** the cleaned experiment CSV stores the 21 predictor
> columns first and `Diabetes_binary` last. The lineage checks intentionally preserve
> that source order because `hash_pandas_object` is column-order-sensitive.

**Evidence provenance note:** the latest exports record the common S3 source, full-profile size, CV design and final run lineage. The XGBoost final export also records `comparison_test_split_sha256`. They do **not** export the full-profile SHA-256 or training-split SHA-256; those values are retained as Notebook 03 operational lineage anchors and independently recomputed from S3.


In [9]:
import hashlib

def parse_s3_uri(uri: str):
    p = urlparse(uri)
    if p.scheme != "s3":
        raise ValueError(f"Invalid S3 URI: {uri}")
    return p.netloc, p.path.lstrip("/")

def dataframe_sha256(frame: pd.DataFrame) -> str:
    """Same stable dataframe signature used by the Model B experiment notebook."""
    hashed = pd.util.hash_pandas_object(frame, index=False).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()

DATA_VERSION = approved["data_version"]
RAW_DATA_URI = approved["data_source_s3_uri"]  # compatibility name for downstream cells

source_bucket, source_key = parse_s3_uri(RAW_DATA_URI)
source_bytes = s3.get_object(Bucket=source_bucket, Key=source_key)["Body"].read()
source_df = pd.read_csv(io.BytesIO(source_bytes)).reset_index(drop=True)

# The cleaned S3 dataset used by the modelling experiments stores the
# 21 predictors first and Diabetes_binary as the final column.
# Keep this exact source order because the logged dataframe SHA-256 is order-sensitive.
expected_columns = FEATURE_COLUMNS + [SOURCE_TARGET_COLUMN]

if list(source_df.columns) != expected_columns:
    raise ValueError(
        "Cleaned experiment dataset schema/order mismatch. "
        f"Expected {expected_columns}; found {list(source_df.columns)}"
    )

if "Diabetes_012" in source_df.columns:
    raise ValueError(
        "Leakage risk: Diabetes_012 must not be present in the cleaned binary modelling dataset."
    )

if len(source_df) != EXPERIMENT_PROFILE_ROWS:
    raise ValueError(
        f"Expected {EXPERIMENT_PROFILE_ROWS:,} full-profile rows; found {len(source_df):,}."
    )

target_values = sorted(source_df[SOURCE_TARGET_COLUMN].dropna().unique().tolist())
if target_values != [0, 1]:
    raise ValueError(
        f"Expected binary target values [0, 1]; found {target_values}."
    )

observed_profile_sha256 = dataframe_sha256(source_df)
if observed_profile_sha256 != EXPERIMENT_PROFILE_SHA256:
    raise ValueError(
        "Full-profile dataframe signature differs from the frozen Notebook 03 operational lineage anchor. "
        f"expected={EXPERIMENT_PROFILE_SHA256}, observed={observed_profile_sha256}"
    )

print("EXPERIMENT DATA LINEAGE PREFLIGHT: PASS")
print("=" * 96)
print("Data version       :", DATA_VERSION)
print("S3 source          :", RAW_DATA_URI)
print("Rows               :", f"{len(source_df):,}")
print("Feature count      :", len(FEATURE_COLUMNS))
print("Target             :", SOURCE_TARGET_COLUMN)
print("Profile SHA-256    :", observed_profile_sha256)
print("Expected train SHA :", EXPERIMENT_TRAIN_SPLIT_SHA256)
print("Expected test SHA  :", EXPERIMENT_TEST_SPLIT_SHA256)
print("Latest CSV test SHA:", COMMON_DATA_LINEAGE["latest_csv_test_split_sha256"])
print("Hash provenance    : profile/train = S3-revalidated lineage anchors; test hash also appears in latest XGB export")
print("Split design       : stratified 80/20, random_state=42")
print("Validation design  : Stratified 5-Fold CV on training set in modelling notebooks")
print("=" * 96)

# Free the full dataframe from the notebook kernel; the managed Processing job
# reloads and validates the same S3 source independently.
del source_df, source_bytes


EXPERIMENT DATA LINEAGE PREFLIGHT: PASS
Data version       : brfss2015-diabetes-binary-6244bec277fe
S3 source          : s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv
Rows               : 253,680
Feature count      : 21
Target             : Diabetes_binary
Profile SHA-256    : 6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5
Expected train SHA : 93531a7b366471a63d29fd970abad6c5d04a2f933f9ffb2aa38dce1c7b286d65
Expected test SHA  : 12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1
Latest CSV test SHA: 12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1
Hash provenance    : profile/train = S3-revalidated lineage anchors; test hash also appears in latest XGB export
Split design       : stratified 80/20, random_state=42
Validation design  : Stratified 5-Fold CV on training set in modelling notebooks


## 4. Verify the Frozen Selected-Champion Configuration

This check confirms that the runtime model type, exact parameters, source run lineage and decision threshold match the champion selected in Section 0A. It also prevents a future change in experiment evidence from silently reusing an obsolete XGBoost deployment configuration.

In [10]:
approved = ensure_runtime_selection()

CHAMPION_MODEL_TYPE = approved["champion_model_type"]
DECISION_THRESHOLD = float(approved["decision_threshold"])
MODEL_PARAMS = dict(selected_handoff["model_params"])

if CHAMPION_MODEL_TYPE != BEST_MODEL_TYPE:
    raise RuntimeError(
        "Runtime champion type differs from the cross-model selection result."
    )

# Current pipeline implementation is XGBoost-specific. Because the latest
# evidence selects XGBoost, this check should pass. If a future comparison
# selects LR, stop rather than deploying the wrong family.
if CHAMPION_MODEL_TYPE != "XGBoost":
    raise RuntimeError(
        f"Selected champion is {CHAMPION_MODEL_TYPE}, but this pipeline's current "
        "trainer/serving path supports XGBoost. Implement the selected family first."
    )

expected_params = BEST_MODEL_CONFIG["model_params"]
if MODEL_PARAMS != expected_params:
    raise RuntimeError("Runtime model parameters differ from the selected champion configuration.")

if DECISION_THRESHOLD != float(BEST_MODEL_CONFIG["decision_threshold"]):
    raise RuntimeError("Runtime threshold differs from the selected champion threshold.")

print("FINAL-CHAMPION CONFIGURATION CHECK: PASS")
print(json.dumps({
    "selection_primary_metric": CHAMPION_PRIMARY_METRIC_LABEL,
    "model_type": CHAMPION_MODEL_TYPE,
    "decision_threshold": DECISION_THRESHOLD,
    "source_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
    "source_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID,
    "source_threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
    "source_final_run_id": CHAMPION_FINAL_RUN_ID,
    "runner_up_model": RUNNER_UP_MODEL_TYPE,
    "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
    "model_params": MODEL_PARAMS,
}, indent=2))

FINAL-CHAMPION CONFIGURATION CHECK: PASS
{
  "selection_primary_metric": "held-out PR-AUC",
  "model_type": "XGBoost",
  "decision_threshold": 0.5,
  "source_candidate_run_id": "02807641d2814f48a084c7c451414917",
  "source_refinement_run_id": "e89987fd9aec4a11a1108c8afaf6561b",
  "source_threshold_tuning_run_id": "a1a47bcc23d844539d92f3a3ed8fec1e",
  "source_final_run_id": "057bb58d81564c0db9a628b224eab8f9",
  "runner_up_model": "LogisticRegression",
  "comparison_lr_final_run_id": "82bc9a4720e04a50ae942128305eb90d",
  "model_params": {
    "n_estimators": 450,
    "max_depth": 4,
    "learning_rate": 0.03,
    "min_child_weight": 5.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 10.0,
    "gamma": 0.05,
    "scale_pos_weight": 3.0,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "tree_method": "hist",
    "n_jobs": -1,
    "random_state": 42
  }
}


### 4.1 Log Complete Champion-Selection and Experiment-to-MLOps Handoff to MLflow

The handoff logs **both model-family finalists**, the frozen champion rule, the selected champion, exact source run IDs, parameters, thresholds, data/split lineage and generated artifacts. This provides an auditable bridge from Model Development to MLOps.

In [11]:
BEST_MODEL_HANDOFF_MLFLOW_RUN_ID = None

selection_df = pd.DataFrame(
    [
        {"model_type": model_type, **values}
        for model_type, values in MODEL_SELECTION_EVIDENCE.items()
    ]
)
display(selection_df)

selection_df.to_csv(
    "best_model_experiment_comparison.csv",
    index=False,
)

parameter_handoff_df = pd.DataFrame([
    {
        "model_type": "XGBoost",
        "operational_role": "champion" if BEST_MODEL_TYPE == "XGBoost" else "runner_up",
        "candidate_run_id": XGB_CANDIDATE_RUN_ID,
        "final_run_id": XGB_FINAL_RUN_ID,
        "decision_threshold": XGB_THRESHOLD,
        **XGB_MODEL_PARAMS,
    },
    {
        "model_type": "LogisticRegression",
        "operational_role": "champion" if BEST_MODEL_TYPE == "LogisticRegression" else "runner_up",
        "candidate_run_id": LR_CANDIDATE_RUN_ID,
        "final_run_id": LR_FINAL_COMPARISON_RUN_ID,
        "decision_threshold": LR_THRESHOLD,
        **LR_MODEL_PARAMS,
    },
])
display(parameter_handoff_df)

parameter_handoff_df.to_csv(
    "model_hyperparameter_handoff.csv",
    index=False,
)

PREPROCESSING_CONFIG_VERSION = "brfss-binary-v1-stratified-80-20-rs42"

PREPROCESSING_CONFIG = {
    "config_version": PREPROCESSING_CONFIG_VERSION,
    "source_target_column": SOURCE_TARGET_COLUMN,
    "pipeline_target_column": TARGET_COLUMN,
    "feature_columns": FEATURE_COLUMNS,
    "feature_count": len(FEATURE_COLUMNS),
    "data_profile": EXPERIMENT_DATA_PROFILE,
    "profile_fraction": EXPERIMENT_PROFILE_FRACTION,
    "split_strategy": "stratified_train_test_split",
    "test_size": EXPERIMENT_TEST_SIZE,
    "random_state": EXPERIMENT_RANDOM_STATE,
    "modelling_validation": f"Stratified {EXPERIMENT_CV_FOLDS}-Fold CV on training only",
    "resampling": "none",
    "xgboost_class_imbalance_control": "scale_pos_weight",
}

reproducibility_manifest = {
    "team_id": TEAM_ID,
    "project_name": PROJECT_NAME,
    "config_version": BEST_MODEL_CONFIG_VERSION,
    "selected_model": BEST_MODEL_TYPE,
    "model": {
        "params": MODEL_PARAMS,
        "decision_threshold": DECISION_THRESHOLD,
        "candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
        "refinement_run_id": CHAMPION_REFINEMENT_RUN_ID,
        "threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
        "final_run_id": CHAMPION_FINAL_RUN_ID,
        "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
        "comparison_lr": {
            "candidate_run_id": LR_CANDIDATE_RUN_ID,
            "threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
            "final_run_id": LR_FINAL_COMPARISON_RUN_ID,
            "decision_threshold": LR_THRESHOLD,
            "params": LR_MODEL_PARAMS,
        },
    },
    "data": {
        "data_version": DATA_VERSION,
        "s3_uri": RAW_DATA_URI,
        "profile_rows": EXPERIMENT_PROFILE_ROWS,
        "profile_sha256": EXPERIMENT_PROFILE_SHA256,
        "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
        "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
        "signature_type": approved["dataset_signature_type"],
    },
    "preprocessing": PREPROCESSING_CONFIG,
    "source_evidence_exports": SOURCE_EVIDENCE_EXPORTS,
    "source_evidence_manifest_file": "source_experiment_evidence_manifest.json",
    "source_evidence_verification_file": "source_experiment_evidence_verification.json",
    "selection_rule": BEST_MODEL_CONFIG["champion_rule"],
    "quality_gate": BEST_MODEL_CONFIG["quality_gate"],
}

Path("champion_model_handoff.json").write_text(
    json.dumps(
        {
            "team_id": TEAM_ID,
            "selected_model": BEST_MODEL_TYPE,
            "config_version": BEST_MODEL_CONFIG_VERSION,
            "selection_source": BEST_MODEL_SELECTION_SOURCE,
            "experiment_comparison": MODEL_SELECTION_EVIDENCE,
            "comparison_logistic_regression": LR_BEST_CONFIG,
            "best_model": BEST_MODEL_CONFIG,
        },
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

Path("team02_reproducibility_manifest.json").write_text(
    json.dumps(reproducibility_manifest, indent=2, default=str),
    encoding="utf-8",
)

mlflow.set_tracking_uri(MLFLOW_APP_ARN)
mlflow.set_experiment(MLFLOW_EXPERIMENT)

with mlflow.start_run(
    run_name=(
        f"{TEAM_ID}_{STUDENT_ID}_final_champion_mlops_handoff_"
        f"{int(time.time())}"
    )
) as handoff_run:
    mlflow.set_tags({
        "course": COURSE,
        "semester": SEMESTER,
        "team_id": TEAM_ID,
        "student_id": STUDENT_ID,
        "project_name": PROJECT_NAME,
        "run_type": "mlops_final_champion_handoff",
        "stage": "final_model_integration",
        "selected_model_type": BEST_MODEL_TYPE,
        "config_version": BEST_MODEL_CONFIG_VERSION,
        "human_oversight": "human_over_the_loop",
        "selection_primary_metric": "held_out_pr_auc",
    })

    mlflow.log_params({
        "selected_model": BEST_MODEL_TYPE,
        "champion_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
        "champion_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID or "n/a",
        "champion_threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
        "champion_final_run_id": CHAMPION_FINAL_RUN_ID,
        "comparison_lr_candidate_run_id": LR_CANDIDATE_RUN_ID,
        "comparison_lr_threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
        "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
        "comparison_lr_decision_threshold": LR_THRESHOLD,
        "decision_threshold": DECISION_THRESHOLD,
        "data_version": DATA_VERSION,
        "data_source_s3_uri": RAW_DATA_URI,
        "profile_rows": EXPERIMENT_PROFILE_ROWS,
        "profile_sha256": EXPERIMENT_PROFILE_SHA256,
        "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
        "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
        "split_strategy": "stratified_80_20_random_state_42",
        "cv_folds": EXPERIMENT_CV_FOLDS,
        "preprocessing_config_version": PREPROCESSING_CONFIG_VERSION,
        "xgb_evidence_export_sha256": SOURCE_EVIDENCE_EXPORTS["XGBoost"]["sha256"],
        "lr_evidence_export_sha256": SOURCE_EVIDENCE_EXPORTS["LogisticRegression"]["sha256"],
        "model_version_identity": CHAMPION_FINAL_RUN_ID,
        **{f"xgb_{k}": v for k, v in XGB_MODEL_PARAMS.items()},
        **{f"lr_{k}": v for k, v in LR_MODEL_PARAMS.items()},
    })

    xgb_metrics = MODEL_SELECTION_EVIDENCE["XGBoost"]
    lr_metrics = MODEL_SELECTION_EVIDENCE["LogisticRegression"]

    metrics_to_log = {
        "xgb_test_pr_auc": xgb_metrics["pr_auc"],
        "xgb_test_roc_auc": xgb_metrics["roc_auc"],
        "xgb_test_f1": xgb_metrics["f1"],
        "xgb_test_recall": xgb_metrics["recall"],
        "xgb_test_precision": xgb_metrics["precision"],
        "lr_test_pr_auc": lr_metrics["pr_auc"],
        "lr_test_roc_auc": lr_metrics["roc_auc"],
        "lr_test_f1": lr_metrics["f1"],
        "lr_test_recall": lr_metrics["recall"],
        "lr_test_precision": lr_metrics["precision"],
        "xgb_minus_lr_test_pr_auc": xgb_metrics["pr_auc"] - lr_metrics["pr_auc"],
        "selected_candidate_cv_pr_auc": BEST_MODEL_CONFIG["selection_evidence"]["candidate_cv_pr_auc_mean"],
        "selected_refined_cv_pr_auc": BEST_MODEL_CONFIG["selection_evidence"]["refined_cv_pr_auc_mean"],
        "lr_selected_candidate_cv_pr_auc": LR_BEST_CONFIG["selection_evidence"]["candidate_cv_pr_auc_mean"],
        "lr_selected_candidate_cv_roc_auc": LR_BEST_CONFIG["selection_evidence"]["candidate_cv_roc_auc_mean"],
        "lr_selected_candidate_cv_f1": LR_BEST_CONFIG["selection_evidence"]["candidate_cv_f1_mean"],
    }
    mlflow.log_metrics(metrics_to_log)

    for artifact_name in [
        "best_model_experiment_comparison.csv",
        "model_hyperparameter_handoff.csv",
        "champion_model_handoff.json",
        "team02_champion_model_config.json",
        "team02_reproducibility_manifest.json",
        "source_experiment_evidence_manifest.json",
        "source_experiment_evidence_verification.json",
    ]:
        mlflow.log_artifact(
            artifact_name,
            artifact_path="final_champion_handoff",
        )

    BEST_MODEL_HANDOFF_MLFLOW_RUN_ID = handoff_run.info.run_id

MODEL_HANDOFF_MLFLOW_RUN_ID = BEST_MODEL_HANDOFF_MLFLOW_RUN_ID

print("Final-champion handoff MLflow run:", BEST_MODEL_HANDOFF_MLFLOW_RUN_ID)
print("Reproducibility manifest: team02_reproducibility_manifest.json")


,model_type,final_run_id,candidate_name,candidate_run_id,threshold_tuning_run_id,decision_threshold,accuracy,precision,recall,f1,pr_auc,roc_auc,balanced_accuracy,refinement_run_id,f2,brier
0,LogisticRegression,82bc9a4720e04a50ae942128305eb90d,lr_candidate_12,74e8884c230349f3ad29021db15f54a2,4d0a1f65fd754905bb85a4459886cb10,0.59,0.777594,0.380738,0.656660,0.482005,0.432749,0.817565,0.728438,NaN,NaN,NaN
1,XGBoost,057bb58d81564c0db9a628b224eab8f9,xgb_candidate_29,02807641d2814f48a084c7c451414917,a1a47bcc23d844539d92f3a3ed8fec1e,0.50,0.798151,0.407815,0.621388,0.492442,0.464397,0.824820,0.726302,e89987fd9aec4a11a1108c8afaf6561b,0.562475,0.133926


,model_type,operational_role,candidate_run_id,final_run_id,decision_threshold,n_estimators,max_depth,learning_rate,min_child_weight,subsample,...,objective,eval_metric,tree_method,n_jobs,random_state,C,class_weight,solver,penalty,max_iter
0,XGBoost,champion,02807641d2814f48a084c7c451414917,057bb58d81564c0db9a628b224eab8f9,0.50,450.0,4.0,0.03,5.0,0.8,...,binary:logistic,aucpr,hist,-1.0,42,NaN,NaN,NaN,NaN,NaN
1,LogisticRegression,runner_up,74e8884c230349f3ad29021db15f54a2,82bc9a4720e04a50ae942128305eb90d,0.59,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,42,100.0,balanced,lbfgs,l2,3000.0


🏃 View run team02_s201_final_champion_mlops_handoff_1787377739 at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1/runs/43aba870c78a4b26ab068ad2454d3bf2
🧪 View experiment at: https://mlflow.sagemaker.ap-southeast-1.app.aws/#/experiments/1
Final-champion handoff MLflow run: 43aba870c78a4b26ab068ad2454d3bf2
Reproducibility manifest: team02_reproducibility_manifest.json


### 4.2 Experiment-to-MLOps Transfer Rule

**MLOps does not choose a different model or retune the champion.** The champion selection occurs once from the frozen final comparison evidence. The pipeline receives the selected family's exact hyperparameters, decision threshold, source run IDs and data lineage.

For the current evidence this means the operational configuration is XGBoost. Logistic Regression remains the runner-up/challenger evidence and is not trained or deployed by Notebook 03.

## 5. Build Reusable SageMaker Pipeline Source Scripts

Create the standalone scripts executed inside SageMaker Processing, Training and inference environments. Keeping these scripts as explicit artifacts improves reproducibility, traceability and reuse across pipeline executions.

### 5.1 Prepare the Pipeline Source Directory

Create the local `diabetes_pipeline_src` directory used to package the preprocessing, training, evaluation and inference scripts for SageMaker jobs.

In [12]:
SRC_DIR = Path("diabetes_pipeline_src")
SRC_DIR.mkdir(parents=True, exist_ok=True)
print("Source directory:", SRC_DIR.resolve())

Source directory: /home/sagemaker-user/workspace/s201_1552444F/FinalReport/diabetes_pipeline_src


### 5.2 Create the Experiment-aligned Data Preprocessing Script

The Processing job validates the exact cleaned binary dataset used in modelling, recreates the deterministic stratified 80/20 split, and refuses to continue unless the **full-profile, training-split and test-split SHA-256 signatures exactly match the frozen Notebook 03 lineage anchors**. The test-split signature is also cross-checked against the latest XGBoost final export; the profile/train signatures are recomputed from S3 because those fields are not present in the latest CSV exports.

This removes the previous 70/15/15-vs-80/20 MLOps/model-development mismatch and makes the operational retraining path reproducible.


In [13]:
%%writefile diabetes_pipeline_src/preprocess.py
"""SageMaker Processing Job — reproduce the final modelling dataset and 80/20 split exactly."""
import argparse
import glob
import hashlib
import json
import os

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

parser = argparse.ArgumentParser()
parser.add_argument("--random-state", type=int, default=42)
parser.add_argument("--test-size", type=float, default=0.20)
parser.add_argument("--expected-profile-sha256", required=True)
parser.add_argument("--expected-train-sha256", required=True)
parser.add_argument("--expected-test-sha256", required=True)
parser.add_argument("--expected-rows", type=int, required=True)
args = parser.parse_args()

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
SOURCE_TARGET = "Diabetes_binary"
TARGET = "target"
# Exact schema/order of the cleaned binary experiment dataset:
# predictors first, target last. This must match the source dataframe used
# to generate the logged profile/train/test lineage hashes.
EXPECTED_COLUMNS = FEATURE_COLUMNS + [SOURCE_TARGET]

BINARY_FEATURES = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]
ALLOWED_DOMAINS = {
    SOURCE_TARGET: {0, 1},
    **{c: {0, 1} for c in BINARY_FEATURES},
    "GenHlth": set(range(1, 6)),
    "MentHlth": set(range(0, 31)),
    "PhysHlth": set(range(0, 31)),
    "Age": set(range(1, 14)),
    "Education": set(range(1, 7)),
    "Income": set(range(1, 9)),
}

def dataframe_sha256(frame: pd.DataFrame) -> str:
    hashed = pd.util.hash_pandas_object(frame, index=False).values.tobytes()
    return hashlib.sha256(hashed).hexdigest()

input_dir = "/opt/ml/processing/input"
csv_files = glob.glob(os.path.join(input_dir, "**", "*.csv"), recursive=True)
if len(csv_files) != 1:
    raise ValueError(f"Expected exactly one CSV input; found {csv_files}")

df = pd.read_csv(csv_files[0]).reset_index(drop=True)

if len(df) != args.expected_rows:
    raise ValueError(
        f"Incoming row count {len(df)} differs from approved full-profile row count "
        f"{args.expected_rows}."
    )

if list(df.columns) != EXPECTED_COLUMNS:
    raise ValueError(
        f"Schema mismatch. Expected {EXPECTED_COLUMNS}; found {list(df.columns)}"
    )

if "Diabetes_012" in df.columns:
    raise ValueError("Leakage control failed: Diabetes_012 is present.")

if df.isna().sum().sum() != 0:
    raise ValueError("Conventional missing values detected. Review the incoming data.")

for c in EXPECTED_COLUMNS:
    numeric = pd.to_numeric(df[c], errors="coerce")
    if numeric.isna().any():
        raise ValueError(f"Non-numeric values detected in {c}.")
    if c != "BMI" and not np.allclose(numeric, np.round(numeric)):
        raise ValueError(f"Non-integer-coded values detected in {c}.")
    df[c] = numeric

for c, allowed in ALLOWED_DOMAINS.items():
    observed = set(df[c].astype(int).unique().tolist())
    unexpected = observed - allowed
    if unexpected:
        raise ValueError(f"Unexpected coded values in {c}: {sorted(unexpected)}")

if not np.isfinite(df["BMI"]).all() or (df["BMI"] <= 0).any():
    raise ValueError("BMI must contain finite positive values.")

profile_sha256 = dataframe_sha256(df)
if profile_sha256 != args.expected_profile_sha256:
    raise ValueError(
        "Full-profile signature mismatch. "
        f"expected={args.expected_profile_sha256}, observed={profile_sha256}"
    )

X = df[FEATURE_COLUMNS].copy()
y = df[SOURCE_TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=args.test_size,
    random_state=args.random_state,
    stratify=y,
)

X_train = X_train.reset_index(drop=True)
X_test = X_test.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)

train_for_hash = pd.concat(
    [X_train, y_train.rename(SOURCE_TARGET)],
    axis=1,
)
test_for_hash = pd.concat(
    [X_test, y_test.rename(SOURCE_TARGET)],
    axis=1,
)

train_sha256 = dataframe_sha256(train_for_hash)
test_sha256 = dataframe_sha256(test_for_hash)

if train_sha256 != args.expected_train_sha256:
    raise ValueError(
        "Training-split signature mismatch. "
        f"expected={args.expected_train_sha256}, observed={train_sha256}"
    )
if test_sha256 != args.expected_test_sha256:
    raise ValueError(
        "Test-split signature mismatch. "
        f"expected={args.expected_test_sha256}, observed={test_sha256}"
    )

train_model = pd.concat(
    [X_train, y_train.rename(TARGET)],
    axis=1,
)
test_model = pd.concat(
    [X_test, y_test.rename(TARGET)],
    axis=1,
)

output_root = "/opt/ml/processing/output"
train_dir = os.path.join(output_root, "train")
test_dir = os.path.join(output_root, "test")
metadata_dir = os.path.join(output_root, "metadata")
os.makedirs(train_dir, exist_ok=True)
os.makedirs(test_dir, exist_ok=True)
os.makedirs(metadata_dir, exist_ok=True)

train_model.to_csv(os.path.join(train_dir, "train.csv"), index=False)
test_model.to_csv(os.path.join(test_dir, "test.csv"), index=False)

lineage = {
    "source_file": os.path.basename(csv_files[0]),
    "rows": int(len(df)),
    "features": FEATURE_COLUMNS,
    "source_target": SOURCE_TARGET,
    "pipeline_target": TARGET,
    "split_strategy": "stratified 80/20 train/test split",
    "test_size": float(args.test_size),
    "random_state": int(args.random_state),
    "modelling_validation": "Stratified 5-Fold CV on training only",
    "profile_sha256": profile_sha256,
    "train_split_sha256": train_sha256,
    "test_split_sha256": test_sha256,
    "train_rows": int(len(train_model)),
    "test_rows": int(len(test_model)),
    "train_positive_rate": float(train_model[TARGET].mean()),
    "test_positive_rate": float(test_model[TARGET].mean()),
}

with open(os.path.join(metadata_dir, "lineage.json"), "w", encoding="utf-8") as f:
    json.dump(lineage, f, indent=2)

print("Experiment-aligned preprocessing complete.")
print(json.dumps(lineage, indent=2))


Overwriting diabetes_pipeline_src/preprocess.py


### 5.3 Create the Frozen Selected-Champion Training Script (Current Winner: XGBoost)

Training uses the exact final-champion XGBoost parameters transferred from Model B. It fits the model on the **entire 80% training partition** after hyperparameter/threshold selection has already been completed upstream.

No new validation set is carved out here, because doing so would change the training rows and break exact reproduction of the final modelling experiment.


In [14]:
%%writefile diabetes_pipeline_src/train.py
"""Train the frozen Team02 selected champion; current cross-model winner is XGBoost."""
import argparse
import json
import os
import pickle
import platform
import shutil

import numpy as np
import pandas as pd
import sklearn

parser = argparse.ArgumentParser()
parser.add_argument("--model-type", required=True)
parser.add_argument("--decision-threshold", type=float, required=True)
parser.add_argument("--model-params-json", required=True)
parser.add_argument("--preprocessing-config-json", required=True)
parser.add_argument("--team-id", required=True)
parser.add_argument("--data-version", required=True)
parser.add_argument("--dataset-sha256", required=True)
parser.add_argument("--approved-run-id", required=True)
parser.add_argument("--source-candidate-run-id", required=True)
parser.add_argument("--source-refinement-run-id", required=True)
parser.add_argument("--source-threshold-run-id", required=True)
parser.add_argument("--config-version", required=True)
parser.add_argument("--experiment-xgboost-version", default="")
parser.add_argument("--pipeline-xgboost-version", required=True)
parser.add_argument("--model-dir", default=os.environ.get("SM_MODEL_DIR", "/opt/ml/model"))
parser.add_argument("--train", default=os.environ.get("SM_CHANNEL_TRAIN", "/opt/ml/input/data/train"))
args = parser.parse_args()

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
TARGET = "target"

if args.model_type != "XGBoost":
    raise ValueError("This final-champion training script supports XGBoost only.")

train_path = os.path.join(args.train, "train.csv")
if not os.path.exists(train_path):
    raise FileNotFoundError(train_path)

train_df = pd.read_csv(train_path)
expected_model_columns = FEATURE_COLUMNS + [TARGET]
if list(train_df.columns) != expected_model_columns:
    raise ValueError(f"Unexpected training schema: {list(train_df.columns)}")

X_train = train_df[FEATURE_COLUMNS]
y_train = train_df[TARGET].astype(int)

params = json.loads(args.model_params_json)
preprocessing_config = json.loads(args.preprocessing_config_json)

import xgboost
from xgboost import XGBClassifier

if xgboost.__version__ != str(args.pipeline_xgboost_version):
    raise RuntimeError(
        f"xgboost runtime mismatch: installed={xgboost.__version__}, "
        f"expected={args.pipeline_xgboost_version}"
    )

def _xgb_int(value):
    return int(float(str(value).strip("'\\\"")))

required_params = [
    "n_estimators", "max_depth", "learning_rate", "min_child_weight",
    "subsample", "colsample_bytree", "reg_alpha", "reg_lambda", "gamma",
    "scale_pos_weight", "objective", "eval_metric", "tree_method",
    "n_jobs", "random_state",
]
missing_params = [k for k in required_params if k not in params]
if missing_params:
    raise ValueError("Missing frozen XGBoost parameters: " + ", ".join(missing_params))

xgb_params = {
    "n_estimators": _xgb_int(params["n_estimators"]),
    "max_depth": _xgb_int(params["max_depth"]),
    "learning_rate": float(params["learning_rate"]),
    "min_child_weight": float(params["min_child_weight"]),
    "subsample": float(params["subsample"]),
    "colsample_bytree": float(params["colsample_bytree"]),
    "reg_alpha": float(params["reg_alpha"]),
    "reg_lambda": float(params["reg_lambda"]),
    "gamma": float(params["gamma"]),
    "scale_pos_weight": float(params["scale_pos_weight"]),
    "tree_method": params["tree_method"],
    "random_state": _xgb_int(params["random_state"]),
    "objective": params["objective"],
    "eval_metric": params["eval_metric"],
    "n_jobs": _xgb_int(params["n_jobs"]),
}

model = XGBClassifier(**xgb_params)

X_train_np = X_train.to_numpy(dtype=np.float32, copy=False)
y_train_np = y_train.to_numpy(dtype=np.int32, copy=False)

print("Training final-champion XGBoost")
print("Rows                    :", len(train_df))
print("Features                :", len(FEATURE_COLUMNS))
print("Pipeline xgboost        :", xgboost.__version__)
print("Experiment xgboost note :", args.experiment_xgboost_version or "NOT_LOGGED_IN_CSV")
print("Frozen parameters:")
print(json.dumps(xgb_params, indent=2))

model.fit(X_train_np, y_train_np)

os.makedirs(args.model_dir, exist_ok=True)

bundle = {
    "model": model,
    "model_type": args.model_type,
    "model_params": xgb_params,
    "decision_threshold": args.decision_threshold,
    "feature_columns": FEATURE_COLUMNS,
    "team_id": args.team_id,
    "data_version": args.data_version,
    "dataset_sha256": args.dataset_sha256,
    "approved_mlflow_run_id": args.approved_run_id,
    "source_candidate_run_id": args.source_candidate_run_id,
    "source_refinement_run_id": args.source_refinement_run_id,
    "source_threshold_run_id": args.source_threshold_run_id,
    "config_version": args.config_version,
    "preprocessing_config": preprocessing_config,
    "experiment_xgboost_version": args.experiment_xgboost_version or None,
    "pipeline_xgboost_version": args.pipeline_xgboost_version,
}

with open(os.path.join(args.model_dir, "model.pkl"), "wb") as f:
    pickle.dump(bundle, f)

# Package custom inference code with the model artifact.
script_dir = os.path.dirname(os.path.abspath(__file__))
deployment_code_dir = os.path.join(args.model_dir, "code")
os.makedirs(deployment_code_dir, exist_ok=True)

for deployment_file in ("inference.py", "requirements.txt"):
    source_path = os.path.join(script_dir, deployment_file)
    destination_path = os.path.join(deployment_code_dir, deployment_file)
    if not os.path.exists(source_path):
        raise FileNotFoundError(
            f"Required deployment source file is missing: {source_path}"
        )
    shutil.copy2(source_path, destination_path)

environment = {
    "python": platform.python_version(),
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "xgboost": xgboost.__version__,
}

with open(os.path.join(args.model_dir, "model_metadata.json"), "w", encoding="utf-8") as f:
    json.dump(
        {
            "model_type": args.model_type,
            "model_params": xgb_params,
            "decision_threshold": args.decision_threshold,
            "data_version": args.data_version,
            "dataset_sha256": args.dataset_sha256,
            "approved_mlflow_run_id": args.approved_run_id,
            "source_candidate_run_id": args.source_candidate_run_id,
            "source_refinement_run_id": args.source_refinement_run_id,
            "source_threshold_run_id": args.source_threshold_run_id,
            "config_version": args.config_version,
            "preprocessing_config": preprocessing_config,
            "environment": environment,
        },
        f,
        indent=2,
    )

required_deployment_files = [
    os.path.join(args.model_dir, "model.pkl"),
    os.path.join(args.model_dir, "model_metadata.json"),
    os.path.join(deployment_code_dir, "inference.py"),
    os.path.join(deployment_code_dir, "requirements.txt"),
]
missing_deployment_files = [
    path for path in required_deployment_files if not os.path.exists(path)
]
if missing_deployment_files:
    raise RuntimeError(
        "Deployment artifact packaging failed. Missing: "
        + ", ".join(missing_deployment_files)
    )

print("Deployment artifact structure check: PASS")
print("Saved final-champion bundle and metadata.")


Overwriting diabetes_pipeline_src/train.py


### 5.4 Create the Independent Held-Out Regression / Evaluation Script

The evaluation job scores the retrained champion on the **same deterministic 20% held-out partition** used by the latest model-development comparison.

It generates:
- Accuracy, Precision, Recall, F1, F2, Balanced Accuracy, PR-AUC, ROC-AUC and Brier score;
- confusion-matrix counts;
- subgroup Recall and FPR by Sex, Age, Education and Income;
- model/data/config lineage in `evaluation.json`.

This evaluation is a deployment regression check; it does not reopen model or threshold selection.


In [15]:
%%writefile diabetes_pipeline_src/evaluate.py
"""Independent held-out regression/evaluation and governance evidence for the selected champion."""
import glob
import json
import os
import pickle
import tarfile

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    fbeta_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]
FAIRNESS_ATTRIBUTES = ["Sex", "Age", "Education", "Income"]
TARGET = "target"

model_dir = "/opt/ml/processing/model"
test_dir = "/opt/ml/processing/test"
output_dir = "/opt/ml/processing/evaluation"
os.makedirs(output_dir, exist_ok=True)

tar_files = glob.glob(os.path.join(model_dir, "**", "model.tar.gz"), recursive=True)
if len(tar_files) != 1:
    raise FileNotFoundError(f"Expected one model.tar.gz; found {tar_files}")

extract_dir = os.path.join(model_dir, "extracted")
os.makedirs(extract_dir, exist_ok=True)
with tarfile.open(tar_files[0], "r:gz") as tar:
    tar.extractall(extract_dir)

required_artifact_members = [
    "model.pkl",
    "model_metadata.json",
    "code/inference.py",
    "code/requirements.txt",
]
missing_artifact_members = [
    member for member in required_artifact_members
    if not os.path.exists(os.path.join(extract_dir, member))
]
if missing_artifact_members:
    raise RuntimeError(
        "Model artifact is not deployment-ready. Missing files: "
        + ", ".join(missing_artifact_members)
    )

with open(os.path.join(extract_dir, "model.pkl"), "rb") as f:
    bundle = pickle.load(f)

model = bundle["model"]
threshold = float(bundle["decision_threshold"])

test_df = pd.read_csv(os.path.join(test_dir, "test.csv"))
expected_columns = FEATURE_COLUMNS + [TARGET]
if list(test_df.columns) != expected_columns:
    raise ValueError(f"Unexpected test schema: {list(test_df.columns)}")

X_test = test_df[FEATURE_COLUMNS]
y_test = test_df[TARGET].astype(int)
X_test_np = X_test.to_numpy(dtype=np.float32, copy=False)

prob = model.predict_proba(X_test_np)[:, 1]
pred = (prob >= threshold).astype(int)

tn, fp, fn, tp = confusion_matrix(y_test, pred, labels=[0, 1]).ravel()

metrics = {
    "accuracy": float(accuracy_score(y_test, pred)),
    "precision": float(precision_score(y_test, pred, zero_division=0)),
    "recall": float(recall_score(y_test, pred, zero_division=0)),
    "f1": float(f1_score(y_test, pred, zero_division=0)),
    "f2": float(fbeta_score(y_test, pred, beta=2, zero_division=0)),
    "balanced_accuracy": float(balanced_accuracy_score(y_test, pred)),
    "average_precision": float(average_precision_score(y_test, prob)),
    "roc_auc": float(roc_auc_score(y_test, prob)),
    "brier_score": float(brier_score_loss(y_test, prob)),
    "decision_threshold": threshold,
    "true_negative": int(tn),
    "false_positive": int(fp),
    "false_negative": int(fn),
    "true_positive": int(tp),
}

subgroup_rows = []
base = X_test.reset_index(drop=True).copy()
base["_y"] = y_test.to_numpy()
base["_pred"] = pred

for attr in FAIRNESS_ATTRIBUTES:
    for group, g in base.groupby(attr, dropna=False):
        y = g["_y"].to_numpy()
        p = g["_pred"].to_numpy()
        g_tn, g_fp, g_fn, g_tp = confusion_matrix(y, p, labels=[0, 1]).ravel()
        subgroup_rows.append({
            "attribute": attr,
            "group": int(group) if pd.notna(group) else None,
            "n": int(len(g)),
            "positives": int((y == 1).sum()),
            "negatives": int((y == 0).sum()),
            "recall": float(g_tp / (g_tp + g_fn)) if (g_tp + g_fn) else None,
            "false_positive_rate": float(g_fp / (g_fp + g_tn)) if (g_fp + g_tn) else None,
        })

subgroup_df = pd.DataFrame(subgroup_rows)
subgroup_df.to_csv(os.path.join(output_dir, "subgroup_metrics.csv"), index=False)

fairness_summary = {}
for attr, g in subgroup_df.groupby("attribute"):
    valid_recall = g["recall"].dropna()
    valid_fpr = g["false_positive_rate"].dropna()
    fairness_summary[attr] = {
        "recall_max_min_gap": (
            float(valid_recall.max() - valid_recall.min())
            if len(valid_recall) >= 2 else None
        ),
        "false_positive_rate_max_min_gap": (
            float(valid_fpr.max() - valid_fpr.min())
            if len(valid_fpr) >= 2 else None
        ),
        "groups_evaluated": int(len(g)),
    }

report = {
    "model_type": bundle["model_type"],
    "team_id": bundle["team_id"],
    "config_version": bundle["config_version"],
    "model_params": bundle["model_params"],
    "decision_threshold": threshold,
    "data_version": bundle["data_version"],
    "dataset_sha256": bundle["dataset_sha256"],
    "approved_mlflow_run_id": bundle["approved_mlflow_run_id"],
    "source_candidate_run_id": bundle["source_candidate_run_id"],
    "source_refinement_run_id": bundle["source_refinement_run_id"],
    "source_threshold_run_id": bundle["source_threshold_run_id"],
    "preprocessing_config": bundle["preprocessing_config"],
    "experiment_xgboost_version": bundle.get("experiment_xgboost_version"),
    "pipeline_xgboost_version": bundle.get("pipeline_xgboost_version"),
    "test": metrics,
    "fairness_gap_summary": fairness_summary,
    "subgroup_metrics_file": "subgroup_metrics.csv",
}

with open(os.path.join(output_dir, "evaluation.json"), "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print(json.dumps(report, indent=2))


Overwriting diabetes_pipeline_src/evaluate.py


### 5.5 Create the Inference Handler

Validate endpoint inputs, enforce the frozen 21-feature order, apply the promoted threshold **0.50**, and return model/data/config lineage with every prediction.

The endpoint is explicitly a **screening-support academic prototype**, not a diagnostic system.


In [16]:
%%writefile diabetes_pipeline_src/inference.py
"""SageMaker inference handler for the Team 02 diabetes screening-support prototype."""
import json
import os
import pickle

import numpy as np
import pandas as pd

FEATURE_COLUMNS = [
    "HighBP", "HighChol", "CholCheck", "BMI", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income",
]

BINARY = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex",
]

DOMAINS = {
    **{c: {0, 1} for c in BINARY},
    "GenHlth": set(range(1, 6)),
    "MentHlth": set(range(0, 31)),
    "PhysHlth": set(range(0, 31)),
    "Age": set(range(1, 14)),
    "Education": set(range(1, 7)),
    "Income": set(range(1, 9)),
}

def model_fn(model_dir):
    with open(os.path.join(model_dir, "model.pkl"), "rb") as f:
        return pickle.load(f)

def input_fn(body, content_type="application/json"):
    if content_type != "application/json":
        raise ValueError(f"Unsupported content type: {content_type}")

    payload = json.loads(body)
    if isinstance(payload, dict):
        payload = [payload]
    if not isinstance(payload, list) or not payload:
        raise ValueError("Input must be one JSON object or a non-empty list of JSON objects.")

    frame = pd.DataFrame(payload)

    missing = [c for c in FEATURE_COLUMNS if c not in frame.columns]
    extra = [c for c in frame.columns if c not in FEATURE_COLUMNS]

    if missing:
        raise ValueError(f"Missing required features: {missing}")
    if extra:
        raise ValueError(
            f"Unexpected features are not accepted under data-minimisation controls: {extra}"
        )

    frame = frame[FEATURE_COLUMNS].copy()

    for c in FEATURE_COLUMNS:
        frame[c] = pd.to_numeric(frame[c], errors="raise")
        if not np.isfinite(frame[c]).all():
            raise ValueError(f"Non-finite value detected in {c}.")

    for c, allowed in DOMAINS.items():
        bad = ~frame[c].isin(allowed)
        if bad.any():
            raise ValueError(f"Invalid coded value for {c}: {frame.loc[bad, c].tolist()}")

    if (frame["BMI"] <= 0).any():
        raise ValueError("BMI must be positive.")

    return frame

def predict_fn(data, bundle):
    model = bundle["model"]
    threshold = float(bundle["decision_threshold"])

    if bundle["model_type"] != "XGBoost":
        raise ValueError("Unexpected deployed model type; final endpoint must be XGBoost.")
    model_input = data.to_numpy(dtype=np.float32, copy=False)

    probabilities = model.predict_proba(model_input)[:, 1]
    predictions = (probabilities >= threshold).astype(int)
    return predictions, probabilities, bundle

def output_fn(prediction, accept="application/json"):
    if accept not in {"application/json", "*/*"}:
        raise ValueError(f"Unsupported accept type: {accept}")

    predictions, probabilities, bundle = prediction
    rows = []

    for pred, prob in zip(predictions, probabilities):
        rows.append({
            "screening_class": int(pred),
            "screening_category": (
                "prediabetes_or_diabetes_screening_positive"
                if int(pred) == 1
                else "screening_negative"
            ),
            "probability": round(float(prob), 6),
            "decision_threshold": float(bundle["decision_threshold"]),
            "model_type": bundle["model_type"],
            "data_version": bundle["data_version"],
            "model_run_id": bundle["approved_mlflow_run_id"],
            "config_version": bundle.get("config_version"),
            "intended_use_notice": (
                "Academic screening-support prototype only. "
                "This output is not a medical diagnosis and does not rule diabetes in or out."
            ),
        })

    return json.dumps(rows), "application/json"


Overwriting diabetes_pipeline_src/inference.py


### 5.6 Define Reproducible Runtime Dependencies

The exported CSV evidence does not contain an explicit XGBoost package-version field, so Notebook 03 does **not invent experiment-version parity**. The operational SageMaker runtime is pinned to `xgboost==2.1.4` for reproducible deployment. Because the latest CSV cannot prove the experiment package version, this limitation is logged explicitly and the independent held-out quality gate remains the runtime regression control.

The pipeline logs this runtime version so any future mismatch is visible and auditable.


In [17]:
# Dependency handoff for training/evaluation/inference.
# The CSV export does not explicitly log xgboost_version, so experiment
# environment parity is documented separately rather than fabricated.

requirements = []

EXPERIMENT_XGBOOST_VERSION = (
    str(EXPERIMENT_XGBOOST_VERSION)
    if EXPERIMENT_XGBOOST_VERSION
    else None
)

PIPELINE_XGBOOST_VERSION = "2.1.4"
requirements.append(f"xgboost=={PIPELINE_XGBOOST_VERSION}")

Path(SRC_DIR / "requirements.txt").write_text(
    "\n".join(requirements) + "\n",
    encoding="utf-8",
)

print("requirements.txt:")
print((SRC_DIR / "requirements.txt").read_text(encoding="utf-8"))
print("Experiment CSV xgboost version :", EXPERIMENT_XGBOOST_VERSION or "NOT_LOGGED")
print("Pinned pipeline xgboost        :", PIPELINE_XGBOOST_VERSION)
print(
    "Governance note: pipeline runtime is explicitly pinned and logged. "
    "The CSV export alone cannot prove experiment package-version parity."
)


requirements.txt:
xgboost==2.1.4

Experiment CSV xgboost version : NOT_LOGGED
Pinned pipeline xgboost        : 2.1.4
Governance note: pipeline runtime is explicitly pinned and logged. The CSV export alone cannot prove experiment package-version parity.


### 5.7 Verify Experiment and Pipeline Runtime Compatibility

Display the XGBoost version used during experimentation and the version used by the SageMaker pipeline. Any environment difference remains explicit and is controlled through the frozen untouched-test quality gates.

In [18]:
if CHAMPION_MODEL_TYPE == "XGBoost":
    print(
        "Experiment version :",
        EXPERIMENT_XGBOOST_VERSION or "NOT_LOGGED",
    )
    print("Pipeline version   :", PIPELINE_XGBOOST_VERSION)

    if (
        EXPERIMENT_XGBOOST_VERSION
        and EXPERIMENT_XGBOOST_VERSION
        == PIPELINE_XGBOOST_VERSION
    ):
        print("Environment parity : exact")
    elif not EXPERIMENT_XGBOOST_VERSION:
        print(
            "Environment parity : cannot be proven — "
            "experiment version not logged"
        )
        print(
            "Risk treatment      : pipeline runtime is pinned and "
            "validated on the canonical pipeline evaluation set."
        )
    else:
        print(
            "Environment parity : deployment-compatible variant"
        )
else:
    print(
        "Selected model is Logistic Regression; "
        "no XGBoost runtime check is required."
    )


Experiment version : NOT_LOGGED
Pipeline version   : 2.1.4
Environment parity : cannot be proven — experiment version not logged
Risk treatment      : pipeline runtime is pinned and validated on the canonical pipeline evaluation set.


## 6. Define Pipeline Session, Parameters and Execution Paths

Create the SageMaker `PipelineSession`, immutable experiment-lineage parameters and per-execution S3 output paths.

For `FINAL_CHAMPION`, registration uses the **pre-existing ROC-AUC ≥ 0.75 quality gate** already logged by the XGBoost final experiment. Recall/F2/PR-AUC are still logged and reviewed, but are not converted into post-hoc hard blockers.


In [19]:
from sagemaker.core.workflow.pipeline_context import PipelineSession
from sagemaker.core.workflow.parameters import ParameterFloat, ParameterString
from sagemaker.core.workflow.execution_variables import ExecutionVariables
from sagemaker.core.workflow.functions import Join

pipeline_session = PipelineSession(
    boto_session=boto_session,
    default_bucket=BUCKET,
    default_bucket_prefix=f"{PIPELINE_PREFIX}/sdk-artifacts",
)

p_input_data = ParameterString(
    name="InputDataUrl",
    default_value=RAW_DATA_URI,
)

p_expected_profile_sha256 = ParameterString(
    name="ExpectedProfileSHA256",
    default_value=EXPERIMENT_PROFILE_SHA256,
)
p_expected_train_sha256 = ParameterString(
    name="ExpectedTrainSplitSHA256",
    default_value=EXPERIMENT_TRAIN_SPLIT_SHA256,
)
p_expected_test_sha256 = ParameterString(
    name="ExpectedTestSplitSHA256",
    default_value=EXPERIMENT_TEST_SPLIT_SHA256,
)

if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    p_gate_roc_auc = ParameterFloat(
        name="MinTestRocAuc",
        default_value=float(approved["quality_gate_threshold"]),
    )
    print("Final deployment regression gate enabled.")
    print("Metric   :", approved["quality_gate_metric"])
    print("Minimum  :", approved["quality_gate_threshold"])
else:
    print(f"{PIPELINE_RUN_MODE}: final ROC-AUC registration gate is not enabled.")

RUN_ROOT = Join(
    on="/",
    values=[
        f"s3://{BUCKET}/{PIPELINE_PREFIX}/pipeline-runs",
        ExecutionVariables.PIPELINE_EXECUTION_ID,
    ],
)

print("Pipeline input default :", RAW_DATA_URI)
print("Profile SHA-256        :", EXPERIMENT_PROFILE_SHA256)
print("Train split SHA-256    :", EXPERIMENT_TRAIN_SPLIT_SHA256)
print("Test split SHA-256     :", EXPERIMENT_TEST_SPLIT_SHA256)
print("Per-execution output root uses PipelineExecutionId.")

from importlib.metadata import version as _pkg_version
print("SageMaker SDK preflight: V3 modular installation")
print("sagemaker-core         :", _pkg_version("sagemaker-core"))
print("sagemaker-train        :", _pkg_version("sagemaker-train"))
print("sagemaker-serve        :", _pkg_version("sagemaker-serve"))
print("sagemaker-mlops        :", _pkg_version("sagemaker-mlops"))
print("PipelineSession ready  :", type(pipeline_session).__name__)


Final deployment regression gate enabled.
Metric   : test_auc_roc
Minimum  : 0.75
Pipeline input default : s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv
Profile SHA-256        : 6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5
Train split SHA-256    : 93531a7b366471a63d29fd970abad6c5d04a2f933f9ffb2aa38dce1c7b286d65
Test split SHA-256     : 12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1
Per-execution output root uses PipelineExecutionId.
SageMaker SDK preflight: V3 modular installation
sagemaker-core         : 2.20.0
sagemaker-train        : 1.20.0
sagemaker-serve        : 1.20.0
sagemaker-mlops        : 1.20.0
PipelineSession ready  : PipelineSession


## 7. Pipeline Step 1 — Ingest, Validate and Recreate the Experiment Split

`team02-preprocess-data` ingests the exact cleaned S3 source, validates schema/data quality, and recreates the experiment's deterministic **80/20 stratified split**.

The step fails if any of the three logged signatures differ:
1. full profile;
2. training split;
3. held-out test split.


In [20]:
# SDK preflight dependency guard.
if "ProcessingStep" not in globals():
    raise RuntimeError(
        "ProcessingStep is unavailable. Run the SageMaker V3 import preflight first."
    )

from sagemaker.core import image_uris
from sagemaker.core.processing import ScriptProcessor
from sagemaker.core.shapes import (
    ProcessingInput,
    ProcessingS3Input,
    ProcessingOutput,
    ProcessingS3Output,
)
from sagemaker.mlops.workflow.steps import ProcessingStep

sklearn_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    py_version="py3",
    instance_type=PROCESSING_INSTANCE_TYPE,
    image_scope="training",
)

processor = ScriptProcessor(
    image_uri=sklearn_image,
    role=role,
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    command=["python3"],
    base_job_name=f"{TEAM_ID}-diabetes-preprocess",
    sagemaker_session=pipeline_session,
    tags=[
        {"Key": "Course", "Value": COURSE},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "StudentId", "Value": STUDENT_ID},
        {"Key": "ProjectName", "Value": PROJECT_NAME},
    ],
)

process_args = processor.run(
    inputs=[
        ProcessingInput(
            input_name="cleaned-data",
            s3_input=ProcessingS3Input(
                s3_uri=p_input_data,
                local_path="/opt/ml/processing/input",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        )
    ],
    outputs=[
        ProcessingOutput(
            output_name="train",
            s3_output=ProcessingS3Output(
                s3_uri=Join(on="/", values=[RUN_ROOT, "processed", "train"]),
                local_path="/opt/ml/processing/output/train",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="test",
            s3_output=ProcessingS3Output(
                s3_uri=Join(on="/", values=[RUN_ROOT, "processed", "test"]),
                local_path="/opt/ml/processing/output/test",
                s3_upload_mode="EndOfJob",
            ),
        ),
        ProcessingOutput(
            output_name="metadata",
            s3_output=ProcessingS3Output(
                s3_uri=Join(on="/", values=[RUN_ROOT, "metadata"]),
                local_path="/opt/ml/processing/output/metadata",
                s3_upload_mode="EndOfJob",
            ),
        ),
    ],
    code=str(SRC_DIR / "preprocess.py"),
    arguments=[
        "--random-state", str(EXPERIMENT_RANDOM_STATE),
        "--test-size", str(EXPERIMENT_TEST_SIZE),
        "--expected-profile-sha256", p_expected_profile_sha256,
        "--expected-train-sha256", p_expected_train_sha256,
        "--expected-test-sha256", p_expected_test_sha256,
        "--expected-rows", str(EXPERIMENT_PROFILE_ROWS),
    ],
)

step_process = ProcessingStep(
    name="team02-preprocess-data",
    step_args=process_args,
)

print("ProcessingStep defined with exact experiment lineage controls.")


[08/22/26 05:49:01] INFO     SageMaker Python SDK will collect telemetry to help us better ]8;id=11131582;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py\telemetry_logging.py]8;;\:]8;id=11131583;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/telemetry/telemetry_logging.py#304\304]8;;\
                             understand our user's needs, diagnose issues, and deliver                             
                             additional features.                                                                  
                             To opt out of telemetry, please disable via TelemetryOptOut                           
                             parameter in SDK defaults config. For more information, refer                         
                             to                                                                                    
                             https://sagemaker.readthedocs.io/en/stable/overview.html#conf                         
                             iguring-and-using-defaults-with-the-sagemaker-python-sdk.                             

/opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/pipeline_context.py:332: UserWarning: Running within a PipelineSession, there will be No Wait, No Logs, and No Job being started.
  warnings.warn(


ProcessingStep defined with exact experiment lineage controls.


## 8. Pipeline Step 2 — Train the Frozen Selected Champion (Current Winner: XGBoost)

`team02-train-model` fits the promoted XGBoost configuration on the exact 80% experiment training partition and packages:
- `model.pkl`;
- `model_metadata.json`;
- inference handler;
- pinned runtime requirements.


In [21]:
# Prerequisite guard for Step 2.
if "step_process" not in globals():
    raise RuntimeError(
        "Step 2 cannot be defined because 'step_process' does not exist. "
        "Run Section 7 first."
    )

from sagemaker.train import ModelTrainer
from sagemaker.core.training.configs import (
    Compute,
    InputData,
    OutputDataConfig,
    SourceCode,
)
from sagemaker.mlops.workflow.steps import TrainingStep

model_params_json = json.dumps(MODEL_PARAMS, separators=(",", ":"))
preprocessing_config_json = json.dumps(PREPROCESSING_CONFIG, separators=(",", ":"))

train_uri = step_process.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri

training_source = SourceCode(
    source_dir=str(SRC_DIR),
    entry_script="train.py",
    requirements="requirements.txt",
)

trainer = ModelTrainer(
    training_image=sklearn_image,
    source_code=training_source,
    compute=Compute(
        instance_type=TRAINING_INSTANCE_TYPE,
        instance_count=1,
    ),
    role=role,
    base_job_name=f"{TEAM_ID}-diabetes-train-model",
    sagemaker_session=pipeline_session,
    output_data_config=OutputDataConfig(
        s3_output_path=Join(on="/", values=[RUN_ROOT, "training"]),
    ),
    hyperparameters={
        "model-type": CHAMPION_MODEL_TYPE,
        "decision-threshold": DECISION_THRESHOLD,
        "model-params-json": model_params_json,
        "preprocessing-config-json": preprocessing_config_json,
        "team-id": TEAM_ID,
        "data-version": DATA_VERSION,
        "dataset-sha256": approved["dataset_sha256"],
        "approved-run-id": CHAMPION_FINAL_RUN_ID,
        "source-candidate-run-id": CHAMPION_CANDIDATE_RUN_ID,
        "source-refinement-run-id": CHAMPION_REFINEMENT_RUN_ID or "n/a",
        "source-threshold-run-id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
        "config-version": BEST_MODEL_CONFIG_VERSION,
        "experiment-xgboost-version": EXPERIMENT_XGBOOST_VERSION or "",
        "pipeline-xgboost-version": PIPELINE_XGBOOST_VERSION,
    },
    input_data_config=[
        InputData(
            channel_name="train",
            data_source=train_uri,
            content_type="text/csv",
        ),
    ],
)

train_args = trainer.train()

step_train = TrainingStep(
    name="team02-train-model",
    step_args=train_args,
)

print("TrainingStep defined for frozen champion:", CHAMPION_MODEL_TYPE)
print("Source final MLflow run:", CHAMPION_FINAL_RUN_ID)


[08/22/26 05:49:03] INFO     Cannot simulate policies for                                  ]8;id=11131590;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=11131591;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#369\369]8;;\
                             'arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113                         
                             -Team02' (access denied); permission verdict unknown.                                 

                    WARNING  Could not verify permissions for role                         ]8;id=11131597;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py\iam_role_resolver.py]8;;\:]8;id=11131598;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/helper/iam_role_resolver.py#591\591]8;;\
                             'arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113                         
                             -Team02' (caller lacks iam:SimulatePrincipalPolicy).                                  
                             Proceeding with it. If the operation later fails with an                              
                             access-denied error, ensure the role has the required                                 
                             permissions for 'training' (see                                                       
                             IamRoleResolver().get_required_actions('training')) or create                         
                             a dedicated role via                                                                  
                             IamRoleResolver().create_execution_role(role_type='training')                         
                             .                                                                                     

                    INFO     Training image URI:                                               ]8;id=11131605;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=11131606;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#558\558]8;;\
                             121021644041.dkr.ecr.ap-southeast-1.amazonaws.com/sagemaker-sciki                     
                             t-learn:1.2-1-cpu-py3                                                                 

TrainingStep defined for frozen champion: XGBoost
Source final MLflow run: 057bb58d81564c0db9a628b224eab8f9


## 9. Pipeline Step 3 — Evaluate on the Exact Held-Out Test Set

`team02-evaluate-model` performs independent evaluation on the deterministic 20% held-out rows whose signature is frozen in the experiment evidence.


In [22]:
# Prerequisite guard for Step 3.
if "step_train" not in globals():
    raise RuntimeError(
        "Step 3 cannot be defined because 'step_train' does not exist. "
        "Run Section 8 first."
    )

from sagemaker.core.processing import FrameworkProcessor
from sagemaker.core.workflow.properties import PropertyFile

evaluation_report = PropertyFile(
    name="EvaluationReport",
    output_name="evaluation",
    path="evaluation.json",
)

test_uri = step_process.properties.ProcessingOutputConfig.Outputs["test"].S3Output.S3Uri

evaluation_processor = FrameworkProcessor(
    image_uri=sklearn_image,
    role=role,
    instance_type=PROCESSING_INSTANCE_TYPE,
    instance_count=1,
    base_job_name=f"{TEAM_ID}-diabetes-evaluate",
    sagemaker_session=pipeline_session,
    tags=[
        {"Key": "Course", "Value": COURSE},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "StudentId", "Value": STUDENT_ID},
        {"Key": "ProjectName", "Value": PROJECT_NAME},
    ],
)

evaluate_args = evaluation_processor.run(
    code=str(SRC_DIR / "evaluate.py"),
    source_dir=str(SRC_DIR),
    requirements="requirements.txt",
    inputs=[
        ProcessingInput(
            input_name="model",
            s3_input=ProcessingS3Input(
                s3_uri=step_train.properties.ModelArtifacts.S3ModelArtifacts,
                local_path="/opt/ml/processing/model",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
        ProcessingInput(
            input_name="test",
            s3_input=ProcessingS3Input(
                s3_uri=test_uri,
                local_path="/opt/ml/processing/test",
                s3_data_type="S3Prefix",
                s3_input_mode="File",
            ),
        ),
    ],
    outputs=[
        ProcessingOutput(
            output_name="evaluation",
            s3_output=ProcessingS3Output(
                s3_uri=Join(on="/", values=[RUN_ROOT, "evaluation"]),
                local_path="/opt/ml/processing/evaluation",
                s3_upload_mode="EndOfJob",
            ),
        )
    ],
)

step_evaluate = ProcessingStep(
    name="team02-evaluate-model",
    step_args=evaluate_args,
    property_files=[evaluation_report],
)

print("Evaluation ProcessingStep defined.")


Evaluation ProcessingStep defined.


## 10. Pipeline Step 4 — Model Registration with a Pre-existing Quality Gate

For `FINAL_CHAMPION`, the pipeline registers a model only when the independent held-out evaluation satisfies the **ROC-AUC ≥ 0.75** gate already recorded in the XGBoost experiment.

Registration status is always `PendingManualApproval`; passing the technical gate does **not** auto-approve or auto-deploy the model.


In [23]:
# Prerequisite guard for Step 4.
missing = [name for name in ("step_train", "step_evaluate", "evaluation_report")
           if name not in globals()]
if missing:
    raise RuntimeError(
        "Step 4 prerequisites are missing: " + ", ".join(missing)
    )

from sagemaker.serve.model_builder import ModelBuilder
from sagemaker.mlops.workflow.model_step import ModelStep
from sagemaker.mlops.workflow import ConditionStep
from sagemaker.core.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.core.workflow.functions import JsonGet

inference_image = image_uris.retrieve(
    framework="sklearn",
    region=region,
    version="1.2-1",
    py_version="py3",
    instance_type="ml.m5.large",
    image_scope="inference",
)

inference_source = SourceCode(
    source_dir=str(SRC_DIR),
    entry_script="inference.py",
    requirements="requirements.txt",
)

model_builder = ModelBuilder(
    s3_model_data_url=step_train.properties.ModelArtifacts.S3ModelArtifacts,
    image_uri=inference_image,
    source_code=inference_source,
    sagemaker_session=pipeline_session,
    role_arn=role,
    content_type="application/json",
    accept_type="application/json",
)

def ensure_model_package_group(group_name, description):
    try:
        sm.describe_model_package_group(ModelPackageGroupName=group_name)
        print("Model Package Group exists:", group_name)
    except ClientError as exc:
        error_code = exc.response.get("Error", {}).get("Code")
        if error_code != "ValidationException":
            raise
        sm.create_model_package_group(
            ModelPackageGroupName=group_name,
            ModelPackageGroupDescription=description,
            Tags=[
                {"Key": "Course", "Value": COURSE},
                {"Key": "TeamId", "Value": TEAM_ID},
                {"Key": "StudentId", "Value": STUDENT_ID},
                {"Key": "ProjectName", "Value": PROJECT_NAME},
            ],
        )
        print("Created Model Package Group:", group_name)

ensure_model_package_group(
    MODEL_PACKAGE_GROUP,
    "Team02 diabetes-risk selected-champion model versions with human-controlled approval.",
)

registration_args = model_builder.register(
    content_types=["application/json"],
    response_types=["application/json"],
    inference_instances=["ml.m5.large"],
    model_package_group_name=MODEL_PACKAGE_GROUP,
    approval_status="PendingManualApproval",
    description=(
        "Team02 Explainable Diabetes Risk Screening academic prototype; "
        f"final {CHAMPION_MODEL_TYPE} champion, subject to independent evaluation and human approval."
    ),
    customer_metadata_properties={
        "team_id": TEAM_ID,
        "project_name": PROJECT_NAME,
        "lifecycle_stage": PIPELINE_RUN_MODE.lower(),
        "model_type": CHAMPION_MODEL_TYPE,
        "config_version": BEST_MODEL_CONFIG_VERSION,
        "preprocessing_config_version": PREPROCESSING_CONFIG_VERSION,
        "data_version": DATA_VERSION,
        "profile_sha256": EXPERIMENT_PROFILE_SHA256,
        "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
        "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
        "decision_threshold": str(DECISION_THRESHOLD),
        "source_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
        "source_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID or "n/a",
        "source_threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
        "source_final_run_id": CHAMPION_FINAL_RUN_ID,
        "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
        "model_handoff_run_id": BEST_MODEL_HANDOFF_MLFLOW_RUN_ID or "n/a",
        "human_oversight": "human_over_the_loop",
    },
)

step_register = ModelStep(
    name="team02-register-model",
    step_args=registration_args,
)

step_condition = None
if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    conditions = [
        ConditionGreaterThanOrEqualTo(
            left=JsonGet(
                step_name=step_evaluate.name,
                property_file=evaluation_report,
                json_path="test.roc_auc",
            ),
            right=p_gate_roc_auc,
        ),
    ]
    step_condition = ConditionStep(
        name="team02-quality-gate",
        conditions=conditions,
        if_steps=[step_register],
        else_steps=[],
    )
    print("Final ROC-AUC quality gate defined:", approved["quality_gate_threshold"])
else:
    # For integration/progress runs, register directly but still PendingManualApproval.
    step_register.add_depends_on([step_evaluate]) if hasattr(step_register, "add_depends_on") else None
    print("Non-final mode: direct PendingManualApproval registration configured.")

print("Initial registry approval status: PendingManualApproval")


[08/22/26 05:49:04] DEBUG    Auto-detecting optimal instance type for model...           ]8;id=11131613;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=11131614;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#341\341]8;;\

                    DEBUG    Using default CPU instance type: ml.m5.large                ]8;id=11131620;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py\model_builder_utils.py]8;;\:]8;id=11131621;file:///opt/conda/lib/python3.12/site-packages/sagemaker/serve/model_builder_utils.py#375\375]8;;\

Model Package Group exists: team02-diabetes-risk


                    WARNING  No models to repack                                                  ]8;id=11131628;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py\model_step.py]8;;\:]8;id=11131629;file:///opt/conda/lib/python3.12/site-packages/sagemaker/mlops/workflow/model_step.py#218\218]8;;\

Final ROC-AUC quality gate defined: 0.75
Initial registry approval status: PendingManualApproval


## 11. Compile, Create and Verify the SageMaker Pipeline

Expected final DAG:

`team02-preprocess-data → team02-train-model → team02-evaluate-model → team02-quality-gate → team02-register-model`

The Model Registry step is reached only when the pre-existing ROC-AUC regression gate passes.


In [24]:
required_pipeline_objects = [
    "step_process", "step_train", "step_evaluate",
    "p_input_data", "p_expected_profile_sha256",
    "p_expected_train_sha256", "p_expected_test_sha256",
]
if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    required_pipeline_objects += ["step_condition", "p_gate_roc_auc"]
else:
    required_pipeline_objects += ["step_register"]

missing = [name for name in required_pipeline_objects if name not in globals()]
if missing:
    raise RuntimeError(
        "Pipeline cannot be compiled because these objects are missing: "
        + ", ".join(missing)
        + ". Restart the kernel and run Notebook 03 from the top without skipping cells."
    )

from sagemaker.mlops.workflow.pipeline import Pipeline

pipeline_parameters = [
    p_input_data,
    p_expected_profile_sha256,
    p_expected_train_sha256,
    p_expected_test_sha256,
]
pipeline_steps = [
    step_process,
    step_train,
    step_evaluate,
]

if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    pipeline_parameters.append(p_gate_roc_auc)
    pipeline_steps.append(step_condition)
else:
    pipeline_steps.append(step_register)

pipeline = Pipeline(
    name=PIPELINE_NAME,
    parameters=pipeline_parameters,
    steps=pipeline_steps,
    sagemaker_session=pipeline_session,
)

pipeline_definition = json.loads(pipeline.definition())

def find_lowercase_key_value_pairs(obj, path="$"):
    findings = []
    if isinstance(obj, dict):
        if "key" in obj and "value" in obj:
            findings.append(path)
        for k, v in obj.items():
            findings.extend(find_lowercase_key_value_pairs(v, f"{path}.{k}"))
    elif isinstance(obj, list):
        for idx, item in enumerate(obj):
            findings.extend(find_lowercase_key_value_pairs(item, f"{path}[{idx}]"))
    return findings

bad_tag_paths = find_lowercase_key_value_pairs(pipeline_definition)
if bad_tag_paths:
    raise RuntimeError(
        "Malformed lowercase key/value members remain in the compiled pipeline at: "
        + ", ".join(bad_tag_paths[:10])
    )

step_names = [step["Name"] for step in pipeline_definition.get("Steps", [])]
parameter_names = [p["Name"] for p in pipeline_definition.get("Parameters", [])]

print("Compiled pipeline steps:")
for name in step_names:
    print(" -", name)

print("\nCompiled pipeline parameters:")
for name in parameter_names:
    print(" -", name)

expected_steps = {
    "team02-preprocess-data",
    "team02-train-model",
    "team02-evaluate-model",
}
if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
    expected_steps.add("team02-quality-gate")
else:
    expected_steps.add("team02-register-model")

missing_steps = expected_steps - set(step_names)
if missing_steps:
    raise RuntimeError(f"Pipeline definition is missing required steps: {sorted(missing_steps)}")

upsert_response = pipeline.upsert(
    role_arn=role,
    tags=[
        {"Key": "Course", "Value": COURSE},
        {"Key": "Semester", "Value": SEMESTER},
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "StudentId", "Value": STUDENT_ID},
        {"Key": "ProjectName", "Value": PROJECT_NAME},
        {"Key": "ModelType", "Value": CHAMPION_MODEL_TYPE},
        {"Key": "ConfigVersion", "Value": BEST_MODEL_CONFIG_VERSION},
    ],
)

print("\nPipeline upsert response:")
print(json.dumps(upsert_response, indent=2, default=str))

pipeline_desc = sm.describe_pipeline(PipelineName=PIPELINE_NAME)
print("\nAWS pipeline verified.")
print("Pipeline name :", pipeline_desc["PipelineName"])
print("Pipeline ARN  :", pipeline_desc["PipelineArn"])


                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=11131636;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131637;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

[08/22/26 05:49:05] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=11131643;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=11131644;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team02/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             02 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

[08/22/26 05:49:06] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=11131649;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=11131650;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team02/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             02 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=11131655;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131656;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=11131661;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131662;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

...

[08/22/26 05:49:13] DEBUG    Could not verify role access to default bucket: An error occurred ]8;id=11131728;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py\model_trainer.py]8;;\:]8;id=11131729;file:///opt/conda/lib/python3.12/site-packages/sagemaker/train/model_trainer.py#911\911]8;;\
                             (AccessDenied) when calling the SimulatePrincipalPolicy                               
                             operation: User:                                                                      
                             arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI                     
                             113-Team02/SageMaker is not authorized to perform:                                    
                             iam:SimulatePrincipalPolicy on resource:                                              
                             arn:aws:iam::044528205969:role/SageMakerExecutionRole-ITI113-Team                     
                             02 because no identity-based policy allows the                                        
                             iam:SimulatePrincipalPolicy action                                                    

                    WARNING  Popping out 'TrainingJobName' from the pipeline definition by default ]8;id=11131734;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131735;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             since it will be overridden at pipeline execution time. Please                        
                             utilize the PipelineDefinitionConfig to persist this field in the                     
                             pipeline definition if desired.                                                       

                    WARNING  Popping out 'ProcessingJobName' from the pipeline definition by       ]8;id=11131740;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131741;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   

                    WARNING  Popping out 'ModelPackageName' from the pipeline definition by        ]8;id=11131746;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py\utilities.py]8;;\:]8;id=11131747;file:///opt/conda/lib/python3.12/site-packages/sagemaker/core/workflow/utilities.py#480\480]8;;\
                             default since it will be overridden at pipeline execution time.                       
                             Please utilize the PipelineDefinitionConfig to persist this field in                  
                             the pipeline definition if desired.                                                   


Pipeline upsert response:
{
  "PipelineArn": "arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk",
  "PipelineVersionId": 17,
  "ResponseMetadata": {
    "RequestId": "5f90b55b-a7f4-4fe6-9e11-6e779002d175",
    "HTTPStatusCode": 200,
    "HTTPHeaders": {
      "x-amzn-requestid": "5f90b55b-a7f4-4fe6-9e11-6e779002d175",
      "strict-transport-security": "max-age=47304000; includeSubDomains",
      "x-frame-options": "DENY",
      "content-security-policy": "frame-ancestors 'none'",
      "cache-control": "no-cache, no-store, must-revalidate",
      "x-content-type-options": "nosniff",
      "content-type": "application/x-amz-json-1.1",
      "content-length": "116",
      "date": "Sat, 22 Aug 2026 05:49:14 GMT"
    },
    "RetryAttempts": 0
  }
}

AWS pipeline verified.
Pipeline name : team02-diabetes-risk
Pipeline ARN  : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk


## 12. Execute and Monitor a Controlled Pipeline Run

Pipeline creation and pipeline execution remain separate. Run the final pipeline only after reviewing the frozen model/data handoff above.


### 12.1 Start a Controlled Manual Execution

Set `RUN_PIPELINE_NOW=True` only when ready to incur SageMaker Processing/Training/Evaluation cost and create a new final-champion registry candidate.


In [25]:
execution = None

if RUN_PIPELINE_NOW:
    execution_parameters = {
        "InputDataUrl": RAW_DATA_URI,
        "ExpectedProfileSHA256": EXPERIMENT_PROFILE_SHA256,
        "ExpectedTrainSplitSHA256": EXPERIMENT_TRAIN_SPLIT_SHA256,
        "ExpectedTestSplitSHA256": EXPERIMENT_TEST_SPLIT_SHA256,
    }

    if PIPELINE_RUN_MODE == "FINAL_CHAMPION":
        execution_parameters["MinTestRocAuc"] = float(
            approved["quality_gate_threshold"]
        )

    print("Starting pipeline with:")
    print(json.dumps(execution_parameters, indent=2))

    execution = pipeline.start(parameters=execution_parameters)

    print("Pipeline execution ARN:", execution.arn)
else:
    print("Pipeline execution is disabled.")
    print("Set RUN_PIPELINE_NOW=True only after reviewing the final-champion handoff.")


Starting pipeline with:
{
  "InputDataUrl": "s3://nyp-26s1-iti113/iti113/team02/data/diabetes/cleaned/diabetes_binary_cleaned_dataset.csv",
  "ExpectedProfileSHA256": "6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5",
  "ExpectedTrainSplitSHA256": "93531a7b366471a63d29fd970abad6c5d04a2f933f9ffb2aa38dce1c7b286d65",
  "ExpectedTestSplitSHA256": "12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1",
  "MinTestRocAuc": 0.75
}
Pipeline execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/6r4j7hisk3rw


### 12.2 Monitor Detailed Pipeline Progress and Extract CloudWatch Failure Logs

Monitor the pipeline at both **pipeline level** and **individual-step level** instead of repeatedly displaying only `Pipeline status: Executing`.

The monitor shows:

- Processing, Training, Evaluation and Registration step status.
- The underlying SageMaker Processing or Training job name when available.
- Training `SecondaryStatus` such as `Starting`, `Downloading`, `Training` and `Uploading`.
- The latest SageMaker status message for the active training stage.
- Failure reason immediately when a step fails.
- Progress output only when the pipeline or underlying job stage changes, reducing repeated `Executing` messages.

If a run fails, the existing diagnostic functions still retrieve the detailed failed step and CloudWatch training logs.

**Previously resolved XGBoost compatibility issue:** if CloudWatch reports `ImportError: cannot import name 'Float32Dtype' from 'pandas'` inside `XGBClassifier.fit()`, the training/evaluation/inference paths avoid that pandas adapter by using ordered NumPy arrays for XGBoost.


In [26]:

def print_training_cloudwatch_tail(training_job_name, max_events=250):
    """Print the latest CloudWatch log events for one SageMaker training job."""
    logs = boto_session.client("logs", region_name=region)
    log_group = "/aws/sagemaker/TrainingJobs"

    streams = logs.describe_log_streams(
        logGroupName=log_group,
        logStreamNamePrefix=f"{training_job_name}/",
        orderBy="LogStreamName",
        descending=False,
        limit=50,
    ).get("logStreams", [])

    if not streams:
        print(
            f"No CloudWatch log streams found yet for {training_job_name!r}. "
            f"Check {log_group} in CloudWatch or verify logs permissions."
        )
        return []

    all_events = []

    for stream in streams:
        stream_name = stream["logStreamName"]
        print("\nCloudWatch stream:", stream_name)

        response = logs.get_log_events(
            logGroupName=log_group,
            logStreamName=stream_name,
            startFromHead=True,
            limit=10000,
        )
        events = response.get("events", [])
        all_events.extend(events)

    all_events.sort(key=lambda x: x.get("timestamp", 0))

    tail = all_events[-max_events:]

    print("\n" + "=" * 90)
    print(f"CLOUDWATCH TRAINING LOG TAIL — last {len(tail)} events")
    print("=" * 90)

    for event in tail:
        print(event.get("message", "").rstrip())

    print("=" * 90)

    return tail



def diagnose_pipeline_failure(execution):
    """Print the real SageMaker Pipeline failure reason and failed job details."""
    if execution is None:
        return None

    execution_arn = execution.arn

    print("\n" + "=" * 90)
    print("PIPELINE FAILURE DIAGNOSTICS")
    print("=" * 90)
    print("Execution ARN:", execution_arn)

    # 1) Overall pipeline failure reason
    exec_desc = sm.describe_pipeline_execution(
        PipelineExecutionArn=execution_arn
    )

    print("Overall status :", exec_desc.get("PipelineExecutionStatus"))
    print(
        "Failure reason :",
        exec_desc.get("FailureReason")
        or "(No overall FailureReason returned by SageMaker)"
    )

    # 2) Step-by-step status and failure reason
    response = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn,
        SortOrder="Ascending",
        MaxResults=100,
    )

    steps = response.get("PipelineExecutionSteps", [])

    print("\nPipeline steps:")
    for step in steps:
        print(
            f" - {step.get('StepName')}: "
            f"{step.get('StepStatus')}"
        )
        if step.get("FailureReason"):
            print("     Step failure:", step["FailureReason"])

    failed_steps = [
        step for step in steps
        if step.get("StepStatus") == "Failed"
    ]

    if not failed_steps:
        print("\nNo individual step is marked Failed.")
        return {
            "execution": exec_desc,
            "steps": steps,
            "failed_steps": [],
        }

    # 3) Drill into the actual SageMaker job behind each failed step
    for step in failed_steps:
        print("\n" + "-" * 90)
        print("FAILED STEP:", step.get("StepName"))
        print("Step reason:", step.get("FailureReason", "(not provided)"))

        metadata = step.get("Metadata", {}) or {}
        print("Metadata keys:", list(metadata.keys()))

        processing_meta = metadata.get("ProcessingJob")
        if processing_meta and processing_meta.get("Arn"):
            processing_arn = processing_meta["Arn"]
            processing_name = processing_arn.rsplit("/", 1)[-1]

            print("Processing job:", processing_name)

            job = sm.describe_processing_job(
                ProcessingJobName=processing_name
            )

            print("Processing status :", job.get("ProcessingJobStatus"))
            print("Failure reason    :", job.get("FailureReason", "(not provided)"))
            print("Exit message      :", job.get("ExitMessage", "(not provided)"))

        training_meta = metadata.get("TrainingJob")
        if training_meta and training_meta.get("Arn"):
            training_arn = training_meta["Arn"]
            training_name = training_arn.rsplit("/", 1)[-1]

            print("Training job:", training_name)

            job = sm.describe_training_job(
                TrainingJobName=training_name
            )

            print("Training status   :", job.get("TrainingJobStatus"))
            print("Secondary status  :", job.get("SecondaryStatus"))
            print("Failure reason    :", job.get("FailureReason", "(not provided)"))

            transitions = job.get("SecondaryStatusTransitions", [])
            if transitions:
                print("Last status transitions:")
                for t in transitions[-5:]:
                    print(
                        "   -",
                        t.get("Status"),
                        ":",
                        t.get("StatusMessage", "")
                    )

            # Pull the actual container stdout/stderr from CloudWatch.
            # This is where Python tracebacks, pip failures, file-not-found errors,
            # version mismatches, and model-fit exceptions are visible.
            try:
                print_training_cloudwatch_tail(training_name, max_events=250)
            except ClientError as exc:
                print(
                    "Could not read CloudWatch training logs:",
                    exc.response.get("Error", {}).get("Message", str(exc)),
                )
                print(
                    "If this is AccessDenied, the notebook role needs CloudWatch Logs "
                    "read permissions such as logs:DescribeLogStreams and logs:GetLogEvents."
                )

        # Registration/model steps do not always expose a separate job to describe.
        if metadata.get("RegisterModel"):
            print("RegisterModel metadata:", metadata["RegisterModel"])

        if metadata.get("Model"):
            print("Model metadata:", metadata["Model"])

    print("=" * 90)

    return {
        "execution": exec_desc,
        "steps": steps,
        "failed_steps": failed_steps,
    }



def get_latest_pipeline_execution(pipeline_name):
    """Return the latest SageMaker pipeline execution summary, or None."""
    response = sm.list_pipeline_executions(
        PipelineName=pipeline_name,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=1,
    )
    items = response.get("PipelineExecutionSummaries", [])
    return items[0] if items else None


def diagnose_latest_pipeline_execution(pipeline_name):
    """Diagnose the latest execution even after a kernel restart."""
    latest = get_latest_pipeline_execution(pipeline_name)
    if latest is None:
        print(f"No executions found for pipeline: {pipeline_name}")
        return None

    execution_arn = latest["PipelineExecutionArn"]
    print("Latest execution ARN   :", execution_arn)
    print("Latest execution status:", latest.get("PipelineExecutionStatus"))
    print(
        "Summary failure reason :",
        latest.get("PipelineExecutionFailureReason", "(not provided)")
    )

    # Minimal object compatible with diagnose_pipeline_failure()
    class _ExecutionRef:
        def __init__(self, arn):
            self.arn = arn

    return diagnose_pipeline_failure(_ExecutionRef(execution_arn))


def _status_icon(status):
    """Compact visual indicator for SageMaker pipeline/job status."""
    return {
        "Succeeded": "✅",
        "Completed": "✅",
        "InService": "✅",
        "Executing": "🔄",
        "Starting": "🔄",
        "InProgress": "🔄",
        "Downloading": "⬇️",
        "Training": "🧠",
        "Uploading": "⬆️",
        "Stopping": "⛔",
        "Stopped": "⛔",
        "Failed": "❌",
    }.get(status, "⏳")


def _training_job_detail(step):
    """Return live TrainingJob status information for a pipeline training step."""
    metadata = step.get("Metadata", {}) or {}
    training_meta = metadata.get("TrainingJob", {}) or {}
    arn = training_meta.get("Arn")

    if not arn:
        return None

    job_name = arn.rsplit("/", 1)[-1]

    try:
        job = sm.describe_training_job(
            TrainingJobName=job_name
        )
    except ClientError as exc:
        return {
            "job_name": job_name,
            "status": "UnableToDescribe",
            "secondary_status": None,
            "message": exc.response.get("Error", {}).get(
                "Message", str(exc)
            ),
        }

    transitions = job.get("SecondaryStatusTransitions", [])
    latest_message = None

    if transitions:
        latest_message = transitions[-1].get("StatusMessage")

    return {
        "job_name": job_name,
        "status": job.get("TrainingJobStatus"),
        "secondary_status": job.get("SecondaryStatus"),
        "message": latest_message,
    }


def _processing_job_detail(step):
    """Return live ProcessingJob status information for processing/evaluation steps."""
    metadata = step.get("Metadata", {}) or {}
    processing_meta = metadata.get("ProcessingJob", {}) or {}
    arn = processing_meta.get("Arn")

    if not arn:
        return None

    job_name = arn.rsplit("/", 1)[-1]

    try:
        job = sm.describe_processing_job(
            ProcessingJobName=job_name
        )
    except ClientError as exc:
        return {
            "job_name": job_name,
            "status": "UnableToDescribe",
            "message": exc.response.get("Error", {}).get(
                "Message", str(exc)
            ),
        }

    return {
        "job_name": job_name,
        "status": job.get("ProcessingJobStatus"),
        "message": (
            job.get("FailureReason")
            or job.get("ExitMessage")
        ),
    }


def _registration_detail(step):
    """Return registration/model metadata when SageMaker exposes it."""
    metadata = step.get("Metadata", {}) or {}

    for key in ("RegisterModel", "Model"):
        item = metadata.get(key)
        if isinstance(item, dict) and item.get("Arn"):
            return {
                "metadata_type": key,
                "arn": item["Arn"],
            }

    return None


def get_pipeline_progress_snapshot(execution_arn):
    """
    Query live pipeline and step status.

    Returns both a displayable snapshot and a compact signature so the monitor
    prints again only when something meaningful changes.
    """
    exec_desc = sm.describe_pipeline_execution(
        PipelineExecutionArn=execution_arn
    )
    pipeline_status = exec_desc["PipelineExecutionStatus"]

    response = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn,
        SortOrder="Ascending",
        MaxResults=100,
    )
    returned_steps = response.get("PipelineExecutionSteps", [])

    step_map = {
        step["StepName"]: step
        for step in returned_steps
    }

    expected_order = [
        "team02-preprocess-data",
        "team02-train-model",
        "team02-evaluate-model",
    ]

    if PIPELINE_RUN_MODE == "PROGRESS_CHECK":
        expected_order.append("team02-register-model")
    elif PIPELINE_RUN_MODE == "FINAL_CHAMPION":
        expected_order.append("team02-quality-gate")

    rows = []
    signature_parts = [pipeline_status]

    for step_name in expected_order:
        step = step_map.get(step_name)

        if step is None:
            row = {
                "name": step_name,
                "status": "NotStarted",
                "job_type": None,
                "job_name": None,
                "job_status": None,
                "substatus": None,
                "message": None,
                "registration_arn": None,
            }
            rows.append(row)
            signature_parts.extend([step_name, "NotStarted"])
            continue

        step_status = step.get("StepStatus", "Unknown")
        row = {
            "name": step_name,
            "status": step_status,
            "job_type": None,
            "job_name": None,
            "job_status": None,
            "substatus": None,
            "message": step.get("FailureReason"),
            "registration_arn": None,
        }

        if step_name == "team02-train-model":
            detail = _training_job_detail(step)
            if detail:
                row.update({
                    "job_type": "Training",
                    "job_name": detail["job_name"],
                    "job_status": detail["status"],
                    "substatus": detail["secondary_status"],
                    "message": (
                        row["message"]
                        or detail["message"]
                    ),
                })

        elif step_name in {
            "team02-preprocess-data",
            "team02-evaluate-model",
        }:
            detail = _processing_job_detail(step)
            if detail:
                row.update({
                    "job_type": "Processing",
                    "job_name": detail["job_name"],
                    "job_status": detail["status"],
                    "message": (
                        row["message"]
                        or detail["message"]
                    ),
                })

        elif step_name == "team02-register-model":
            detail = _registration_detail(step)
            if detail:
                row.update({
                    "job_type": detail["metadata_type"],
                    "registration_arn": detail["arn"],
                })

        rows.append(row)

        signature_parts.extend([
            step_name,
            step_status,
            row.get("job_status"),
            row.get("substatus"),
            row.get("message"),
            row.get("registration_arn"),
        ])

    return {
        "pipeline_status": pipeline_status,
        "failure_reason": exec_desc.get("FailureReason"),
        "rows": rows,
        "signature": tuple(signature_parts),
    }


def print_pipeline_progress(snapshot):
    """Print one concise, presentation-friendly pipeline progress update."""
    from datetime import datetime

    now = datetime.now().strftime("%H:%M:%S")
    pipeline_status = snapshot["pipeline_status"]

    print("\n" + "=" * 96)
    print(
        f"[{now}] PIPELINE STATUS: "
        f"{_status_icon(pipeline_status)} {pipeline_status}"
    )
    print("=" * 96)

    for row in snapshot["rows"]:
        print(
            f"{_status_icon(row['status'])} "
            f"{row['name']:<30} : {row['status']}"
        )

        if row["job_name"]:
            print(
                f"   {row['job_type']} job"
                f"{' ' * max(1, 20 - len(row['job_type']))}: "
                f"{row['job_name']}"
            )

        if row["job_status"]:
            print(
                f"   Job status                  : "
                f"{_status_icon(row['job_status'])} {row['job_status']}"
            )

        if row["substatus"]:
            print(
                f"   Current stage               : "
                f"{_status_icon(row['substatus'])} {row['substatus']}"
            )

        if row["message"]:
            print(
                f"   Detail                      : "
                f"{row['message']}"
            )

        if row["registration_arn"]:
            print(
                f"   Registered model ARN        : "
                f"{row['registration_arn']}"
            )

    if snapshot.get("failure_reason"):
        print(
            "Pipeline failure reason       :",
            snapshot["failure_reason"],
        )

    print("=" * 96)


def wait_for_pipeline(execution, poll_seconds=30):
    """
    Wait for the SageMaker Pipeline and display meaningful stage changes.

    Output is printed only when the pipeline status, step status, training
    secondary status, status message or registration metadata changes.
    """
    if execution is None:
        return None

    terminal_statuses = {
        "Succeeded",
        "Failed",
        "Stopped",
    }

    previous_signature = None

    print(
        f"Monitoring pipeline every {poll_seconds} seconds. "
        "A new status block is printed only when something changes."
    )

    while True:
        snapshot = get_pipeline_progress_snapshot(
            execution.arn
        )

        if snapshot["signature"] != previous_signature:
            print_pipeline_progress(snapshot)
            previous_signature = snapshot["signature"]

        status = snapshot["pipeline_status"]

        if status in terminal_statuses:
            return status

        time.sleep(poll_seconds)


pipeline_status = wait_for_pipeline(execution)

if execution is None:
    print("No manual execution was requested.")

elif pipeline_status == "Succeeded":
    print("Pipeline execution succeeded.")

else:
    failure_details = diagnose_pipeline_failure(execution)

    failed_names = [
        step.get("StepName", "UnknownStep")
        for step in failure_details.get("failed_steps", [])
    ]

    overall_reason = (
        failure_details.get("execution", {}).get("FailureReason")
        or "See detailed diagnostics printed above."
    )

    raise RuntimeError(
        f"Pipeline ended with status={pipeline_status}. "
        f"Failed step(s)={failed_names}. "
        f"Reason={overall_reason}"
    )


Monitoring pipeline every 30 seconds. A new status block is printed only when something changes.

[05:49:15] PIPELINE STATUS: 🔄 Executing
⏳ team02-preprocess-data         : NotStarted
⏳ team02-train-model             : NotStarted
⏳ team02-evaluate-model          : NotStarted
⏳ team02-quality-gate            : NotStarted

[05:49:45] PIPELINE STATUS: 🔄 Executing
🔄 team02-preprocess-data         : Executing
   Processing job          : pipelines-6r4j7hisk3rw-team02-preprocess-da-xSuR2eLKo1
   Job status                  : 🔄 InProgress
⏳ team02-train-model             : NotStarted
⏳ team02-evaluate-model          : NotStarted
⏳ team02-quality-gate            : NotStarted

[05:52:16] PIPELINE STATUS: 🔄 Executing
✅ team02-preprocess-data         : Succeeded
   Processing job          : pipelines-6r4j7hisk3rw-team02-preprocess-da-xSuR2eLKo1
   Job status                  : ✅ Completed
🔄 team02-train-model             : Executing
   Training job            : pipelines-6r4j7hisk3rw-team02-train

### 12.3 Generate Script-based Pipeline and CloudWatch Evidence

Produce an assessment-ready pipeline summary directly from SageMaker APIs and retrieve bounded CloudWatch log tails for the Processing and Training jobs associated with the execution.

This works without the SageMaker visual Pipeline view and is suitable for notebook evidence, troubleshooting and later reuse by an MLOps Admin application.


In [27]:
if execution is None:
    print("No pipeline execution object is available in this session.")
else:
    pipeline_summary = get_pipeline_execution_summary(
        sm,
        execution.arn,
    )

    print("PIPELINE EXECUTION SUMMARY")
    print("=" * 100)
    print("Execution ARN :", pipeline_summary["execution_arn"])
    print("Overall status:", pipeline_summary["pipeline_status"])
    print("Failure reason:", pipeline_summary.get("failure_reason") or "(none)")
    print("=" * 100)

    step_summary_df = pd.DataFrame(pipeline_summary["steps"])
    if not step_summary_df.empty:
        display(
            step_summary_df[
                [
                    "step_name",
                    "step_status",
                    "failure_reason",
                    "start_time",
                    "end_time",
                ]
            ]
        )

    print("\nRetrieving bounded CloudWatch log tails by API...")
    cloudwatch_report = get_execution_cloudwatch_report(
        sm,
        execution.arn,
        region_name=region,
        max_events_per_job=40,
    )

    if not cloudwatch_report:
        print("No Processing/Training CloudWatch streams were found yet.")
    else:
        for item in cloudwatch_report:
            print("\n" + "-" * 100)
            print("Step      :", item["step_name"])
            print("Job type  :", item["job_type"])
            print("Job name  :", item["job_name"])
            print("Log group :", item["log_group"])
            print("-" * 100)
            if not item["events"]:
                print("(No log events returned.)")
            else:
                for event in item["events"]:
                    print(event["message"])


PIPELINE EXECUTION SUMMARY
Execution ARN : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/6r4j7hisk3rw
Overall status: Succeeded
Failure reason: (none)


,step_name,step_status,failure_reason,start_time,end_time
0,team02-preprocess-data,Succeeded,None,2026-08-22 05:49:15.466000+00:00,2026-08-22 05:51:50.240000+00:00
1,team02-train-model,Succeeded,None,2026-08-22 05:51:51.052000+00:00,2026-08-22 05:54:45.329000+00:00
2,team02-evaluate-model,Succeeded,None,2026-08-22 05:54:46.113000+00:00,2026-08-22 05:59:49.435000+00:00
3,team02-quality-gate,Succeeded,None,2026-08-22 05:59:50.034000+00:00,2026-08-22 05:59:50.296000+00:00
4,team02-register-model,Succeeded,None,2026-08-22 05:59:51.033000+00:00,2026-08-22 05:59:52.834000+00:00



Retrieving bounded CloudWatch log tails by API...

----------------------------------------------------------------------------------------------------
Step      : team02-preprocess-data
Job type  : ProcessingJob
Job name  : pipelines-6r4j7hisk3rw-team02-preprocess-da-xSuR2eLKo1
Log group : /aws/sagemaker/ProcessingJobs
----------------------------------------------------------------------------------------------------
Experiment-aligned preprocessing complete.
{
  "source_file": "diabetes_binary_cleaned_dataset.csv",
  "rows": 253680,
  "features": [
    "HighBP",
    "HighChol",
    "CholCheck",
    "BMI",
    "Smoker",
    "Stroke",
    "HeartDiseaseorAttack",
    "PhysActivity",
    "Fruits",
    "Veggies",
    "HvyAlcoholConsump",
    "AnyHealthcare",
    "NoDocbcCost",
    "GenHlth",
    "MentHlth",
    "PhysHlth",
    "DiffWalk",
    "Sex",
    "Age",
    "Education",
    "Income"
  ],
  "source_target": "Diabetes_binary",
  "pipeline_target": "target",
  "split_strategy": "str

### 12.4 Diagnose the Latest Failed Execution Without Retraining

Inspect the most recent pipeline execution and identify any failed Processing or Training step. For a failed training job, retrieve the container stdout/stderr from CloudWatch Logs under `/aws/sagemaker/TrainingJobs`.

This diagnostic only reads existing execution evidence and does **not** start another pipeline run.

In [28]:
# Diagnose the most recent pipeline execution without launching a new run.
latest_failure_details = diagnose_latest_pipeline_execution(PIPELINE_NAME)


Latest execution ARN   : arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/6r4j7hisk3rw
Latest execution status: Succeeded
Summary failure reason : (not provided)

PIPELINE FAILURE DIAGNOSTICS
Execution ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/6r4j7hisk3rw
Overall status : Succeeded
Failure reason : (No overall FailureReason returned by SageMaker)

Pipeline steps:
 - team02-preprocess-data: Succeeded
 - team02-train-model: Succeeded
 - team02-evaluate-model: Succeeded
 - team02-quality-gate: Succeeded
 - team02-register-model: Succeeded

No individual step is marked Failed.


## 13. Record Completed Pipeline Evidence in MLflow

After a successful execution, recover `evaluation.json` and log the full operational evidence:
- pipeline parameters and definition;
- data/preprocessing version and split signatures;
- source modelling run IDs;
- XGBoost runtime version;
- independent test metrics;
- fairness artifact;
- registration/approval context;
- pipeline execution ARN.

This produces a second MLflow run that links the development experiments to the deployed operational lifecycle.


In [29]:
PIPELINE_MLFLOW_RUN_ID = None
evaluation_json = None
PIPELINE_EVIDENCE_EXECUTION_ARN = None

if execution is not None:
    PIPELINE_EVIDENCE_EXECUTION_ARN = execution.arn
    print("Using pipeline execution from current kernel:", PIPELINE_EVIDENCE_EXECUTION_ARN)
else:
    recent = sm.list_pipeline_executions(
        PipelineName=PIPELINE_NAME,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=20,
    ).get("PipelineExecutionSummaries", [])

    succeeded = [
        item for item in recent
        if item.get("PipelineExecutionStatus") == "Succeeded"
    ]
    if succeeded:
        PIPELINE_EVIDENCE_EXECUTION_ARN = succeeded[0]["PipelineExecutionArn"]
        print(
            "Recovered latest successful pipeline execution:",
            PIPELINE_EVIDENCE_EXECUTION_ARN,
        )
    else:
        print("No successful pipeline execution is available for Section 13 logging.")

if PIPELINE_EVIDENCE_EXECUTION_ARN:
    steps = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=PIPELINE_EVIDENCE_EXECUTION_ARN
    ).get("PipelineExecutionSteps", [])

    for step in steps:
        print(step["StepName"], "->", step["StepStatus"])

    evaluation_step_info = next(
        step for step in steps
        if step["StepName"] == "team02-evaluate-model"
    )

    processing_job_arn = evaluation_step_info["Metadata"]["ProcessingJob"]["Arn"]
    processing_job_name = processing_job_arn.rsplit("/", 1)[-1]
    evaluation_job = sm.describe_processing_job(
        ProcessingJobName=processing_job_name
    )

    evaluation_s3_uri = None
    for output in evaluation_job["ProcessingOutputConfig"]["Outputs"]:
        if output["OutputName"] == "evaluation":
            evaluation_s3_uri = output["S3Output"]["S3Uri"]
            break

    if not evaluation_s3_uri:
        raise RuntimeError("Evaluation S3 output was not found.")

    eval_bucket, eval_prefix = parse_s3_uri(evaluation_s3_uri)
    eval_key = eval_prefix.rstrip("/") + "/evaluation.json"

    evaluation_json = json.loads(
        s3.get_object(Bucket=eval_bucket, Key=eval_key)["Body"]
        .read()
        .decode("utf-8")
    )

    if evaluation_json.get("model_type") != CHAMPION_MODEL_TYPE:
        raise RuntimeError(
            "Pipeline evaluation model type does not match the selected champion."
        )
    if evaluation_json.get("data_version") != DATA_VERSION:
        raise RuntimeError("Pipeline evaluation data_version mismatch.")
    if evaluation_json.get("dataset_sha256") != EXPERIMENT_PROFILE_SHA256:
        raise RuntimeError("Pipeline evaluation profile signature mismatch.")
    if evaluation_json.get("config_version") != BEST_MODEL_CONFIG_VERSION:
        raise RuntimeError("Pipeline evaluation config version mismatch.")

    print(json.dumps(evaluation_json, indent=2))

    mlflow.set_tracking_uri(MLFLOW_APP_ARN)
    mlflow.set_experiment(MLFLOW_EXPERIMENT)

    run_name = (
        f"{TEAM_ID}_{STUDENT_ID}_sagemaker_final_champion_pipeline_"
        f"{int(time.time())}"
    )

    with mlflow.start_run(run_name=run_name) as run:
        mlflow.set_tags({
            "course": COURSE,
            "semester": SEMESTER,
            "team_id": TEAM_ID,
            "student_id": STUDENT_ID,
            "project_name": PROJECT_NAME,
            "stage": "final_pipeline_evaluation_and_registration",
            "pipeline_run_mode": PIPELINE_RUN_MODE,
            "selected_model_type": CHAMPION_MODEL_TYPE,
            "source_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
            "source_final_run_id": CHAMPION_FINAL_RUN_ID,
            "comparison_lr_candidate_run_id": LR_CANDIDATE_RUN_ID,
            "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
            "best_model_handoff_mlflow_run_id": BEST_MODEL_HANDOFF_MLFLOW_RUN_ID or "n/a",
            "best_model_config_version": BEST_MODEL_CONFIG_VERSION,
            "preprocessing_config_version": PREPROCESSING_CONFIG_VERSION,
            "model_version_identity": CHAMPION_FINAL_RUN_ID,
            "xgb_evidence_export_sha256": SOURCE_EVIDENCE_EXPORTS["XGBoost"]["sha256"],
            "lr_evidence_export_sha256": SOURCE_EVIDENCE_EXPORTS["LogisticRegression"]["sha256"],
            "data_version": DATA_VERSION,
            "profile_sha256": EXPERIMENT_PROFILE_SHA256,
            "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
            "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
            "pipeline_execution_arn": PIPELINE_EVIDENCE_EXECUTION_ARN,
            "human_oversight": "human_over_the_loop",
            "pipeline_xgboost_version": PIPELINE_XGBOOST_VERSION,
        })

        mlflow.log_params({
            "model_type": CHAMPION_MODEL_TYPE,
            "decision_threshold": DECISION_THRESHOLD,
            "best_model_config_version": BEST_MODEL_CONFIG_VERSION,
            "source_candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
            "source_refinement_run_id": CHAMPION_REFINEMENT_RUN_ID or "n/a",
            "source_threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
            "source_final_run_id": CHAMPION_FINAL_RUN_ID,
            "comparison_lr_candidate_run_id": LR_CANDIDATE_RUN_ID,
            "comparison_lr_threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
            "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
            "comparison_lr_decision_threshold": LR_THRESHOLD,
            "data_source_s3_uri": RAW_DATA_URI,
            "data_version": DATA_VERSION,
            "profile_sha256": EXPERIMENT_PROFILE_SHA256,
            "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
            "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
            "preprocessing_config_version": PREPROCESSING_CONFIG_VERSION,
            "preprocessing_split_strategy": "stratified_80_20_random_state_42",
            "preprocessing_cv_provenance": f"Stratified_{EXPERIMENT_CV_FOLDS}_Fold_CV_training_only",
            "quality_gate_metric": approved["quality_gate_metric"],
            "quality_gate_threshold": approved["quality_gate_threshold"],
            **{f"xgb_{key}": value for key, value in MODEL_PARAMS.items()},
            **{f"lr_{key}": value for key, value in LR_MODEL_PARAMS.items()},
        })

        mlflow.log_metrics({
            f"test_{key}": float(value)
            for key, value in evaluation_json["test"].items()
            if isinstance(value, (int, float))
        })

        Path("pipeline_evaluation.json").write_text(
            json.dumps(evaluation_json, indent=2),
            encoding="utf-8",
        )
        Path("pipeline_approval_record.json").write_text(
            json.dumps(approved, indent=2),
            encoding="utf-8",
        )
        Path("pipeline_definition.json").write_text(
            json.dumps(pipeline_definition, indent=2),
            encoding="utf-8",
        )

        completed_pipeline_lineage = {
            "team_id": TEAM_ID,
            "model_type": CHAMPION_MODEL_TYPE,
            "config_version": BEST_MODEL_CONFIG_VERSION,
            "candidate_run_id": CHAMPION_CANDIDATE_RUN_ID,
            "refinement_run_id": CHAMPION_REFINEMENT_RUN_ID,
            "threshold_tuning_run_id": CHAMPION_THRESHOLD_TUNING_RUN_ID,
            "final_model_run_id": CHAMPION_FINAL_RUN_ID,
            "comparison_lr_candidate_run_id": LR_CANDIDATE_RUN_ID,
            "comparison_lr_threshold_tuning_run_id": LR_THRESHOLD_TUNING_RUN_ID,
            "comparison_lr_final_run_id": LR_FINAL_COMPARISON_RUN_ID,
            "comparison_lr_decision_threshold": LR_THRESHOLD,
            "comparison_lr_model_params": LR_MODEL_PARAMS,
            "decision_threshold": DECISION_THRESHOLD,
            "model_params": MODEL_PARAMS,
            "preprocessing": PREPROCESSING_CONFIG,
            "source_evidence_exports": SOURCE_EVIDENCE_EXPORTS,
            "data_version": DATA_VERSION,
            "profile_sha256": EXPERIMENT_PROFILE_SHA256,
            "train_split_sha256": EXPERIMENT_TRAIN_SPLIT_SHA256,
            "test_split_sha256": EXPERIMENT_TEST_SPLIT_SHA256,
            "pipeline_execution_arn": PIPELINE_EVIDENCE_EXECUTION_ARN,
            "evaluation": evaluation_json,
        }
        Path("completed_champion_pipeline_lineage.json").write_text(
            json.dumps(completed_pipeline_lineage, indent=2, default=str),
            encoding="utf-8",
        )

        for artifact_name, artifact_path in [
            ("pipeline_evaluation.json", "pipeline"),
            ("pipeline_definition.json", "pipeline"),
            ("pipeline_approval_record.json", "governance"),
            ("completed_champion_pipeline_lineage.json", "lineage"),
            ("team02_reproducibility_manifest.json", "lineage"),
            ("team02_champion_model_config.json", "final_champion_handoff"),
            ("best_model_experiment_comparison.csv", "final_champion_handoff"),
            ("model_hyperparameter_handoff.csv", "final_champion_handoff"),
            ("source_experiment_evidence_manifest.json", "lineage"),
            ("source_experiment_evidence_verification.json", "lineage"),
        ]:
            if Path(artifact_name).exists():
                mlflow.log_artifact(artifact_name, artifact_path=artifact_path)

        PIPELINE_MLFLOW_RUN_ID = run.info.run_id

    print("Pipeline evidence logged to MLflow run:", PIPELINE_MLFLOW_RUN_ID)


Using pipeline execution from current kernel: arn:aws:sagemaker:ap-southeast-1:044528205969:pipeline/team02-diabetes-risk/execution/6r4j7hisk3rw
team02-register-model -> Succeeded
team02-quality-gate -> Succeeded
team02-evaluate-model -> Succeeded
team02-train-model -> Succeeded
team02-preprocess-data -> Succeeded
{
  "model_type": "XGBoost",
  "team_id": "team02",
  "config_version": "2026-08-22-cross-model-champion-v5",
  "model_params": {
    "n_estimators": 450,
    "max_depth": 4,
    "learning_rate": 0.03,
    "min_child_weight": 5.0,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 10.0,
    "gamma": 0.05,
    "scale_pos_weight": 3.0,
    "tree_method": "hist",
    "random_state": 42,
    "objective": "binary:logistic",
    "eval_metric": "aucpr",
    "n_jobs": -1
  },
  "decision_threshold": 0.5,
  "data_version": "brfss2015-diabetes-binary-6244bec277fe",
  "dataset_sha256": "6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd020010

### 13.1 Final Champion Selection Evidence

This cell recomputes the champion from the frozen final comparison table and confirms that the pipeline model exactly matches that result. This is an explicit assessment artifact showing that Notebook 03 operationalises the **best team model**, rather than a modeller-specific default.

In [30]:
BEST_MODEL_DECISION_DF = pd.DataFrame(
    [
        {"model_type": model_type, **values}
        for model_type, values in MODEL_SELECTION_EVIDENCE.items()
    ]
).sort_values(CHAMPION_PRIMARY_METRIC, ascending=False)

display(BEST_MODEL_DECISION_DF)

lr_metrics = MODEL_SELECTION_EVIDENCE["LogisticRegression"]
xgb_metrics = MODEL_SELECTION_EVIDENCE["XGBoost"]

comparison_metrics = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "pr_auc",
    "roc_auc",
    "balanced_accuracy",
]

print("\nLATEST FULL-DATA DELTAS: XGBoost minus Logistic Regression")
print("=" * 92)
for metric in comparison_metrics:
    delta = float(xgb_metrics[metric]) - float(lr_metrics[metric])
    print(f"{metric:<24}: {delta:+.6f}")
print("=" * 92)

recomputed_champion = max(
    MODEL_SELECTION_EVIDENCE,
    key=lambda model_name: float(
        MODEL_SELECTION_EVIDENCE[model_name][CHAMPION_PRIMARY_METRIC]
    ),
)

if recomputed_champion != BEST_MODEL_TYPE:
    raise RuntimeError(
        "Cross-model champion recomputation does not match BEST_MODEL_TYPE."
    )

if CHAMPION_MODEL_TYPE != recomputed_champion:
    raise RuntimeError(
        "Pipeline model does not match the champion selected from final evidence."
    )

winner_metric = float(MODEL_SELECTION_EVIDENCE[recomputed_champion][CHAMPION_PRIMARY_METRIC])
runner_metric = float(MODEL_SELECTION_EVIDENCE[RUNNER_UP_MODEL_TYPE][CHAMPION_PRIMARY_METRIC])
metric_delta = winner_metric - runner_metric

if metric_delta <= 0:
    raise RuntimeError("Champion selection evidence is internally inconsistent.")

print("Champion primary metric :", CHAMPION_PRIMARY_METRIC_LABEL)
print("Selected champion        :", recomputed_champion)
print("Champion PR-AUC          :", winner_metric)
print("Runner-up                :", RUNNER_UP_MODEL_TYPE)
print("Runner-up PR-AUC         :", runner_metric)
print("PR-AUC delta             :", metric_delta)
print("\nDocumented trade-offs:")
print(" - Recall: XGB", xgb_metrics["recall"], "vs LR", lr_metrics["recall"])
print(
    " - Balanced accuracy: XGB", xgb_metrics["balanced_accuracy"],
    "vs LR", lr_metrics["balanced_accuracy"]
)
print(" - Precision: XGB", xgb_metrics["precision"], "vs LR", lr_metrics["precision"])
print("\nFinal technical champion for MLOps implementation:", recomputed_champion)
print("Human approval remains required before deployment.")

,model_type,final_run_id,candidate_name,candidate_run_id,threshold_tuning_run_id,decision_threshold,accuracy,precision,recall,f1,pr_auc,roc_auc,balanced_accuracy,refinement_run_id,f2,brier
1,XGBoost,057bb58d81564c0db9a628b224eab8f9,xgb_candidate_29,02807641d2814f48a084c7c451414917,a1a47bcc23d844539d92f3a3ed8fec1e,0.50,0.798151,0.407815,0.621388,0.492442,0.464397,0.824820,0.726302,e89987fd9aec4a11a1108c8afaf6561b,0.562475,0.133926
0,LogisticRegression,82bc9a4720e04a50ae942128305eb90d,lr_candidate_12,74e8884c230349f3ad29021db15f54a2,4d0a1f65fd754905bb85a4459886cb10,0.59,0.777594,0.380738,0.656660,0.482005,0.432749,0.817565,0.728438,NaN,NaN,NaN



LATEST FULL-DATA DELTAS: XGBoost minus Logistic Regression
accuracy                : +0.020557
precision               : +0.027077
recall                  : -0.035272
f1                      : +0.010437
pr_auc                  : +0.031649
roc_auc                 : +0.007255
balanced_accuracy       : -0.002136
Champion primary metric : held-out PR-AUC
Selected champion        : XGBoost
Champion PR-AUC          : 0.464397478
Runner-up                : LogisticRegression
Runner-up PR-AUC         : 0.432748858
PR-AUC delta             : 0.03164861999999996

Documented trade-offs:
 - Recall: XGB 0.621388368 vs LR 0.656660413
 - Balanced accuracy: XGB 0.726302148 vs LR 0.728437831
 - Precision: XGB 0.407814809 vs LR 0.38073827

Final technical champion for MLOps implementation: XGBoost
Human approval remains required before deployment.


## 14. Review and Manually Approve the Registered Model by API

The technical pipeline can register a candidate, but it cannot approve itself.

Human-over-the-loop control requires a reviewer to inspect:
- final-champion model/run IDs and hyperparameters;
- held-out performance and fairness evidence;
- data/split signatures;
- model artifact deployment preflight;
- known recall/FNR trade-off;
- intended-use limitation.

Only then may the reviewer set `APPROVE_REGISTERED_MODEL=True`.


### 14.1 Review the Latest Registered Champion Model Package

Review the newest package in the selected Model Package Group. It should initially be `PendingManualApproval`.

For `FINAL_CHAMPION`, the group is:

`team02-diabetes-risk`


In [31]:
print("Current Model Package Group:", MODEL_PACKAGE_GROUP)

registry_rows = get_model_registry_summary(
    sm,
    MODEL_PACKAGE_GROUP,
    max_results=10,
)

LATEST_PACKAGE_ARN = None

if not registry_rows:
    print("No package is currently registered in:", MODEL_PACKAGE_GROUP)
else:
    display(pd.DataFrame(registry_rows))
    LATEST_PACKAGE_ARN = registry_rows[0]["model_package_arn"]

    print("\nMODEL REGISTRY SUMMARY")
    print("=" * 90)
    print("Package Group     :", MODEL_PACKAGE_GROUP)
    print("Latest Package ARN:", LATEST_PACKAGE_ARN)
    print("Latest Version    :", registry_rows[0]["model_package_version"])
    print("Approval Status   :", registry_rows[0]["approval_status"])
    print("Creation Time     :", registry_rows[0]["creation_time"])
    print("=" * 90)


Current Model Package Group: team02-diabetes-risk


,model_package_arn,model_package_version,approval_status,creation_time
0,arn:aws:sagemaker:ap-southeast-1:044528205969:...,3,PendingManualApproval,2026-08-22 05:59:52.769000+00:00
1,arn:aws:sagemaker:ap-southeast-1:044528205969:...,2,Approved,2026-08-22 05:38:20.921000+00:00
2,arn:aws:sagemaker:ap-southeast-1:044528205969:...,1,Approved,2026-08-21 08:54:15.185000+00:00



MODEL REGISTRY SUMMARY
Package Group     : team02-diabetes-risk
Latest Package ARN: arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team02-diabetes-risk/3
Latest Version    : 3
Approval Status   : PendingManualApproval
Creation Time     : 2026-08-22 05:59:52.769000+00:00


### 14.2 Apply Explicit Human-over-the-loop Approval

The XGBoost Model Registry candidate is intentionally created as `PendingManualApproval`.

Approval is a separate deliberate action controlled by:

```python
APPROVE_REGISTERED_MODEL = False
```

Keep it `False` while reviewing the evaluation, fairness, explainability and deployment-readiness evidence. Change it to `True` only when the registered XGBoost candidate is intentionally approved for test deployment.

The approval evidence records the Model Package ARN, approval status, AWS identity, timestamp, data lineage, decision threshold and model type.

This implements **human-over-the-loop** oversight: the pipeline can process, train, evaluate and register automatically, while deployment eligibility requires explicit human approval.


In [32]:
# ------------------------------------------------------------------
# HUMAN-OVER-THE-LOOP APPROVAL
# ------------------------------------------------------------------
# Default is deliberately False.
# Review the model/evaluation evidence first. Change to True only when
# you intentionally approve this registered candidate for deployment.
# ------------------------------------------------------------------

# APPROVE_REGISTERED_MODEL is controlled from the Human Control Panel.

LATEST_PACKAGE_STATUS = None
HUMAN_APPROVAL_EVIDENCE = None

if not LATEST_PACKAGE_ARN:
    print("No registered model package is available to review.")

else:
    package_before = sm.describe_model_package(
        ModelPackageName=LATEST_PACKAGE_ARN
    )

    status_before = package_before.get("ModelApprovalStatus")
    LATEST_PACKAGE_STATUS = status_before

    print("=" * 90)
    print("HUMAN-OVER-THE-LOOP MODEL REVIEW")
    print("=" * 90)
    print("Pipeline mode       :", PIPELINE_RUN_MODE)
    print("Model Package Group :", MODEL_PACKAGE_GROUP)
    print("Model Package ARN   :", LATEST_PACKAGE_ARN)
    print("Current status      :", status_before)
    print("Model type          :", CHAMPION_MODEL_TYPE)
    print("Decision threshold  :", DECISION_THRESHOLD)
    print("Data version        :", DATA_VERSION)
    print("Dataset SHA-256     :", approved["dataset_sha256"])

    if PIPELINE_RUN_MODE == "PROGRESS_CHECK":
        print("Lifecycle purpose   : registered test deployment candidate")
        print("Operational model   : XGBoost selected from best experiment results")
    elif PIPELINE_RUN_MODE == "FINAL_CHAMPION":
        print("Lifecycle purpose   : Final champion deployment candidate")
    else:
        print("Lifecycle purpose   : Integration only")

    print("=" * 90)

    if status_before == "Approved":
        print("✅ This model package is already Approved.")
        print("You may continue to Section 15.")

    elif status_before not in {"PendingManualApproval", "Rejected"}:
        print(
            "⚠️ Approval action not attempted because the current Model Registry "
            f"status is {status_before!r}."
        )

    elif not APPROVE_REGISTERED_MODEL:
        print()
        print("⏸ HUMAN APPROVAL REQUIRED")
        print("The model remains:", status_before)
        print()
        print("Before approving, review at minimum:")
        print("  1. Section 12 pipeline status = Succeeded")
        print("  2. Section 13 MLflow/pipeline evidence")
        print("  3. Section 14.1 registered model version and metadata")
        print("  4. Evaluation metrics and known limitations")
        print()
        print(
            "When you intentionally approve this candidate for deployment, "
            "change:"
        )
        print("    APPROVE_REGISTERED_MODEL = True")
        print("and rerun only this Section 14.2 cell.")

    else:
        if PIPELINE_RUN_MODE not in {"PROGRESS_CHECK", "FINAL_CHAMPION"}:
            raise RuntimeError(
                "Approval is blocked in INTEGRATION_TEST mode."
            )

        sts = boto3.client("sts")
        caller = sts.get_caller_identity()
        approver_arn = caller["Arn"]

        from datetime import datetime, timezone
        approval_time_utc = datetime.now(timezone.utc).isoformat()

        if PIPELINE_RUN_MODE == "PROGRESS_CHECK":
            approval_description = (
                "Human-over-the-loop approval for Team02 registered test "
                "deployment after review of the SageMaker pipeline and evaluation "
                "evidence. This approval authorizes only the selected registered "
                "best-model XGBoost test candidate after review of pipeline and governance evidence."
            )
        else:
            approval_description = (
                "Human-over-the-loop approval for the Team02 final champion after "
                "review of the frozen quality-gate and model evidence."
            )

        print()
        print("Applying explicit human approval...")
        print("Approver ARN :", approver_arn)
        print("Timestamp UTC:", approval_time_utc)

        try:
            approval_result = approve_model_package(
                sm,
                LATEST_PACKAGE_ARN,
                approval_description,
            )

        except ClientError as exc:
            error = exc.response.get("Error", {})
            error_code = error.get("Code", "Unknown")
            error_message = error.get("Message", str(exc))

            if error_code in {
                "AccessDeniedException",
                "AccessDenied",
                "UnauthorizedOperation",
            }:
                print()
                print("❌ IAM permission blocked the approval action.")
                print("Required action: sagemaker:UpdateModelPackage")
                print("AWS error:", error_message)
                print()
                print(
                    "The model remains PendingManualApproval. "
                    "Do not continue to deployment until an authorized reviewer "
                    "performs the approval."
                )
            else:
                raise

        else:
            package_after = sm.describe_model_package(
                ModelPackageName=LATEST_PACKAGE_ARN
            )
            status_after = approval_result["status_after"]
            LATEST_PACKAGE_STATUS = status_after

            HUMAN_APPROVAL_EVIDENCE = {
                "model_package_arn": LATEST_PACKAGE_ARN,
                "model_package_group": MODEL_PACKAGE_GROUP,
                "pipeline_run_mode": PIPELINE_RUN_MODE,
                "model_type": CHAMPION_MODEL_TYPE,
                "decision_threshold": float(DECISION_THRESHOLD),
                "data_version": DATA_VERSION,
                "dataset_sha256": approved["dataset_sha256"],
                "approval_status_before": status_before,
                "approval_status_after": status_after,
                "approved_by_arn": approver_arn,
                "approval_time_utc": approval_time_utc,
                "approval_description": approval_description,
                "human_oversight": "human_over_the_loop",
            }

            print()
            print("✅ Human approval action completed.")
            print("Previous status :", status_before)
            print("New status      :", status_after)
            print("Approved by     :", approver_arn)
            print("Approval time   :", approval_time_utc)

            if status_after == "Approved":
                print()
                print("✅ Deployment eligibility confirmed.")
                print("Continue to Section 15.")
            else:
                print()
                print(
                    "⚠️ Approval API call completed, but the package is not "
                    "currently showing Approved. Recheck the Model Registry status."
                )


HUMAN-OVER-THE-LOOP MODEL REVIEW
Pipeline mode       : FINAL_CHAMPION
Model Package Group : team02-diabetes-risk
Model Package ARN   : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team02-diabetes-risk/3
Current status      : PendingManualApproval
Model type          : XGBoost
Decision threshold  : 0.5
Data version        : brfss2015-diabetes-binary-6244bec277fe
Dataset SHA-256     : 6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5
Lifecycle purpose   : Final champion deployment candidate

Applying explicit human approval...
Approver ARN : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Timestamp UTC: 2026-08-22T06:00:24.908477+00:00

✅ Human approval action completed.
Previous status : PendingManualApproval
New status      : Approved
Approved by     : arn:aws:sts::044528205969:assumed-role/SageMakerExecutionRole-ITI113-Team02/SageMaker
Approval time   : 2026-08-22T06:00:24.908477+00:00

✅ Deployment eligibility confir

## 15. Deploy the Human-approved Selected Champion

After manual approval, deploy the latest approved final package to:

`team02-diabetes-risk`

The endpoint serves one model family only: XGBoost.


In [33]:
print("Deployment mode :", PIPELINE_RUN_MODE)
print("Package group   :", MODEL_PACKAGE_GROUP)
print("Endpoint target :", ENDPOINT_NAME)

# Re-check Model Registry approval independently of Section 14 variables.
# This prevents deployment if the kernel was restarted or the approval cell
# was not executed in the current session.
DEPLOYMENT_APPROVAL_CONFIRMED = False

if LATEST_PACKAGE_ARN:
    deploy_package_desc = sm.describe_model_package(
        ModelPackageName=LATEST_PACKAGE_ARN
    )
    deploy_package_status = deploy_package_desc.get("ModelApprovalStatus")
    DEPLOYMENT_APPROVAL_CONFIRMED = (
        deploy_package_status == "Approved"
    )
    print("Registry status  :", deploy_package_status)
else:
    deploy_package_status = None
    print("Registry status  : No model package selected")

# DEPLOY_ENDPOINT_AFTER_APPROVAL is controlled from the Human Control Panel.

if (
    DEPLOY_ENDPOINT_AFTER_APPROVAL
    and PIPELINE_RUN_MODE == "INTEGRATION_TEST"
):
    raise RuntimeError(
        "Endpoint deployment is blocked in INTEGRATION_TEST mode. "
        "Use PROGRESS_CHECK for the milestone test endpoint or "
        "FINAL_CHAMPION for the final endpoint."
    )

if (
    DEPLOY_ENDPOINT_AFTER_APPROVAL
    and not DEPLOYMENT_APPROVAL_CONFIRMED
):
    raise RuntimeError(
        "Deployment blocked: the selected Model Registry package is not Approved. "
        "Complete the explicit human-over-the-loop approval in Section 14.2 first."
    )

if not DEPLOY_ENDPOINT_AFTER_APPROVAL:
    print("Endpoint deployment is disabled.")
    print(
        "After human review and Model Registry approval, set "
        "DEPLOY_ENDPOINT_AFTER_APPROVAL=True and rerun this section."
    )
else:
    approved_packages = sm.list_model_packages(
        ModelPackageGroupName=MODEL_PACKAGE_GROUP,
        ModelApprovalStatus="Approved",
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=20,
    ).get("ModelPackageSummaryList", [])

    if not approved_packages:
        raise RuntimeError(
            "Deployment was requested but no Approved model package exists. "
            "Complete human review and approval first."
        )

    DEPLOY_PACKAGE_ARN = approved_packages[0]["ModelPackageArn"]

    # Single best-model routing check.
    if PIPELINE_RUN_MODE == "PROGRESS_CHECK":
        if MODEL_PACKAGE_GROUP != PROGRESS_MODEL_PACKAGE_GROUP:
            raise RuntimeError(
                "Unexpected Progress Check Model Package Group."
            )
        if ENDPOINT_NAME != PROGRESS_ENDPOINT_NAME:
            raise RuntimeError(
                "Unexpected Progress Check endpoint."
            )

    deployment_tags = [
        {"Key": "Course", "Value": COURSE},
        {"Key": "TeamId", "Value": TEAM_ID},
        {"Key": "ProjectName", "Value": PROJECT_NAME},
        {"Key": "LifecycleStage", "Value": PIPELINE_RUN_MODE},
        {"Key": "ModelType", "Value": CHAMPION_MODEL_TYPE},
        {"Key": "BestModelConfigVersion", "Value": BEST_MODEL_CONFIG_VERSION},
    ]

    print("\nMODEL ARTIFACT DEPLOYMENT PREFLIGHT")
    print("=" * 90)
    artifact_preflight = validate_model_package_artifact(
        sm=sm,
        s3=s3,
        model_package_arn=DEPLOY_PACKAGE_ARN,
    )
    print("Model Package :", DEPLOY_PACKAGE_ARN)
    print("Model artifact:", artifact_preflight["model_data_url"])
    print("Artifact valid:", artifact_preflight["valid"])
    print("Required      :", artifact_preflight["required_members"])

    if artifact_preflight["missing_members"]:
        print("Missing       :", artifact_preflight["missing_members"])
        print("=" * 90)
        raise RuntimeError(
            "Deployment blocked before endpoint creation. This registered model "
            "artifact was produced by the old pipeline and is missing serving files. "
            "Rerun the corrected pipeline, approve the new Model Package version, "
            "then deploy again."
        )

    print("Artifact deployment preflight: PASS")
    print("=" * 90)

    deployment_result = deploy_serverless_model_package(
        sm=sm,
        model_package_arn=DEPLOY_PACKAGE_ARN,
        role_arn=role,
        endpoint_name=ENDPOINT_NAME,
        tags=deployment_tags,
        memory_size_mb=2048,
        max_concurrency=5,
    )

    print("\nENDPOINT DEPLOYMENT SUMMARY")
    print("=" * 90)
    print("Endpoint         :", deployment_result["endpoint_name"])
    print("Action           :", deployment_result["deployment_action"])
    print("Status           :", deployment_result["endpoint_status"])
    print("Deployment OK    :", deployment_result["deployment_succeeded"])
    print("Model Package    :", deployment_result["model_package_arn"])

    if deployment_result.get("endpoint_config_name"):
        print("Endpoint Config  :", deployment_result["endpoint_config_name"])
    if deployment_result.get("serverless_memory_mb"):
        print("Serverless Memory:", deployment_result["serverless_memory_mb"], "MB")
    if deployment_result.get("serverless_max_concurrency"):
        print("Max Concurrency  :", deployment_result["serverless_max_concurrency"])

    if not deployment_result["deployment_succeeded"]:
        print("Failure reason   :", deployment_result.get("failure_reason"))
        print(
            "CloudWatch group :",
            deployment_result.get(
                "cloudwatch_log_group",
                f"/aws/sagemaker/Endpoints/{ENDPOINT_NAME}",
            ),
        )
        events = deployment_result.get("cloudwatch_events", [])
        if events:
            print("\nENDPOINT CLOUDWATCH LOG TAIL")
            print("-" * 90)
            for event in events:
                print(event["message"])
            print("-" * 90)

        if deployment_result.get("requires_explicit_delete"):
            print(
                "\nRemediation: endpoint is Failed. "
                "Set DELETE_ENDPOINT_NOW=True in the Human Control Panel, "
                "run Section 17 cleanup, then retry deployment."
            )
    else:
        print("Endpoint is ready for invocation.")

    print("=" * 90)


Deployment mode : FINAL_CHAMPION
Package group   : team02-diabetes-risk
Endpoint target : team02-diabetes-risk
Registry status  : Approved

MODEL ARTIFACT DEPLOYMENT PREFLIGHT
Model Package : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team02-diabetes-risk/3
Model artifact: s3://nyp-26s1-iti113/iti113/team02/mlops/diabetes-risk/pipeline-runs/6r4j7hisk3rw/training/pipelines-6r4j7hisk3rw-team02-train-model-moIzIPQLOk/output/model.tar.gz
Artifact valid: True
Required      : ['model.pkl', 'model_metadata.json', 'code/inference.py', 'code/requirements.txt']
Artifact deployment preflight: PASS

ENDPOINT DEPLOYMENT SUMMARY
Endpoint         : team02-diabetes-risk
Action           : created
Status           : InService
Deployment OK    : True
Model Package    : arn:aws:sagemaker:ap-southeast-1:044528205969:model-package/team02-diabetes-risk/3
Endpoint Config  : team02-diabetes-risk-config-1787378425
Serverless Memory: 2048 MB
Max Concurrency  : 5
Endpoint is ready for invocation

## 16. Validate the Selected Champion Endpoint with Held-Out Data

Invoke the serverless endpoint using canonical experiment-aligned held-out rows and verify:
- response contract;
- XGBoost model type;
- threshold 0.50;
- data version;
- config version;
- latency and basic label agreement.

This is an operational smoke test, not a replacement for the formal held-out evaluation.


In [34]:
def get_latest_pipeline_processing_output_uri(
    pipeline_name,
    step_name,
    output_name,
):
    executions = sm.list_pipeline_executions(
        PipelineName=pipeline_name,
        SortBy="CreationTime",
        SortOrder="Descending",
        MaxResults=20,
    ).get("PipelineExecutionSummaries", [])

    successful = [
        x for x in executions
        if x.get("PipelineExecutionStatus") == "Succeeded"
    ]
    if not successful:
        return None

    execution_arn = successful[0]["PipelineExecutionArn"]
    steps = sm.list_pipeline_execution_steps(
        PipelineExecutionArn=execution_arn
    ).get("PipelineExecutionSteps", [])

    step = next((s for s in steps if s.get("StepName") == step_name), None)
    if not step:
        return None

    processing_arn = step.get("Metadata", {}).get("ProcessingJob", {}).get("Arn")
    if not processing_arn:
        return None

    job_name = processing_arn.rsplit("/", 1)[-1]
    job = sm.describe_processing_job(ProcessingJobName=job_name)

    for output in job["ProcessingOutputConfig"]["Outputs"]:
        if output["OutputName"] == output_name:
            return output["S3Output"]["S3Uri"].rstrip("/") + f"/{output_name}.csv"
    return None

LATEST_TEST_S3_URI = get_latest_pipeline_processing_output_uri(
    PIPELINE_NAME,
    "team02-preprocess-data",
    "test",
)

if LATEST_TEST_S3_URI:
    print("Latest pipeline held-out test artifact:", LATEST_TEST_S3_URI)
else:
    print("No successful pipeline test artifact found yet.")


def read_s3_csv_uri(uri):
    b, k = parse_s3_uri(uri)
    return pd.read_csv(
        io.BytesIO(s3.get_object(Bucket=b, Key=k)["Body"].read())
    )

endpoint_desc = None
try:
    endpoint_desc = sm.describe_endpoint(EndpointName=ENDPOINT_NAME)
except ClientError as exc:
    if exc.response.get("Error", {}).get("Code") not in {"ValidationException"}:
        raise

if endpoint_desc is None:
    print("Endpoint does not exist yet; inference test skipped.")
elif endpoint_desc["EndpointStatus"] != "InService":
    print("Endpoint is not InService; inference test skipped.")
    print("Current status:", endpoint_desc["EndpointStatus"])
else:
    if not LATEST_TEST_S3_URI:
        raise RuntimeError("Endpoint is InService but no successful pipeline held-out artifact was found.")
    held_out = read_s3_csv_uri(LATEST_TEST_S3_URI)
    sample_features = held_out.iloc[0][FEATURE_COLUMNS].to_dict()

    sample_features = {
        k: (int(v) if float(v).is_integer() else float(v))
        for k, v in sample_features.items()
    }

    runtime = boto3.client("sagemaker-runtime", region_name=region)

    endpoint_result = invoke_json_endpoint(
        runtime,
        ENDPOINT_NAME,
        sample_features,
    )

    print("\nENDPOINT INVOCATION SUMMARY")
    print("=" * 90)
    print("Endpoint    :", ENDPOINT_NAME)
    print("Input source: canonical held-out test row")
    print("Response:")
    print(json.dumps(endpoint_result, indent=2))
    print("=" * 90)


Latest pipeline held-out test artifact: s3://nyp-26s1-iti113/iti113/team02/mlops/diabetes-risk/pipeline-runs/6r4j7hisk3rw/processed/test/test.csv

ENDPOINT INVOCATION SUMMARY
Endpoint    : team02-diabetes-risk
Input source: canonical held-out test row
Response:
[
  {
    "screening_class": 1,
    "screening_category": "prediabetes_or_diabetes_screening_positive",
    "probability": 0.5022,
    "decision_threshold": 0.5,
    "model_type": "XGBoost",
    "data_version": "brfss2015-diabetes-binary-6244bec277fe",
    "model_run_id": "057bb58d81564c0db9a628b224eab8f9",
    "config_version": "2026-08-22-cross-model-champion-v5",
    "intended_use_notice": "Academic screening-support prototype only. This output is not a medical diagnosis and does not rule diabetes in or out."
  }
]


In [35]:
# ============================================================
# SECTION 16.1 — BATCH ENDPOINT SMOKE TEST
# Uses canonical held-out test data.
# This is an operational/API test, NOT a replacement for
# the full test-set evaluation already performed in Section 9.
# ============================================================

import time
import pandas as pd
import boto3

TEST_CASES_PER_CLASS = 4

# ------------------------------------------------------------
# 1. Confirm endpoint is available
# ------------------------------------------------------------
endpoint_state = get_endpoint_summary(
    sm,
    ENDPOINT_NAME,
)

print("ENDPOINT SMOKE TEST")
print("=" * 100)
print("Endpoint :", ENDPOINT_NAME)
print("Status   :", endpoint_state["endpoint_status"])

if (
    not endpoint_state["exists"]
    or endpoint_state["endpoint_status"] != "InService"
):
    raise RuntimeError(
        f"Endpoint is not ready. Current state: {endpoint_state}"
    )

# ------------------------------------------------------------
# 2. Load canonical untouched test data from S3
# ------------------------------------------------------------
if not LATEST_TEST_S3_URI:
    raise RuntimeError("No successful pipeline held-out test artifact is available.")
held_out = read_s3_csv_uri(
    LATEST_TEST_S3_URI
)

if "target" not in held_out.columns:
    raise RuntimeError(
        "Expected target column was not found in the held-out test data."
    )

# Select known negative and positive examples.
negative_cases = (
    held_out[held_out["target"] == 0]
    .sample(
        n=TEST_CASES_PER_CLASS,
        random_state=42,
    )
)

positive_cases = (
    held_out[held_out["target"] == 1]
    .sample(
        n=TEST_CASES_PER_CLASS,
        random_state=42,
    )
)

test_cases = pd.concat(
    [negative_cases, positive_cases],
    ignore_index=True,
)

actual_target = test_cases["target"].astype(int).tolist()

# ------------------------------------------------------------
# 3. Build endpoint JSON payload
# ------------------------------------------------------------
payload = []

for _, row in test_cases.iterrows():

    record = {}

    for feature in FEATURE_COLUMNS:
        value = float(row[feature])

        if value.is_integer():
            record[feature] = int(value)
        else:
            record[feature] = value

    payload.append(record)

print("Test cases :", len(payload))
print("Negative   :", actual_target.count(0))
print("Positive   :", actual_target.count(1))

# ------------------------------------------------------------
# 4. Invoke SageMaker Serverless endpoint
# ------------------------------------------------------------
runtime = boto3.client(
    "sagemaker-runtime",
    region_name=region,
)

start_time = time.perf_counter()

endpoint_response = invoke_json_endpoint(
    runtime,
    ENDPOINT_NAME,
    payload,
)

elapsed_seconds = time.perf_counter() - start_time

# ------------------------------------------------------------
# 5. Validate response contract
# ------------------------------------------------------------
if len(endpoint_response) != len(payload):
    raise RuntimeError(
        "Endpoint returned a different number of predictions "
        "than input records."
    )

threshold_consistent = all(
    int(row["screening_class"])
    ==
    int(
        float(row["probability"])
        >= float(row["decision_threshold"])
    )
    for row in endpoint_response
)

model_consistent = all(
    row["model_type"] == CHAMPION_MODEL_TYPE
    for row in endpoint_response
)

data_version_consistent = all(
    row["data_version"] == DATA_VERSION
    for row in endpoint_response
)


model_run_consistent = all(
    row["model_run_id"] == CHAMPION_FINAL_RUN_ID
    for row in endpoint_response
)

config_version_consistent = all(
    row.get("config_version") == BEST_MODEL_CONFIG_VERSION
    for row in endpoint_response
)

# ------------------------------------------------------------
# 6. Build readable evidence table
# ------------------------------------------------------------
results = []

for case_id, (actual, response) in enumerate(
    zip(actual_target, endpoint_response),
    start=1,
):
    results.append({
        "case": case_id,
        "actual_target": actual,
        "predicted_class": response["screening_class"],
        "probability": response["probability"],
        "threshold": response["decision_threshold"],
        "category": response["screening_category"],
    })

results_df = pd.DataFrame(results)

display(results_df)

sample_agreement = (
    results_df["actual_target"]
    == results_df["predicted_class"]
).mean()

# ------------------------------------------------------------
# 7. Operational test summary
# ------------------------------------------------------------
print("\nENDPOINT TEST SUMMARY")
print("=" * 100)
print("Endpoint status             : InService")
print("Records tested              :", len(payload))
print(
    "Batch latency              :",
    f"{elapsed_seconds:.3f} seconds",
)
print(
    "Approx. latency per record :",
    f"{elapsed_seconds / len(payload):.3f} seconds",
)
print(
    "Threshold logic consistent :",
    threshold_consistent,
)
print(
    "Model metadata consistent  :",
    model_consistent,
)
print(
    "Data version consistent    :",
    data_version_consistent,
)
print(
    "Model run/version consistent:",
    model_run_consistent,
)
print(
    "Config version consistent  :",
    config_version_consistent,
)
print(
    "Sample label agreement     :",
    f"{sample_agreement:.1%}",
)
print("=" * 100)

print(
    "\nNOTE: Sample label agreement above is only an endpoint "
    "smoke-test check. Use the full untouched-test metrics from "
    "Section 9 for formal model evaluation."
)

# Strong operational assertions
assert threshold_consistent
assert model_consistent
assert data_version_consistent
assert model_run_consistent
assert config_version_consistent

print("\n✅ Serverless endpoint batch smoke test PASSED.")

ENDPOINT SMOKE TEST
Endpoint : team02-diabetes-risk
Status   : InService
Test cases : 8
Negative   : 4
Positive   : 4


,case,actual_target,predicted_class,probability,threshold,category
0,1,0,0,0.008477,0.5,screening_negative
1,2,0,0,0.165874,0.5,screening_negative
2,3,0,0,0.016041,0.5,screening_negative
3,4,0,0,0.098228,0.5,screening_negative
4,5,1,1,0.568049,0.5,prediabetes_or_diabetes_screening_positive
5,6,1,1,0.839591,0.5,prediabetes_or_diabetes_screening_positive
6,7,1,0,0.411662,0.5,screening_negative
7,8,1,1,0.567546,0.5,prediabetes_or_diabetes_screening_positive



ENDPOINT TEST SUMMARY
Endpoint status             : InService
Records tested              : 8
Batch latency              : 0.097 seconds
Approx. latency per record : 0.012 seconds
Threshold logic consistent : True
Model metadata consistent  : True
Data version consistent    : True
Model run/version consistent: True
Config version consistent  : True
Sample label agreement     : 87.5%

NOTE: Sample label agreement above is only an endpoint smoke-test check. Use the full untouched-test metrics from Section 9 for formal model evaluation.

✅ Serverless endpoint batch smoke test PASSED.


## 17. Cleanup / Failed Endpoint Remediation

Cleanup targets the **currently active endpoint name for the selected pipeline mode**:
- `FINAL_CHAMPION` → `team02-diabetes-risk`
- `PROGRESS_CHECK` / integration modes → `team02-diabetes-risk-test`

Deletion is deliberately disabled by default. To prevent accidental removal of the final demo endpoint, the destructive action requires both `DELETE_ENDPOINT_NOW=True` **and** an exact typed confirmation in `CONFIRM_DELETE_ENDPOINT_NAME`.


In [ ]:
# DELETE_ENDPOINT_NOW is controlled from the Human Control Panel.

if DELETE_ENDPOINT_NOW:
    if CONFIRM_DELETE_ENDPOINT_NAME != ENDPOINT_NAME:
        raise RuntimeError(
            "Endpoint deletion blocked: set CONFIRM_DELETE_ENDPOINT_NAME exactly "
            f"to {ENDPOINT_NAME!r} before rerunning cleanup."
        )
    cleanup_result = delete_endpoint_if_exists(
        sm,
        ENDPOINT_NAME,
    )
    print("Endpoint cleanup result:")
    print(json.dumps(cleanup_result, indent=2))
else:
    endpoint_state = get_endpoint_summary(sm, ENDPOINT_NAME)
    print("Endpoint not deleted.")
    print("Current endpoint state:")
    print(json.dumps(endpoint_state, indent=2, default=str))


## 18. Script-only Operational Dashboard Snapshot

Provide API-based evidence for:
- recent pipeline executions;
- Model Registry versions and approval states;
- endpoint health;
- current human-control switches.

The same helper functions can be reused by the Streamlit MLOps Admin Dashboard.


In [36]:
print("TEAM02 BEST-MODEL MLOPS OPERATIONAL SNAPSHOT")
print("=" * 100)

print("\n1) RECENT PIPELINE EXECUTIONS")
recent_executions = get_recent_pipeline_executions(
    sm,
    PIPELINE_NAME,
    max_results=5,
)
if recent_executions:
    display(pd.DataFrame([
        {
            "execution_arn": x.get("PipelineExecutionArn"),
            "status": x.get("PipelineExecutionStatus"),
            "start_time": x.get("StartTime"),
        }
        for x in recent_executions
    ]))
else:
    print("No pipeline executions found.")

print("\n2) BEST-MODEL REGISTRY")
registry_rows = get_model_registry_summary(
    sm,
    MODEL_PACKAGE_GROUP,
    max_results=5,
)
if registry_rows:
    display(pd.DataFrame(registry_rows))
else:
    print("No model packages found.")

print("\n3) SINGLE ENDPOINT")
endpoint_state = get_endpoint_summary(
    sm,
    ENDPOINT_NAME,
)
print(json.dumps(endpoint_state, indent=2, default=str))

print("\n4) CURRENT HUMAN CONTROL STATE")
print("Pipeline mode     :", PIPELINE_RUN_MODE)
print("Best model        :", CHAMPION_MODEL_TYPE)
print("Config version    :", BEST_MODEL_CONFIG_VERSION)
print("Registry          :", MODEL_PACKAGE_GROUP)
print("Endpoint          :", ENDPOINT_NAME)
print("Selection source  :", BEST_MODEL_SELECTION_SOURCE)
print("Run pipeline      :", RUN_PIPELINE_NOW)
print("Human approve     :", APPROVE_REGISTERED_MODEL)
print("Deploy endpoint   :", DEPLOY_ENDPOINT_AFTER_APPROVAL)
print("Delete endpoint   :", DELETE_ENDPOINT_NOW)
print("=" * 100)


TEAM02 BEST-MODEL MLOPS OPERATIONAL SNAPSHOT

1) RECENT PIPELINE EXECUTIONS


,execution_arn,status,start_time
0,arn:aws:sagemaker:ap-southeast-1:044528205969:...,Succeeded,2026-08-22 05:49:14.706000+00:00
1,arn:aws:sagemaker:ap-southeast-1:044528205969:...,Succeeded,2026-08-22 05:25:22.836000+00:00
2,arn:aws:sagemaker:ap-southeast-1:044528205969:...,Succeeded,2026-08-21 08:41:04.409000+00:00
3,arn:aws:sagemaker:ap-southeast-1:044528205969:...,Succeeded,2026-08-14 15:40:31.063000+00:00
4,arn:aws:sagemaker:ap-southeast-1:044528205969:...,Succeeded,2026-08-14 15:31:27.103000+00:00



2) BEST-MODEL REGISTRY


,model_package_arn,model_package_version,approval_status,creation_time
0,arn:aws:sagemaker:ap-southeast-1:044528205969:...,3,Approved,2026-08-22 05:59:52.769000+00:00
1,arn:aws:sagemaker:ap-southeast-1:044528205969:...,2,Approved,2026-08-22 05:38:20.921000+00:00
2,arn:aws:sagemaker:ap-southeast-1:044528205969:...,1,Approved,2026-08-21 08:54:15.185000+00:00



3) SINGLE ENDPOINT
{
  "exists": true,
  "endpoint_name": "team02-diabetes-risk",
  "endpoint_status": "InService",
  "endpoint_arn": "arn:aws:sagemaker:ap-southeast-1:044528205969:endpoint/team02-diabetes-risk",
  "endpoint_config_name": "team02-diabetes-risk-config-1787378425",
  "creation_time": "2026-08-22 06:00:26.824000+00:00",
  "last_modified_time": "2026-08-22 06:03:10.451000+00:00",
  "failure_reason": null
}

4) CURRENT HUMAN CONTROL STATE
Pipeline mode     : FINAL_CHAMPION
Best model        : XGBoost
Config version    : 2026-08-22-cross-model-champion-v5
Registry          : team02-diabetes-risk
Endpoint          : team02-diabetes-risk
Selection source  : 2026-08-22_cross_model_champion_same_holdout_pr_auc_primary
Run pipeline      : True
Human approve     : True
Deploy endpoint   : True
Delete endpoint   : True


## 19. Data / Model Drift Monitoring

Monitoring is separated into four layers:

1. **Operational monitoring** — SageMaker endpoint state and CloudWatch logs/latency/errors.
2. **Data drift** — compare recent feature distributions against the training reference using Population Stability Index (PSI).
3. **Model-performance / concept-drift evidence** — when delayed labels are available, compare PR-AUC, ROC-AUC, Recall and error rates with the approved baseline.

A data-distribution change is **not automatically called concept drift**. Concept drift requires evidence that the relationship between inputs and outcome/performance has changed.

The policy below writes an auditable monitoring contract and, when a successful pipeline run exists, creates a train-vs-held-out PSI snapshot as a plumbing test. In production, replace the held-out batch with recent captured inference data.

4. **Model/configuration drift monitoring** — endpoint responses are checked against the frozen champion final run ID, config version, data version and decision-threshold contract. A mismatch is treated as deployment/configuration drift and blocks the operational smoke test.


In [37]:
import math
import numpy as np

MONITORING_POLICY = {
    "config_version": BEST_MODEL_CONFIG_VERSION,
    "reference_data_version": DATA_VERSION,
    "reference_profile_sha256": EXPERIMENT_PROFILE_SHA256,
    "data_drift": {
        "metric": "PSI",
        "warning_threshold": 0.10,
        "alert_threshold": 0.25,
        "scope": FEATURE_COLUMNS,
    },
    "model_performance": {
        "requires_labels": True,
        "baseline": {
            "pr_auc": MODEL_SELECTION_EVIDENCE[CHAMPION_MODEL_TYPE]["pr_auc"],
            "roc_auc": MODEL_SELECTION_EVIDENCE[CHAMPION_MODEL_TYPE]["roc_auc"],
            "recall": MODEL_SELECTION_EVIDENCE[CHAMPION_MODEL_TYPE]["recall"],
            "f1": MODEL_SELECTION_EVIDENCE[CHAMPION_MODEL_TYPE]["f1"],
        },
        "alerts": {
            "roc_auc_below": 0.75,
            "pr_auc_absolute_drop": 0.05,
            "recall_absolute_drop": 0.05,
        },
    },
    "operational": {
        "sources": ["SageMaker endpoint status", "CloudWatch endpoint logs", "pipeline execution status"],
        "review": "each demo/run; scheduled production review recommended",
    },
    "model_configuration_drift": {
        "expected_model_run_id": CHAMPION_FINAL_RUN_ID,
        "expected_config_version": BEST_MODEL_CONFIG_VERSION,
        "expected_data_version": DATA_VERSION,
        "expected_decision_threshold": DECISION_THRESHOLD,
        "detective_control": "Section 16.1 endpoint metadata assertions",
        "action_on_mismatch": "block smoke test; investigate registry/package/endpoint lineage before reuse",
    },
    "human_action": (
        "Alert triggers investigation and human review; it does not automatically retrain, "
        "approve or deploy a replacement model."
    ),
}

Path("team02_monitoring_policy.json").write_text(
    json.dumps(MONITORING_POLICY, indent=2),
    encoding="utf-8",
)

def psi_for_series(reference, current, epsilon=1e-6):
    ref = pd.Series(reference).dropna()
    cur = pd.Series(current).dropna()
    categories = sorted(set(ref.unique()).union(set(cur.unique())))
    ref_p = ref.value_counts(normalize=True).reindex(categories, fill_value=0.0) + epsilon
    cur_p = cur.value_counts(normalize=True).reindex(categories, fill_value=0.0) + epsilon
    return float(((cur_p - ref_p) * np.log(cur_p / ref_p)).sum())

def assess_labelled_performance_drift(y_true, probabilities, threshold=DECISION_THRESHOLD):
    """Assess labelled-batch performance drift against the frozen champion baseline."""
    from sklearn.metrics import average_precision_score, f1_score, recall_score, roc_auc_score

    y_true = np.asarray(y_true, dtype=int)
    probabilities = np.asarray(probabilities, dtype=float)
    predictions = (probabilities >= float(threshold)).astype(int)

    current = {
        "pr_auc": float(average_precision_score(y_true, probabilities)),
        "roc_auc": float(roc_auc_score(y_true, probabilities)),
        "recall": float(recall_score(y_true, predictions, zero_division=0)),
        "f1": float(f1_score(y_true, predictions, zero_division=0)),
    }
    baseline = MONITORING_POLICY["model_performance"]["baseline"]
    alerts = MONITORING_POLICY["model_performance"]["alerts"]

    triggered = {
        "roc_auc_below": current["roc_auc"] < alerts["roc_auc_below"],
        "pr_auc_drop": (baseline["pr_auc"] - current["pr_auc"]) >= alerts["pr_auc_absolute_drop"],
        "recall_drop": (baseline["recall"] - current["recall"]) >= alerts["recall_absolute_drop"],
    }
    return {
        "current": current,
        "baseline": baseline,
        "triggered": triggered,
        "status": "ALERT" if any(triggered.values()) else "OK",
        "action": MONITORING_POLICY["human_action"],
    }

LATEST_TRAIN_S3_URI = get_latest_pipeline_processing_output_uri(
    PIPELINE_NAME,
    "team02-preprocess-data",
    "train",
)

drift_df = None
if LATEST_TRAIN_S3_URI and LATEST_TEST_S3_URI:
    train_ref = read_s3_csv_uri(LATEST_TRAIN_S3_URI)
    current_demo_batch = read_s3_csv_uri(LATEST_TEST_S3_URI)

    drift_rows = []
    for feature in FEATURE_COLUMNS:
        psi = psi_for_series(train_ref[feature], current_demo_batch[feature])
        if psi >= MONITORING_POLICY["data_drift"]["alert_threshold"]:
            status = "ALERT"
        elif psi >= MONITORING_POLICY["data_drift"]["warning_threshold"]:
            status = "WARNING"
        else:
            status = "OK"
        drift_rows.append({
            "feature": feature,
            "psi": psi,
            "status": status,
        })

    drift_df = pd.DataFrame(drift_rows).sort_values("psi", ascending=False)
    display(drift_df)
    drift_df.to_csv("team02_data_drift_snapshot.csv", index=False)
    print(
        "NOTE: train-vs-held-out PSI is a monitoring plumbing check. "
        "For production monitoring, compare training reference data with recent inference batches."
    )
else:
    print(
        "No successful pipeline train/test artifacts are available yet. "
        "Monitoring policy was still written to team02_monitoring_policy.json."
    )

print("\nMonitoring policy:")
print(json.dumps(MONITORING_POLICY, indent=2))


,feature,psi,status
3,BMI,2.082713e-03,OK
15,PhysHlth,1.070503e-03,OK
18,Age,9.446337e-04,OK
14,MentHlth,7.340522e-04,OK
13,GenHlth,1.992775e-04,OK
12,NoDocbcCost,1.445102e-04,OK
20,Income,1.398846e-04,OK
17,Sex,1.012284e-04,OK
10,HvyAlcoholConsump,4.199197e-05,OK
19,Education,3.442295e-05,OK


NOTE: train-vs-held-out PSI is a monitoring plumbing check. For production monitoring, compare training reference data with recent inference batches.

Monitoring policy:
{
  "config_version": "2026-08-22-cross-model-champion-v5",
  "reference_data_version": "brfss2015-diabetes-binary-6244bec277fe",
  "reference_profile_sha256": "6244bec277fe3cefce56d46c583a4c5eb73f179ef64c8a1b18557fd0200105c5",
  "data_drift": {
    "metric": "PSI",
    "warning_threshold": 0.1,
    "alert_threshold": 0.25,
    "scope": [
      "HighBP",
      "HighChol",
      "CholCheck",
      "BMI",
      "Smoker",
      "Stroke",
      "HeartDiseaseorAttack",
      "PhysActivity",
      "Fruits",
      "Veggies",
      "HvyAlcoholConsump",
      "AnyHealthcare",
      "NoDocbcCost",
      "GenHlth",
      "MentHlth",
      "PhysHlth",
      "DiffWalk",
      "Sex",
      "Age",
      "Education",
      "Income"
    ]
  },
  "model_performance": {
    "requires_labels": true,
    "baseline": {
      "pr_auc": 0.464

## 20. CI/CD and Change-trigger Automation

For the course demo, the notebook retains the AWS-only S3 change-trigger workflow discussed with the lecturer: approved code/data changes can trigger this single SageMaker Pipeline.

For stronger industry alignment, the notebook also generates a **GitHub Actions CI/CD template** that:
- validates the notebook as JSON;
- performs Python syntax checks on exported source scripts;
- uses AWS OIDC rather than static long-lived keys;
- triggers the existing SageMaker Pipeline only after code review/merge or manual dispatch.

The template is evidence of the intended CI/CD architecture and should only be enabled after the team configures the repository and OIDC role.


In [38]:
print("AWS-only change-trigger target")
print("=" * 80)
print("Pipeline :", PIPELINE_NAME)
print("Model    :", CHAMPION_MODEL_TYPE)
print("Registry :", MODEL_PACKAGE_GROUP)
print("Endpoint :", ENDPOINT_NAME)
print(
    "Course demo path: use the S3 change-monitor workflow to trigger this pipeline "
    "after an approved data/code change."
)
print("=" * 80)

workflow_yaml = f"""name: team02-mlops-ci-cd

on:
  workflow_dispatch:
  push:
    branches: [ main ]
    paths:
      - 'notebooks/**'
      - 'diabetes_pipeline_src/**'
      - '.github/workflows/team02-mlops-ci-cd.yml'

permissions:
  id-token: write
  contents: read

jobs:
  validate-and-trigger:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: '3.12'

      - name: Validate notebook and scripts
        run: |
          python -m pip install --upgrade pip nbformat
          python - <<'PY'
          import ast, nbformat
          from pathlib import Path

          notebook = next(Path("notebooks").glob("*sagemaker_pipeline*champion*.ipynb"))
          nb = nbformat.read(notebook, as_version=4)
          print("Notebook cells:", len(nb.cells))

          for script in Path("diabetes_pipeline_src").glob("*.py"):
              ast.parse(script.read_text(encoding="utf-8"))
              print("Syntax OK:", script)
          PY

      - name: Configure AWS credentials with OIDC
        uses: aws-actions/configure-aws-credentials@v4
        with:
          role-to-assume: ${{{{ secrets.AWS_GITHUB_OIDC_ROLE_ARN }}}}
          aws-region: {REGION}

      - name: Trigger approved SageMaker Pipeline
        run: |
          aws sagemaker start-pipeline-execution \
            --pipeline-name {PIPELINE_NAME}
"""

workflow_path = Path(".github/workflows/team02-mlops-ci-cd.yml")
workflow_path.parent.mkdir(parents=True, exist_ok=True)
workflow_path.write_text(workflow_yaml, encoding="utf-8")

print("\nGenerated CI/CD template:", workflow_path)
print(
    "Template only: do not enable until GitHub OIDC role, repository permissions "
    "and review/approval rules are configured."
)


AWS-only change-trigger target
Pipeline : team02-diabetes-risk
Model    : XGBoost
Registry : team02-diabetes-risk
Endpoint : team02-diabetes-risk
Course demo path: use the S3 change-monitor workflow to trigger this pipeline after an approved data/code change.

Generated CI/CD template: .github/workflows/team02-mlops-ci-cd.yml
Template only: do not enable until GitHub OIDC role, repository permissions and review/approval rules are configured.


## 21. Demo Application Architecture and Technology Stack

### User prediction path

```text
User
 ↓
Streamlit diabetes-risk UI
 ↓ HTTPS
FastAPI / Uvicorn relay
 ↓ authenticated AWS SDK call
SageMaker Serverless Endpoint — team02-diabetes-risk
 ↓
Selected champion probability + thresholded screening result
 ↓
Streamlit explanation / intended-use notice
```

### MLOps administration path

```text
MLOps Admin Dashboard
 ├─ Champion evidence → LR vs XGBoost final comparison
 ├─ SageMaker Pipeline API → execution status
 ├─ SageMaker Model Registry API → versions / approval
 ├─ SageMaker Endpoint API → endpoint health
 ├─ MLflow → parameters / metrics / artifacts / run IDs
 └─ CloudWatch → job and endpoint diagnostics
```

**Technology stack:** Python, pandas, scikit-learn processing container, XGBoost 2.1.4 for the current selected champion, SageMaker Processing/Training/Pipelines/Model Registry/Serverless Inference, MLflow, boto3, CloudWatch, Streamlit, FastAPI/Uvicorn, and an ngrok relay for the classroom demo where required.

The UI is intentionally **single-champion**: users do not select the algorithm. Notebook 03 selects the best final model from the frozen LR-vs-XGBoost evidence, and the endpoint serves only the human-approved champion. With the current evidence that champion is XGBoost.

## 22. Final-Champion Selection and Governance Review

### Source evidence exports used for this update

| Evidence export | SHA-256 | Selected lineage |
|---|---|---|
| `S202_8440104B_XGBoost_Experiments_Evidence_Runs.csv` | `d95703aa1e4b91197ca08fa545db7a7f10adb1e37a233938f19c50f3f6617228` | candidate `02807641d2814f48a084c7c451414917` → refinement `e89987fd9aec4a11a1108c8afaf6561b` → threshold `a1a47bcc23d844539d92f3a3ed8fec1e` → final `057bb58d81564c0db9a628b224eab8f9` |
| `S203_2730358J_Logistic_Regression_Experiment_Evidence_Runs.csv` | `f95bfd7d23b0257b2bec267064b9f661004531337be989f0728b0ae0e86a4153` | candidate `74e8884c230349f3ad29021db15f54a2` → threshold `4d0a1f65fd754905bb85a4459886cb10` → final `82bc9a4720e04a50ae942128305eb90d` |

**Lineage limitation:** the latest exports do not include the full-profile or training-split SHA-256 values used by Notebook 03. Those remain operational lineage anchors that are recomputed from the S3 dataset and deterministic split. The XGBoost final export does include the exact comparison test-split hash `12de3cf4074e8656a059c938acbe27143794910116dda3dc217a101698a368c1`.

### Cross-model decision

| Model | Final run | Held-out PR-AUC | Decision threshold | Operational status |
|---|---|---:|---:|---|
| Logistic Regression | `82bc9a4720e04a50ae942128305eb90d` | 0.432749 | 0.59 | Runner-up / comparison evidence |
| **XGBoost** | `057bb58d81564c0db9a628b224eab8f9` | **0.464397** | 0.50 | **Selected champion** |

The notebook computes the winner from the frozen primary metric rather than pre-setting the model family. **XGBoost wins by +0.031649 held-out PR-AUC.**

### Selected XGBoost champion configuration

| Item | Latest 22 Aug 2026 evidence |
|---|---|
| Candidate | `xgb_candidate_29` |
| Candidate run | `02807641d2814f48a084c7c451414917` |
| Refinement run | `e89987fd9aec4a11a1108c8afaf6561b` |
| Threshold run | `a1a47bcc23d844539d92f3a3ed8fec1e` |
| Final XGBoost run | `057bb58d81564c0db9a628b224eab8f9` |
| `n_estimators` | **450** |
| `max_depth` | **4** |
| `learning_rate` | **0.03** |
| `min_child_weight` | **5.0** |
| `subsample` | **0.80** |
| `colsample_bytree` | **0.80** |
| `gamma` | **0.05** |
| `reg_alpha` | **0.10** |
| `reg_lambda` | **10.0** |
| `scale_pos_weight` | **3.0** |
| Threshold | **0.50** |

### Latest Logistic Regression runner-up configuration

| Item | Latest 22 Aug 2026 evidence |
|---|---|
| Candidate | `lr_candidate_12` |
| Candidate run | `74e8884c230349f3ad29021db15f54a2` |
| Threshold run | `4d0a1f65fd754905bb85a4459886cb10` |
| Final LR run | `82bc9a4720e04a50ae942128305eb90d` |
| `C` | **100.0** |
| `class_weight` | **balanced** |
| `solver` | **lbfgs** |
| `penalty` | **l2** |
| `max_iter` | **3000** |
| Threshold | **0.59** |

### Champion evidence and trade-offs

- XGBoost held-out PR-AUC: **0.464397**
- Logistic Regression held-out PR-AUC: **0.432749**
- Delta: **+0.031649**
- XGBoost ROC-AUC: **0.824820** vs LR **0.817565**
- XGBoost F1: **0.492442** vs LR **0.482005**
- XGBoost Precision: **0.407815** vs LR **0.380738**
- Trade-off: XGBoost Recall **0.621388** is lower than LR **0.656660**; Balanced Accuracy is also slightly lower (**0.726302** vs **0.728438**).

The champion decision therefore follows the **predefined PR-AUC-primary rule** rather than claiming XGBoost dominates every metric.

### Governance controls retained

- champion model is selected from explicit cross-model evidence, not manually hard-coded;
- no post-hoc metric switching;
- exact candidate/refinement/threshold/final run lineage;
- exact data and split signatures;
- human-over-the-loop Model Registry approval;
- no automatic production deployment;
- subgroup error/fairness evidence;
- SHAP explainability evidence remains linked to Model B;
- intended-use limitation: screening support only, not diagnosis;
- drift alerts trigger review, not automatic retraining/approval;
- if future evidence selects a different model family, the notebook fails safely rather than deploying stale XGBoost code.

## 23. Requirements / Rubric Evidence Produced by This Notebook

### Requirement 1 — clear end-to-end flow
**Achieved in design:** ingestion → preprocessing → training → evaluation → experiment tracking → registration → human approval → deployment → inference → monitoring.

### Requirement 2 — reproducible experiment tracking
MLflow and JSON artifacts record:
- parameters;
- metrics;
- model/config version and source final model run ID;
- source modelling run IDs;
- exact source evidence CSV hashes and verification artifact;
- data version and S3 URI;
- profile/train/test SHA-256 signatures;
- explicit preprocessing configuration version + split configuration;
- artifacts;
- pipeline execution ARN;
- MLflow run ID;
- runtime package version.

### Requirement 3 — complete MLOps and deployment setup
- **Data pipeline:** exact experiment-aligned cleaned source and deterministic 80/20 split.
- **Experiment/model tracking:** SageMaker MLflow App.
- **Model Registry:** PendingManualApproval → human approval.
- **CI/CD:** AWS S3 change-trigger demo path + GitHub Actions/OIDC template.
- **Monitoring:** CloudWatch operational checks + PSI data-drift policy + labelled performance-drift logic + model/configuration drift assertions.
- **Deployment/inference:** SageMaker Serverless Endpoint with validated JSON contract.
- **Demo application:** Streamlit → FastAPI relay → SageMaker endpoint.
- **Application administration:** API-based pipeline/registry/endpoint snapshot suitable for Streamlit MLOps Admin Dashboard.

### Assessment category mapping

**A — Data & EDA**
- exact cleaned dataset lineage;
- schema/domain/missing-value checks;
- reproducible split signatures.

**B — Model Development**
- cross-model champion selected from the latest LR-vs-XGBoost final evidence using the frozen held-out PR-AUC rule;
- exact selected XGBoost champion hyperparameters and threshold transferred from Model B evidence;
- Logistic Regression hyperparameters, threshold and run lineage retained as runner-up/comparison evidence;
- final selection trade-offs documented without post-hoc metric switching.

**C — MLOps & Deployment**
- automated SageMaker pipeline;
- MLflow traceability;
- model registry;
- human approval;
- serverless deployment;
- CI/CD design;
- monitoring and drift controls.

**D — Analysis & Communication**
- explicit cross-model PR-AUC-primary champion rationale;
- secondary metric trade-offs documented rather than hidden;
- machine-readable evidence artifacts, including `model_hyperparameter_handoff.csv` for the latest LR/XGBoost parameter lineage.

**E — AI Governance**
- human-over-the-loop;
- no post-hoc metric switching;
- fairness evidence;
- explainability linkage;
- traceability/reproducibility;
- intended-use controls;
- drift/incident review triggers.


### Responsible AI / risk-management alignment

- **ISAGO operations management:** lineage, reproducible preprocessing, separate held-out evaluation, explainability/fairness evidence, active monitoring and human review are retained as inspectable controls.
- **ISO/IEC 42001 planning (6.1 / 6.2):** the notebook identifies data, performance, drift and over-reliance risks; assigns preventive/detective controls; keeps approval/deployment under human authority; and uses measurable objectives such as the pre-existing ROC-AUC regression gate and drift alert thresholds.
- **ISO/IEC 23894 risk lifecycle:** risks are identified (data mismatch, model degradation, subgroup error, deployment/configuration drift), analysed through metrics and monitoring, evaluated against thresholds, and treated through hash validation, independent evaluation, PendingManualApproval, drift alerts and rollback/incident review.
- **AI impact / human oversight:** the endpoint is a screening-support prototype, not a diagnosis engine. Positive/negative outputs require appropriate human interpretation, and automated retraining/approval/deployment is intentionally disabled by default.

These controls support **E — AI Governance** while also strengthening **C — MLOps & Deployment** through traceability, reproducibility, monitoring and controlled release management.
